# FIT5196 Ass2 2026 中文从零讲解 Notebook

**用途：教学 / tutoring，不是直接提交版。**

我们模拟第一次拿到 assignment 的过程，一步一步思考：

- 题目到底要求我们交什么？
- 哪些信息来自 assignment guide？
- 哪些规则要从数据里发现？
- 每一步为什么要这么做？
- 每一步运行后，我们根据输出决定下一步。

> 课堂原则：不要一上来就清洗数据。先读题、拆任务、看数据，再决定方法。

## Cell 1：环境准备

第一步只做最基础的环境准备：

- 确认当前工作目录；
- 导入后面会用到的基础库；
- 设置 pandas 显示选项，方便课堂观察表格。

这一格的目标不是解决 assignment，而是确保我们站在正确的文件夹里。

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd()
print('Current working directory:', ROOT)
print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

Current working directory: /Users/songhaifan/Documents/GitHub/teaching-materials/courses/lincoin/materials/fit5196/ass2_2026
pandas version: 3.0.3
numpy version: 2.4.4


## Cell 2：先把 Task 1 的任务边界讲清楚

在正式读数据之前，我们先回答三个问题。

### 1. Task 1 要交什么？

Task 1 是 **Data Cleansing**，需要提交 5 个文件：

- `Group024_dirty_data_solution.csv`
- `Group024_missing_data_solution.csv`
- `Group024_outlier_data_solution.csv`
- `Group024_ass2_task1.ipynb`
- `Group024_ass2_task1.py`

其中 `.ipynb` 要展示完整方法、过程、解释和输出；`.py` 主要用于 plagiarism checking。

### 2. 哪三个 CSV 分别做什么？

| 文件 | 我们要做什么 | 关键限制 |
|---|---|---|
| `Group024_dirty_data.csv` | 找出并修复错误值 | 每一行最多只有一个 anomaly；每个 anomaly 只有一个正确修法 |
| `Group024_missing_data.csv` | 填补缺失值 | 只有 coverage / missing anomalies，没有其他脏数据问题 |
| `Group024_outlier_data.csv` | 删除 outlier rows | 只针对 `delivery_fee` 判断 outlier，删除整行，不是改值 |

所以这三个文件不能用同一套粗暴方法处理。它们代表三类不同的数据质量问题。

### 3. 哪些列不能乱改？

题目明确说，在 dirty data 里不要找这些列的错误：

- `order_id`
- `time`
- `order_items` 里的 **数量**
- `delivery_fee`

注意：`order_items` 里的 item name 可能有问题，但 quantity 不应该改。  
`delivery_fee` 在 dirty data 里不改；它只在 missing / outlier 任务里发挥作用。

### 课堂提醒

这一步非常重要。我们不是看到奇怪值就改，而是先根据 assignment guide 定义“什么可以改、什么不能改”。这就是 FIT5196 Week 7-8 讲的数据质量审计思路。

## Cell 3：读取 Task 1 需要的文件

现在才开始读数据。第一轮只看：

- 文件是否能读；
- 每个文件有多少行、多少列；
- 缺失值总数；
- dirty data 长什么样。

这一步对应 data discovery / profiling。我们先观察，不急着修。

In [2]:
files = {
    'dirty': 'Group024_dirty_data.csv',
    'missing': 'Group024_missing_data.csv',
    'outlier': 'Group024_outlier_data.csv',
    'branches': 'branches.csv',
    'nodes': 'nodes.csv',
    'edges': 'edges.csv',
}

data = {name: pd.read_csv(path) for name, path in files.items()}

file_profile = pd.DataFrame([
    {
        'dataset': name,
        'filename': files[name],
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
    }
    for name, df in data.items()
])

display(file_profile)

print('First 3 rows of dirty data:')
display(data['dirty'].head(3))

,dataset,filename,rows,columns,missing_cells
0,dirty,Group024_dirty_data.csv,500,12,0
1,missing,Group024_missing_data.csv,500,12,200
2,outlier,Group024_outlier_data.csv,500,12,0
3,branches,branches.csv,3,4,0
4,nodes,nodes.csv,17117,3,0
5,edges,edges.csv,42224,6,0


First 3 rows of dirty data:


,order_id,date,time,order_type,branch_code,order_items,order_price,customer_lat,customer_lon,customerHasloyalty?,distance_to_customer_KM,delivery_fee
0,ORDX00699,03-08-2018,15:05:54,Lunch,BK,"[('Chicken', 2), ('Steak', 1)]",109.0000,-37.8123,144.9737,0,6.7760,12.6337
1,ORDX06260,2018-03-14,10:21:58,Breakfast,BK,"[('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)]",548.5000,-37.8175,144.9778,0,6.8330,11.8394
2,ORDI04941,2018-04-22,11:43:05,Breakfast,NS,"[('Cereal', 5), ('Pancake', 6), ('Coffee', 8), ('Eggs', 7)]",464.5000,-37.8191,144.9535,0,9.2260,16.3654


### Cell 3 讨论：我们从第一次读取数据得到什么？

**问题 1：为什么第一步只看 shape / missing cells / head，而不是直接开始清洗？**

参考回答：因为我们还不知道数据的整体结构。如果不先看行数、列数、缺失情况和样例，很容易把三类任务混在一起处理。Data wrangling 的第一步是 discovery / profiling，不是修改数据。

**问题 2：从 file profile 里，我们应该立刻注意到什么？**

参考回答：三个 order 数据集都是 500 行、12 列，说明它们很可能共享同一套 schema。`nodes.csv` 和 `edges.csv` 很大，说明距离计算不是简单经纬度公式，而是 graph / shortest path 问题。

**问题 3：为什么要展示 dirty data 前几行？**

参考回答：`head()` 可以帮助学生认识字段含义，例如 `order_id`, `date`, `time`, `order_type`, `branch_code`, `order_items`, `delivery_fee` 等。它不是为了找出所有错误，而是为了建立对一行订单记录的直觉。

**下一步自然问题：**

既然三个 order CSV 看起来都是 12 列，我们下一步应该确认它们的 column names 是否完全一致。因为 assignment 明确要求输出 CSV 必须保持 exact same columns。

## Cell 3.5：dirty_data 全表 overview

在正式修复任何问题之前，我们先对 `dirty_data` 做一个完整 overview。

这一步的目标是回答：每一列是什么类型？有没有缺失？唯一值多不多？样例长什么样？数值列的范围是否可疑？分类列有哪些常见取值？

In [3]:
dirty = data['dirty'].copy()

column_overview = pd.DataFrame({
    'column': dirty.columns,
    'dtype': [str(dirty[column].dtype) for column in dirty.columns],
    'missing_count': [int(dirty[column].isna().sum()) for column in dirty.columns],
    'missing_rate': [dirty[column].isna().mean() for column in dirty.columns],
    'unique_count': [int(dirty[column].nunique(dropna=True)) for column in dirty.columns],
    'example_values': [dirty[column].dropna().head(3).tolist() for column in dirty.columns],
})

display(column_overview)

numeric_columns = dirty.select_dtypes(include='number').columns.tolist()
numeric_summary = dirty[numeric_columns].describe().T

display(numeric_summary)

categorical_columns = [
    column for column in dirty.columns
    if column not in numeric_columns
]

for column in categorical_columns:
    print(f'Column: {column}')
    print(dirty[column].astype(str).value_counts(dropna=False).head(10))
    print('-' * 60)

,column,dtype,missing_count,missing_rate,unique_count,example_values
0,order_id,str,0,0.0000,500,"[ORDX00699, ORDX06260, ORDI04941]"
1,date,str,0,0.0000,295,"[03-08-2018, 2018-03-14, 2018-04-22]"
2,time,str,0,0.0000,72,"[15:05:54, 10:21:58, 11:43:05]"
3,order_type,str,0,0.0000,3,"[Lunch, Breakfast, Breakfast]"
4,branch_code,str,0,0.0000,6,"[BK, BK, NS]"
5,order_items,str,0,0.0000,499,"[[('Chicken', 2), ('Steak', 1)], [('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)], [('Cereal', 5), ('Panc..."
6,order_price,float64,0,0.0000,428,"[109.0, 548.5, 464.5]"
7,customer_lat,float64,0,0.0000,493,"[-37.8122756, -37.8174594, -37.8190803]"
8,customer_lon,float64,0,0.0000,493,"[144.973739, 144.9778062, 144.9535332]"
9,customerHasloyalty?,int64,0,0.0000,2,"[0, 0, 0]"


,count,mean,std,min,25%,50%,75%,max
order_price,500.0000,498.5134,273.6692,43.0000,272.7125,452.2500,665.0750,"1,370.0000"
customer_lat,500.0000,-30.7536,25.3377,-37.8328,-37.8178,-37.8121,-37.8042,145.0074
customer_lon,500.0000,143.5055,16.2995,-37.8246,144.9530,144.9651,144.9827,145.0184
customerHasloyalty?,500.0000,0.1320,0.3388,0.0000,0.0000,0.0000,0.0000,1.0000
distance_to_customer_KM,500.0000,8.7449,1.5913,3.2340,7.8385,8.8685,9.7350,13.6850
delivery_fee,500.0000,13.6844,2.5464,4.5387,12.5833,13.8590,15.2141,21.0097


Column: order_id
order_id
ORDX00699    1
ORDX06260    1
ORDI04941    1
ORDI03372    1
ORDK03173    1
ORDB02744    1
ORDB07772    1
ORDI02220    1
ORDY07813    1
ORDJ02557    1
Name: count, dtype: int64
------------------------------------------------------------
Column: date
date
2018-09-30    5
2018-07-25    5
2018-10-19    5
2018-01-07    4
2018-08-12    4
2018-07-16    4
2018-03-21    4
2018-02-05    4
2018-09-09    4
2018-01-11    4
Name: count, dtype: int64
------------------------------------------------------------
Column: time
time
18:08:27    13
11:22:49    13
12:33:48    12
08:10:08    12
10:11:49    12
10:21:58    11
14:45:38    11
09:21:07    11
17:17:44    11
13:04:13    10
Name: count, dtype: int64
------------------------------------------------------------
Column: order_type
order_type
Breakfast    179
Lunch        173
Dinner       148
Name: count, dtype: int64
------------------------------------------------------------
Column: branch_code
branch_code
TP    188
BK    1

### Cell 3.5 讨论：overview 告诉我们先看什么？

**Q1：为什么在正式清洗前要先做 overview？**  
A：因为 dirty task 的错误类型可能分散在不同列里。我们不能一开始只盯着某一列，而应该先知道每列的数据类型、取值范围、唯一值数量和典型样例。

**Q2：这个 overview 对每类字段有什么帮助？**  
A：`date`、`time`、`order_type`、`branch_code` 更像格式和类别规则检查；`order_items` 和 `order_price` 涉及菜单价格和订单金额规则；`customer_lat`、`customer_lon`、`distance_to_customer_KM` 涉及空间和图距离；`delivery_fee` 后面可能需要模型判断。

**Q3：为什么 dirty_data 没有缺失也不代表干净？**  
A：dirty 文件的问题通常不是空值，而是错误值。例如日期格式可能不一致，类别可能大小写错误，价格可能不等于菜单计算结果，距离可能不等于图最短路径。

**Q4：这一步给后面任务建立了什么路线？**  
A：我们会按列的业务含义逐步检查：先看 schema 和基本格式，再看 branch/order type/date/time 等规则字段，然后检查 order_items 与 order_price，最后检查距离和 delivery fee。

**下一步我们应该问：**  
三个订单文件的列结构是否完全一致？如果不一致，后面的输出 CSV 就可能不符合 submission 要求。

## Cell 4：检查三个 order CSV 的 schema 是否一致

现在我们检查 `dirty`, `missing`, `outlier` 三个订单文件的 column names 是否完全一致。

这一步很关键，因为 assignment guide 明确说：输出 CSV 必须和对应输入 CSV 有完全一样的 columns。

In [4]:
order_datasets = ['dirty', 'missing', 'outlier']

schema_table = pd.DataFrame({
    name: pd.Series(list(data[name].columns))
    for name in order_datasets
})

display(schema_table)

schema_matches_dirty = {
    name: list(data[name].columns) == list(data['dirty'].columns)
    for name in order_datasets
}

schema_check = pd.DataFrame([
    {
        'dataset': name,
        'matches_dirty_schema': matches,
        'column_count': data[name].shape[1],
    }
    for name, matches in schema_matches_dirty.items()
])

display(schema_check)

print('All order datasets share the same schema:', all(schema_matches_dirty.values()))

,dirty,missing,outlier
0,order_id,order_id,order_id
1,date,date,date
2,time,time,time
3,order_type,order_type,order_type
4,branch_code,branch_code,branch_code
5,order_items,order_items,order_items
6,order_price,order_price,order_price
7,customer_lat,customer_lat,customer_lat
8,customer_lon,customer_lon,customer_lon
9,customerHasloyalty?,customerHasloyalty?,customerHasloyalty?


,dataset,matches_dirty_schema,column_count
0,dirty,True,12
1,missing,True,12
2,outlier,True,12


All order datasets share the same schema: True


### Cell 4 讨论：为什么 schema check 是第一道防线？

**问题 1：这个输出告诉我们什么？**

参考回答：`dirty`, `missing`, `outlier` 三个订单文件的 columns 完全一致，都是 12 列。这说明我们后面可以设计一套共享的读取、验证和输出检查逻辑。

**问题 2：为什么 schema 一致不等于可以用同一种方法清洗？**

参考回答：schema 一致只说明字段结构一样，但 assignment 给三份文件定义了不同的问题类型：dirty 是错误值，missing 是缺失值，outlier 只针对 `delivery_fee`。所以方法仍然要分开。

**问题 3：为什么这个检查和 automarker 有关？**

参考回答：automarker 通常会按固定 column names 读取我们的输出。如果列名拼错、少列、多列或顺序异常，可能导致该输出文件直接拿 0。

## Cell 5：根据 assignment guide，先做 dirty_data

Assignment guide 对 Task 1 的顺序是：

1. Detect and fix errors in `Group<group_id>_dirty_data.csv`。
2. Impute missing values in `Group<group_id>_missing_data.csv`。
3. Detect and remove outlier rows in `Group<group_id>_outlier_data.csv`，而且 outlier 只针对 `delivery_fee`。

所以我们现在先不看 missing，也先不看 outlier。接下来只围绕 `dirty_data` 做一件事：逐列检测错误，并为每一类错误找到唯一、可解释的修复方法。

## Cell 5.5：dirty_data 逐列检查路线图

在处理 `dirty_data` 时，我们不要随机挑列修。更好的做法是先为每一列建立检查路线图：

- 这一列应该满足什么规则？
- 可能出现什么错误？
- 需要用哪些外部文件或业务规则判断？
- 修复后如何验证没有引入新错误？

In [5]:
dirty_column_plan = pd.DataFrame([
    {
        'column': 'order_id',
        'role': 'unique order identifier',
        'main_check': 'format, uniqueness, and prefix consistency',
        'evidence_needed': 'order_id pattern and branch_code relationship',
    },
    {
        'column': 'date',
        'role': 'order date',
        'main_check': 'parseability and valid 2018 date format',
        'evidence_needed': 'datetime parsing and assignment rules',
    },
    {
        'column': 'time',
        'role': 'order time',
        'main_check': 'parseability and meal time window consistency',
        'evidence_needed': 'Breakfast, Lunch, Dinner time rules',
    },
    {
        'column': 'order_type',
        'role': 'meal category',
        'main_check': 'must match time window',
        'evidence_needed': 'time column and meal time rules',
    },
    {
        'column': 'branch_code',
        'role': 'restaurant branch code',
        'main_check': 'legal code, case consistency, and order_id prefix consistency',
        'evidence_needed': 'branches.csv and order_id prefix mapping',
    },
    {
        'column': 'order_items',
        'role': 'items and quantities in the order',
        'main_check': 'valid item names, quantities, and menu-price consistency',
        'evidence_needed': 'menu prices inferred from reliable rows',
    },
    {
        'column': 'order_price',
        'role': 'total food price before delivery',
        'main_check': 'must equal sum of item price times quantity',
        'evidence_needed': 'order_items and inferred menu prices',
    },
    {
        'column': 'customer_lat',
        'role': 'customer latitude',
        'main_check': 'valid coordinate and distance consistency',
        'evidence_needed': 'nodes.csv and graph nearest-node logic',
    },
    {
        'column': 'customer_lon',
        'role': 'customer longitude',
        'main_check': 'valid coordinate and distance consistency',
        'evidence_needed': 'nodes.csv and graph nearest-node logic',
    },
    {
        'column': 'customerHasloyalty?',
        'role': 'loyalty indicator',
        'main_check': 'binary value and delivery-fee consistency',
        'evidence_needed': 'allowed values and delivery fee model',
    },
    {
        'column': 'distance_to_customer_KM',
        'role': 'network distance from branch to customer',
        'main_check': 'must match shortest path distance',
        'evidence_needed': 'branches.csv, nodes.csv, edges.csv, shortest path algorithm',
    },
    {
        'column': 'delivery_fee',
        'role': 'delivery charge',
        'main_check': 'must be plausible under fee model',
        'evidence_needed': 'distance, time/date features, loyalty, and regression residuals',
    },
])

display(dirty_column_plan)

print('dirty_data columns covered:', len(dirty_column_plan))
print('actual dirty_data columns:', len(data['dirty'].columns))
print('all columns included:', set(dirty_column_plan['column']) == set(data['dirty'].columns))

,column,role,main_check,evidence_needed
0,order_id,unique order identifier,"format, uniqueness, and prefix consistency",order_id pattern and branch_code relationship
1,date,order date,parseability and valid 2018 date format,datetime parsing and assignment rules
2,time,order time,parseability and meal time window consistency,"Breakfast, Lunch, Dinner time rules"
3,order_type,meal category,must match time window,time column and meal time rules
4,branch_code,restaurant branch code,"legal code, case consistency, and order_id prefix consistency",branches.csv and order_id prefix mapping
5,order_items,items and quantities in the order,"valid item names, quantities, and menu-price consistency",menu prices inferred from reliable rows
6,order_price,total food price before delivery,must equal sum of item price times quantity,order_items and inferred menu prices
7,customer_lat,customer latitude,valid coordinate and distance consistency,nodes.csv and graph nearest-node logic
8,customer_lon,customer longitude,valid coordinate and distance consistency,nodes.csv and graph nearest-node logic
9,customerHasloyalty?,loyalty indicator,binary value and delivery-fee consistency,allowed values and delivery fee model


dirty_data columns covered: 12
actual dirty_data columns: 12
all columns included: True


### Cell 5.5 讨论：为什么 dirty_data 要逐列检查？

**Q1：为什么不能只检查我们已经怀疑有问题的列？**  
A：因为 dirty data 的目标是 detect and fix errors。我们不知道错误分布在哪些列，所以要先让每一列都有明确检查策略。

**Q2：结合 unit 内容，常见 anomaly 有哪些？**  
A：常见类型包括 syntactic errors（格式错误）、semantic errors（业务含义错误）、integrity constraint violations（唯一性/外键/范围约束错误）、functional dependency violations（字段依赖关系错误）、coverage anomalies（缺失/覆盖不足）和 outliers。不同 anomaly 要用不同证据判断。

**Q3：是不是每一列都一定会有错误？**  
A：不一定。逐列检查不是说每列都要改，而是每列都要有证据证明“需要修”或“不需要修”。

**Q4：检查顺序应该怎么安排？**  
A：先检查唯一标识和基础规则字段，例如 `order_id`、`branch_code`、`date`、`time`、`order_type`；再检查依赖更强的字段，例如 `order_items`、`order_price`、`distance_to_customer_KM`、`delivery_fee`。

**Q5：这一步如何对应 marking rubric？**  
A：rubric 不只看输出是否对，也看 methodology 和 documentation。逐列路线图可以展示我们是系统性检测，而不是碰巧修了几个明显错误。

**下一步我们开始第一列检查：**  
先从 `order_id` 入手，因为它既是唯一标识，也包含后续判断 `branch_code` 的 prefix 线索。

## Cell 5.6：关键字段的 domain sanity checks

在逐列修复之前，除了 dtype/missing/unique count，我们还应该做 domain sanity checks。

这些检查不是最终修复，而是早期预警：某些值虽然类型正确、没有缺失，但违反业务常识或字段定义。

In [6]:
domain_check = data['dirty'].copy()

domain_check['customer_lat_in_melbourne_range'] = domain_check['customer_lat'].between(-38.5, -37.0)
domain_check['customer_lon_in_melbourne_range'] = domain_check['customer_lon'].between(144.0, 146.0)
domain_check['order_price_non_negative'] = domain_check['order_price'] >= 0
domain_check['distance_non_negative'] = domain_check['distance_to_customer_KM'] >= 0
domain_check['loyalty_is_binary'] = domain_check['customerHasloyalty?'].isin([0, 1])

domain_summary = pd.DataFrame([
    {
        'check': 'customer_lat outside Melbourne-like range',
        'failed_rows': int((~domain_check['customer_lat_in_melbourne_range']).sum()),
    },
    {
        'check': 'customer_lon outside Melbourne-like range',
        'failed_rows': int((~domain_check['customer_lon_in_melbourne_range']).sum()),
    },
    {
        'check': 'negative order_price',
        'failed_rows': int((~domain_check['order_price_non_negative']).sum()),
    },
    {
        'check': 'negative distance_to_customer_KM',
        'failed_rows': int((~domain_check['distance_non_negative']).sum()),
    },
    {
        'check': 'customerHasloyalty? not binary',
        'failed_rows': int((~domain_check['loyalty_is_binary']).sum()),
    },
])

display(domain_summary)

display(
    domain_check.loc[
        (~domain_check['customer_lat_in_melbourne_range']) |
        (~domain_check['customer_lon_in_melbourne_range']) |
        (~domain_check['order_price_non_negative']) |
        (~domain_check['distance_non_negative']) |
        (~domain_check['loyalty_is_binary']),
        [
            'order_id',
            'customer_lat',
            'customer_lon',
            'order_price',
            'distance_to_customer_KM',
            'customerHasloyalty?',
        ]
    ]
)

print('customer_lat range:', domain_check['customer_lat'].min(), 'to', domain_check['customer_lat'].max())
print('customer_lon range:', domain_check['customer_lon'].min(), 'to', domain_check['customer_lon'].max())

,check,failed_rows
0,customer_lat outside Melbourne-like range,41
1,customer_lon outside Melbourne-like range,4
2,negative order_price,0
3,negative distance_to_customer_KM,0
4,customerHasloyalty? not binary,0


,order_id,customer_lat,customer_lon,order_price,distance_to_customer_KM,customerHasloyalty?
32,ORDZ03318,37.8062,144.9395,312.4000,9.9510,0
38,ORDZ06323,37.8142,144.9610,666.8000,8.2150,0
57,ORDK02131,37.8056,144.9487,465.5000,8.7280,0
62,ORDX01429,37.8142,144.9503,448.2500,10.6950,0
66,ORDA02101,37.8222,145.0041,703.2000,4.9750,0
78,ORDB06092,37.8083,144.9585,308.0000,8.9280,0
79,ORDB00776,37.8103,144.9471,474.0000,9.6130,0
93,ORDZ02223,37.8243,144.9544,512.0000,9.8940,0
111,ORDY03043,37.8120,144.9514,169.0000,9.1430,0
123,ORDA09210,37.8117,145.0121,311.0000,3.2340,0


customer_lat range: -37.832819 to 145.0073992
customer_lon range: -37.8246287 to 145.0183699


### Cell 5.6 讨论：为什么 dtype 正确也可能是错？

**Q1：为什么这一步能更早发现 latitude sign error？**  
A：因为 `37.x` 是合法 float，也没有 missing，但对 Melbourne 来说纬度应该接近 `-37.x`。只有 domain range check 才会把它标出来。

**Q2：domain sanity check 和最终 anomaly detection 有什么区别？**  
A：domain sanity check 是早期预警，不一定直接修。最终修复还要结合 tracker、assignment 规则和字段依赖关系。

**Q3：为什么检查 `customerHasloyalty?` 是否 binary？**  
A：这是 integrity/range constraint。即使 guide 后面说 delivery fee 与 loyalty 相关，首先也要确认 loyalty 字段本身只取 0/1。

**Q4：这个 cell 对教学有什么帮助？**  
A：它提醒学生：EDA 不能只依赖 `describe()`。不同字段有自己的业务范围和语义规则。

**下一步：**  
继续按逐列路线图检查，但带着这些 domain warnings 回看后续修复。

## Cell 6：检查 order_id 的格式、唯一性和 prefix

第一列先检查 `order_id`。

`order_id` 至少有三个作用：

1. 它应该唯一标识一条订单记录；
2. 它应该符合稳定格式；
3. 它的 prefix 后面可能用于检查 `branch_code`。

In [7]:
dirty = data['dirty'].copy()

order_id_pattern = r'^ORD[A-Z][0-9]{5}$'

dirty_order_id_check = dirty.copy()
dirty_order_id_check['order_id_format_valid'] = dirty_order_id_check['order_id'].astype(str).str.match(order_id_pattern)
dirty_order_id_check['order_prefix'] = dirty_order_id_check['order_id'].astype(str).str.extract(r'^ORD([A-Z])')
dirty_order_id_check['order_id_is_duplicate'] = dirty_order_id_check['order_id'].duplicated(keep=False)

order_id_summary = pd.DataFrame([
    {
        'check': 'rows',
        'value': len(dirty_order_id_check),
    },
    {
        'check': 'unique order_id count',
        'value': dirty_order_id_check['order_id'].nunique(dropna=False),
    },
    {
        'check': 'invalid order_id format rows',
        'value': int((~dirty_order_id_check['order_id_format_valid']).sum()),
    },
    {
        'check': 'duplicate order_id rows',
        'value': int(dirty_order_id_check['order_id_is_duplicate'].sum()),
    },
    {
        'check': 'missing extracted prefix rows',
        'value': int(dirty_order_id_check['order_prefix'].isna().sum()),
    },
])

display(order_id_summary)

prefix_counts = (
    dirty_order_id_check['order_prefix']
    .value_counts(dropna=False)
    .rename_axis('order_prefix')
    .reset_index(name='count')
    .sort_values('order_prefix')
)

display(prefix_counts)

display(
    dirty_order_id_check.loc[
        (~dirty_order_id_check['order_id_format_valid']) |
        dirty_order_id_check['order_id_is_duplicate'] |
        dirty_order_id_check['order_prefix'].isna(),
        ['order_id', 'order_id_format_valid', 'order_prefix', 'order_id_is_duplicate']
    ]
)

print('order_id format pattern:', order_id_pattern)

,check,value
0,rows,500
1,unique order_id count,500
2,invalid order_id format rows,0
3,duplicate order_id rows,0
4,missing extracted prefix rows,0


,order_prefix,count
3,A,55
0,B,72
8,C,45
6,I,48
1,J,65
5,K,53
4,X,54
2,Y,62
7,Z,46


,order_id,order_id_format_valid,order_prefix,order_id_is_duplicate


order_id format pattern: ^ORD[A-Z][0-9]{5}$


### Cell 6 讨论：order_id 可能有哪些 anomaly？

**Q1：为什么先检查 `order_id`？**  
A：因为它是记录级 identifier。如果 `order_id` 重复或格式错误，后面用它追踪 anomaly flag、提取 prefix、解释修复记录都会受到影响。

**Q2：这一格检查了哪些 anomaly？**  
A：检查了 syntactic error（格式是否符合 `ORD + 大写字母 + 5位数字`）、integrity constraint violation（是否重复）、以及 prefix extraction 是否成功。

**Q3：如果出现重复 order_id，应该怎么办？**  
A：先不要直接删除。dirty task 是 fix errors，不是 remove rows。要结合整行内容判断是 identifier 写错，还是重复记录。由于 assignment 说每行最多一个 anomaly，重复 ID 会成为候选 anomaly，需要后续定位唯一修复。

**Q4：如果当前 output 没有 order_id 问题，是否还要写在 notebook 里？**  
A：要写。因为逐列检查不仅是为了发现错误，也是为了证明某些列通过了检查。

**下一步：**  
如果 `order_id` 格式和唯一性通过，我们就可以安全使用它的 prefix 来检查 `branch_code`。

## Cell 6.1：打印 order_id suspects 并初始化 dirty anomaly tracker

在继续检查下一列之前，我们先把 `order_id` 的候选异常记录打印出来。

同时初始化一个行级 anomaly tracker。因为 assignment guide 说 dirty data 每行最多只有一个 anomaly，所以后面每发现一类错误，都要先检查这行是否已经被标记过。

In [8]:
order_id_suspects = dirty_order_id_check.loc[
    (~dirty_order_id_check['order_id_format_valid']) |
    dirty_order_id_check['order_id_is_duplicate'] |
    dirty_order_id_check['order_prefix'].isna()
].copy()

order_id_suspects['suspect_reason'] = ''
order_id_suspects.loc[~order_id_suspects['order_id_format_valid'], 'suspect_reason'] += 'invalid_format;'
order_id_suspects.loc[order_id_suspects['order_id_is_duplicate'], 'suspect_reason'] += 'duplicate_order_id;'
order_id_suspects.loc[order_id_suspects['order_prefix'].isna(), 'suspect_reason'] += 'missing_prefix;'
order_id_suspects['suspect_reason'] = order_id_suspects['suspect_reason'].str.rstrip(';')

display(
    order_id_suspects[
        ['order_id', 'order_id_format_valid', 'order_prefix', 'order_id_is_duplicate', 'suspect_reason']
    ]
)

dirty_issue_flags = pd.DataFrame({
    'order_id': dirty_order_id_check['order_id'],
    'issue_column': pd.Series([pd.NA] * len(dirty_order_id_check), dtype='object'),
    'issue_type': pd.Series([pd.NA] * len(dirty_order_id_check), dtype='object'),
    'evidence': pd.Series([pd.NA] * len(dirty_order_id_check), dtype='object'),
}).set_index('order_id')

for _, row in order_id_suspects.iterrows():
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'order_id'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = row['suspect_reason']
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"format_valid={row['order_id_format_valid']}; "
        f"prefix={row['order_prefix']}; "
        f"duplicate={row['order_id_is_duplicate']}"
    )

tracker_summary = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary)

print('order_id suspect records:', len(order_id_suspects))
print('currently flagged rows:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,order_id_format_valid,order_prefix,order_id_is_duplicate,suspect_reason


,issue_column,issue_type,row_count
0,unflagged,unflagged,500


order_id suspect records: 0
currently flagged rows: 0


### Cell 6.1 讨论：order_id 检查之后我们得到什么？

**Q1：为什么要打印 suspects，即使可能是空表？**  
A：空表也是证据。它说明我们检查过 `order_id` 的格式、重复和 prefix 提取，当前没有发现需要修复的记录。

**Q2：为什么现在就初始化 `dirty_issue_flags`？**  
A：因为后面每一列都要遵守“每行最多一个 anomaly”。从第一列开始建立 tracker，可以避免后面重复给同一行打多个错误标签。

**Q3：如果 `order_id_suspects` 是空的，说明什么？**  
A：说明 `order_id` 这一列暂时通过基础检查：格式可解析、没有重复、prefix 可用。于是我们可以更放心地用 prefix 去辅助检查 `branch_code`。

**Q4：下一步怎么用这个 tracker？**  
A：检查 `branch_code` 时，如果发现 suspect records，要先和 `dirty_issue_flags` 对照。若某行已经被 `order_id` 标记，就不能直接再标成 `branch_code`，必须重新判断唯一 anomaly 到底在哪一列。

**下一步：**  
进入 `branch_code`，先检查合法值和大小写，再检查它是否和 `order_id` prefix 一致。

## Cell 6.5：dirty_data 的 branch_code 初步检查

我们从 `branch_code` 开始，因为它会影响后面的距离计算。这里先不修复，只观察：

- 原始 `branch_code` 有哪些取值；
- 是否存在大小写不一致；
- 与 `order_id` 前缀组合后是否出现可疑不一致。

In [9]:
dirty = data['dirty'].copy()

dirty_branch_profile = (
    dirty['branch_code']
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis('branch_code_raw')
    .reset_index(name='count')
)

dirty_branch_profile['branch_code_upper'] = dirty_branch_profile['branch_code_raw'].str.upper()

display(dirty_branch_profile)

dirty_branch_prefix = dirty.copy()
dirty_branch_prefix['order_prefix'] = dirty_branch_prefix['order_id'].str.extract(r'^ORD([A-Z])')
dirty_branch_prefix['branch_code_upper'] = dirty_branch_prefix['branch_code'].astype(str).str.upper()

prefix_branch_counts = (
    dirty_branch_prefix
    .groupby(['order_prefix', 'branch_code_upper'])
    .size()
    .reset_index(name='count')
    .sort_values(['order_prefix', 'count'], ascending=[True, False])
)

display(prefix_branch_counts)

print('raw branch_code values:', sorted(dirty['branch_code'].astype(str).unique()))
print('upper-cased branch_code values:', sorted(dirty_branch_prefix['branch_code_upper'].unique()))

,branch_code_raw,count,branch_code_upper
0,TP,188,TP
1,BK,150,BK
2,NS,134,NS
3,tp,10,TP
4,ns,10,NS
5,bk,8,BK


,order_prefix,branch_code_upper,count
0,A,BK,53
1,A,NS,1
2,A,TP,1
4,B,TP,69
3,B,NS,3
5,C,NS,43
6,C,TP,2
8,I,NS,43
9,I,TP,4
7,I,BK,1


raw branch_code values: ['BK', 'NS', 'TP', 'bk', 'ns', 'tp']
upper-cased branch_code values: ['BK', 'NS', 'TP']


### Cell 6.5 讨论：dirty_data 的 branch_code 暗示了什么？

**Q1：这个 output 里第一眼要看什么？**  
A：先看 `branch_code_raw`。如果同一个分店 code 同时出现 `BK` 和 `bk`，说明至少存在大小写格式错误。

**Q2：为什么要同时看 `order_prefix` 和 `branch_code_upper`？**  
A：大小写统一只能修复格式问题。如果统一大小写后，同一个 `order_prefix` 仍对应多个 branch code，就说明可能存在真正的分店编码错误。

**Q3：现在能不能直接修？**  
A：还不能。我们现在只确认 dirty 里有可疑 branch_code 问题。下一步要建立判断标准：哪些 code 是合法值？`order_id` prefix 是否稳定对应某个 branch？

**Q4：为什么这一步符合 assignment guide？**  
A：因为我们现在只在做 dirty_data 的 error detection。我们还没有使用 missing_data，也没有做 imputation。

**下一步我们应该问：**  
`branches.csv` 里定义了哪些合法 branch code？`dirty_data` 中哪些 branch_code 只是大小写错误，哪些可能与 `order_id` 前缀不一致？

## Cell 7：检查 branch_code 的合法值和大小写问题

我们正式开始逐列检查。第一列先看 `branch_code`。

这一格只检查最基础的规则：`branch_code` 是否属于 `branches.csv` 里定义的合法分店 code，以及是否存在大小写不一致。这里暂时不检查 `order_id` prefix 是否匹配。

In [10]:
dirty_working = data['dirty'].copy()

valid_branch_codes = set(data['branches']['branch_code'].astype(str))
valid_branch_codes_upper = set(data['branches']['branch_code'].astype(str).str.upper())

dirty_working['branch_code_upper'] = dirty_working['branch_code'].astype(str).str.upper()

dirty_working['branch_code_is_raw_valid'] = dirty_working['branch_code'].astype(str).isin(valid_branch_codes)
dirty_working['branch_code_is_valid_after_upper'] = dirty_working['branch_code_upper'].isin(valid_branch_codes_upper)
dirty_working['branch_code_has_case_issue'] = (
    dirty_working['branch_code'].astype(str) != dirty_working['branch_code_upper']
)

branch_code_validation_summary = pd.DataFrame([
    {
        'check': 'valid branch codes from branches.csv',
        'value': sorted(valid_branch_codes),
    },
    {
        'check': 'rows with raw invalid branch_code',
        'value': int((~dirty_working['branch_code_is_raw_valid']).sum()),
    },
    {
        'check': 'rows still invalid after upper-case normalization',
        'value': int((~dirty_working['branch_code_is_valid_after_upper']).sum()),
    },
    {
        'check': 'rows with lower-case branch_code issue',
        'value': int(dirty_working['branch_code_has_case_issue'].sum()),
    },
])

display(branch_code_validation_summary)

display(
    dirty_working.loc[
        dirty_working['branch_code_has_case_issue'],
        ['order_id', 'branch_code', 'branch_code_upper']
    ].head(15)
)

print('raw branch_code values:', sorted(dirty_working['branch_code'].astype(str).unique()))
print('upper-case branch_code values:', sorted(dirty_working['branch_code_upper'].unique()))

,check,value
0,valid branch codes from branches.csv,"[BK, NS, TP]"
1,rows with raw invalid branch_code,28
2,rows still invalid after upper-case normalization,0
3,rows with lower-case branch_code issue,28


,order_id,branch_code,branch_code_upper
4,ORDK03173,tp,TP
31,ORDK06897,ns,NS
33,ORDA05281,tp,TP
48,ORDI09968,tp,TP
52,ORDK07377,ns,NS
85,ORDY03742,tp,TP
86,ORDA01223,ns,NS
98,ORDB00412,tp,TP
100,ORDX08256,ns,NS
153,ORDJ10549,bk,BK


raw branch_code values: ['BK', 'NS', 'TP', 'bk', 'ns', 'tp']
upper-case branch_code values: ['BK', 'NS', 'TP']


### Cell 7 讨论：branch_code 的第一类错误是什么？

**Q1：这一步检查了什么？**  
A：只检查 `branch_code` 自身是否合法。合法 code 来自 `branches.csv`，所以 dirty 文件里的 code 应该是 `BK`、`NS`、`TP` 这类标准形式。

**Q2：为什么要比较 raw valid 和 upper-case valid？**  
A：如果 raw invalid 但 upper-case 后 valid，说明问题主要是大小写格式错误。例如 `bk` 本身不是标准形式，但转成 `BK` 后就是合法分店 code。

**Q3：如果 upper-case 后仍然 invalid，说明什么？**  
A：那就不是简单大小写问题，可能是不存在的分店 code，需要进一步检查。但当前 output 可以帮助我们判断有没有这种情况。

**Q4：这一步能不能直接完成 branch_code 修复？**  
A：还不能。大小写只是第一层。下一步还要检查 `order_id` prefix 和 `branch_code` 是否一致，因为一个 code 即使是合法值，也可能填错分店。

**下一步我们应该问：**  
`order_id` 前缀和 `branch_code` 是否存在稳定关系？如果存在，哪些 dirty rows 的合法 branch code 其实对应错了？

## Cell 8：检查 order_id prefix 与 branch_code 的一致性

上一格只解决了 `branch_code` 的大小写格式问题。现在我们检查更深一层：一个格式合法的 `branch_code` 是否和 `order_id` 的 prefix 一致。

因为 dirty 文件本身可能包含错误，所以这一格会先用 dirty 中的多数规律建立一个候选 mapping，再检查哪些记录偏离这个 mapping。

In [11]:
dirty_prefix_check = dirty_working.copy()
dirty_prefix_check['order_prefix'] = dirty_prefix_check['order_id'].str.extract(r'^ORD([A-Z])')

prefix_branch_frequency = (
    dirty_prefix_check
    .groupby(['order_prefix', 'branch_code_upper'])
    .size()
    .reset_index(name='count')
    .sort_values(['order_prefix', 'count'], ascending=[True, False])
)

display(prefix_branch_frequency)

candidate_prefix_mapping = (
    prefix_branch_frequency
    .sort_values(['order_prefix', 'count'], ascending=[True, False])
    .drop_duplicates('order_prefix')
    .rename(columns={'branch_code_upper': 'expected_branch_code'})
    [['order_prefix', 'expected_branch_code', 'count']]
)

display(candidate_prefix_mapping)

prefix_to_expected_branch = dict(
    zip(candidate_prefix_mapping['order_prefix'], candidate_prefix_mapping['expected_branch_code'])
)

dirty_prefix_check['expected_branch_code'] = dirty_prefix_check['order_prefix'].map(prefix_to_expected_branch)
dirty_prefix_check['branch_code_matches_prefix'] = (
    dirty_prefix_check['branch_code_upper'] == dirty_prefix_check['expected_branch_code']
)

branch_prefix_summary = pd.DataFrame([
    {
        'check': 'unique order prefixes',
        'value': dirty_prefix_check['order_prefix'].nunique(),
    },
    {
        'check': 'rows matching prefix mapping',
        'value': int(dirty_prefix_check['branch_code_matches_prefix'].sum()),
    },
    {
        'check': 'rows not matching prefix mapping',
        'value': int((~dirty_prefix_check['branch_code_matches_prefix']).sum()),
    },
])

display(branch_prefix_summary)

display(
    dirty_prefix_check.loc[
        ~dirty_prefix_check['branch_code_matches_prefix'],
        ['order_id', 'order_prefix', 'branch_code', 'branch_code_upper', 'expected_branch_code']
    ].head(20)
)

print('candidate prefix mapping:', prefix_to_expected_branch)

,order_prefix,branch_code_upper,count
0,A,BK,53
1,A,NS,1
2,A,TP,1
4,B,TP,69
3,B,NS,3
5,C,NS,43
6,C,TP,2
8,I,NS,43
9,I,TP,4
7,I,BK,1


,order_prefix,expected_branch_code,count
0,A,BK,53
4,B,TP,69
5,C,NS,43
8,I,NS,43
12,J,TP,62
13,K,BK,48
16,X,BK,52
20,Y,TP,57
21,Z,NS,45


,check,value
0,unique order prefixes,9
1,rows matching prefix mapping,472
2,rows not matching prefix mapping,28


,order_id,order_prefix,branch_code,branch_code_upper,expected_branch_code
4,ORDK03173,K,tp,TP,BK
21,ORDI09297,I,BK,BK,NS
22,ORDC07228,C,TP,TP,NS
31,ORDK06897,K,ns,NS,BK
33,ORDA05281,A,tp,TP,BK
48,ORDI09968,I,tp,TP,NS
52,ORDK07377,K,ns,NS,BK
74,ORDK04119,K,NS,NS,BK
86,ORDA01223,A,ns,NS,BK
100,ORDX08256,X,ns,NS,BK


candidate prefix mapping: {'A': 'BK', 'B': 'TP', 'C': 'NS', 'I': 'NS', 'J': 'TP', 'K': 'BK', 'X': 'BK', 'Y': 'TP', 'Z': 'NS'}


### Cell 8 讨论：如何判断合法 code 也可能是错误 code？

**Q1：为什么合法 branch code 还可能是错的？**  
A：因为 `BK`、`NS`、`TP` 都是合法值，但某一条订单应该属于哪个分店，还要看 `order_id` prefix 的规律。如果 `ORDX` 大多数稳定对应 `BK`，某条 `ORDX` 写成 `NS` 就很可疑。

**Q2：为什么这里用多数规律建立 candidate mapping？**  
A：dirty data 每行最多只有一个 anomaly，而且错误通常是少数。对同一个 prefix 来说，出现次数最多的 branch code 往往是正确规律，少数偏离值就是候选错误。

**Q3：这一步有没有真正修复数据？**  
A：还没有。我们只是在 detect suspicious rows。下一步才会决定如何修复：先统一大小写，再把不匹配 prefix 的 branch code 改成 expected branch code。

**Q4：这一步要注意什么风险？**  
A：多数规律是一种数据证据，但最好还要和后续 distance、branches.csv、以及 assignment 规则互相验证。不能只因为一个值少就盲目改，应该看它是否违反稳定映射。

**下一步我们应该问：**  
如果每行最多只有一个 anomaly，那么修复 `branch_code` 时应该怎样避免误改其他列？修完后如何验证 branch_code 全部合法且与 prefix 一致？

## Cell 8.5：打印 branch_code 候选异常 records

在修复之前，我们先把怀疑有问题的 records 打印出来。

这里我们把问题分成两类：

1. `case_issue_only`：branch code 只是大小写不标准，但 upper-case 后和 expected branch code 一致。
2. `prefix_mismatch`：branch code upper-case 后仍然和 expected branch code 不一致。

In [12]:
branch_suspects = dirty_prefix_check.loc[
    dirty_prefix_check['branch_code_has_case_issue'] |
    (~dirty_prefix_check['branch_code_matches_prefix'])
].copy()

branch_suspects['suspect_reason'] = np.select(
    [
        branch_suspects['branch_code_has_case_issue'] & branch_suspects['branch_code_matches_prefix'],
        ~branch_suspects['branch_code_matches_prefix'],
    ],
    [
        'case_issue_only',
        'prefix_mismatch',
    ],
    default='review_needed'
)

branch_suspect_columns = [
    'order_id',
    'order_prefix',
    'branch_code',
    'branch_code_upper',
    'expected_branch_code',
    'suspect_reason',
]

branch_suspect_summary = (
    branch_suspects['suspect_reason']
    .value_counts()
    .rename_axis('suspect_reason')
    .reset_index(name='record_count')
)

display(branch_suspect_summary)

display(
    branch_suspects[branch_suspect_columns]
    .sort_values(['suspect_reason', 'order_prefix', 'order_id'])
)

print('total branch_code suspect records:', len(branch_suspects))

,suspect_reason,record_count
0,prefix_mismatch,28
1,case_issue_only,9


,order_id,order_prefix,branch_code,branch_code_upper,expected_branch_code,suspect_reason
284,ORDA07364,A,bk,BK,BK,case_issue_only
286,ORDA09899,A,bk,BK,BK,case_issue_only
98,ORDB00412,B,tp,TP,TP,case_issue_only
169,ORDJ07435,J,tp,TP,TP,case_issue_only
268,ORDJ09868,J,tp,TP,TP,case_issue_only
377,ORDK03706,K,bk,BK,BK,case_issue_only
495,ORDX01661,X,bk,BK,BK,case_issue_only
247,ORDX04643,X,bk,BK,BK,case_issue_only
85,ORDY03742,Y,tp,TP,TP,case_issue_only
86,ORDA01223,A,ns,NS,BK,prefix_mismatch


total branch_code suspect records: 37


### Cell 8.5 讨论：修复前为什么要打印 suspects？

**Q1：为什么不直接进入修复？**  
A：因为教学和报告里需要展示 evidence。先打印 suspects，可以让学生看到我们到底准备改哪些 records，以及每条为什么被怀疑。

**Q2：为什么要区分 `case_issue_only` 和 `prefix_mismatch`？**  
A：这两类问题的严重程度不同。`case_issue_only` 通常只需要标准化大小写；`prefix_mismatch` 表示 branch code 即使合法，也可能指向了错误分店。

**Q3：这些 suspects 是否一定都是 branch_code 错？**  
A：在 assignment 规则下，dirty data 每行最多一个 anomaly，并且每个 anomaly 有唯一修复。结合稳定 prefix mapping，这些 records 是 branch_code 错误的强候选。但正式修复后仍要做 validation。

**Q4：下一步修复时要做什么？**  
A：只改这些 branch_code suspect records 的 `branch_code`，不要碰同一行的其他字段。修完后重新检查合法值、大小写和 prefix 一致性。

## Cell 8.6：把 branch_code suspects 加入 dirty anomaly tracker

现在我们已经有了 `dirty_issue_flags`。这一格不重新创建 tracker，而是把 `branch_code` suspects 加进去。

关键检查是：这些 branch_code suspects 里有没有 records 已经被前面的 `order_id` 检查标记过。如果有，说明同一行出现多个候选问题，需要回头判断唯一 anomaly；如果没有，就可以安全标记为 `branch_code` 问题。

In [13]:
branch_suspect_flags = branch_suspects.set_index('order_id')

branch_suspect_order_ids = branch_suspect_flags.index
already_flagged_branch_suspects = dirty_issue_flags.loc[
    dirty_issue_flags.index.intersection(branch_suspect_order_ids)
].dropna(subset=['issue_column'])

display(already_flagged_branch_suspects.reset_index())

for order_id, row in branch_suspect_flags.iterrows():
    if pd.isna(dirty_issue_flags.loc[order_id, 'issue_column']):
        dirty_issue_flags.loc[order_id, 'issue_column'] = 'branch_code'
        dirty_issue_flags.loc[order_id, 'issue_type'] = row['suspect_reason']
        dirty_issue_flags.loc[order_id, 'evidence'] = (
            f"prefix={row['order_prefix']}; "
            f"observed={row['branch_code']}; "
            f"expected={row['expected_branch_code']}"
        )

flag_summary = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(flag_summary)

display(
    dirty_issue_flags
    .dropna(subset=['issue_column'])
    .reset_index()
    .head(30)
)

print('branch_code suspect records:', len(branch_suspects))
print('branch_code suspects already flagged before:', len(already_flagged_branch_suspects))
print('total flagged rows:', int(dirty_issue_flags['issue_column'].notna().sum()))
print('unflagged rows:', int(dirty_issue_flags['issue_column'].isna().sum()))

,order_id,issue_column,issue_type,evidence


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,unflagged,unflagged,463


,order_id,issue_column,issue_type,evidence
0,ORDK03173,branch_code,prefix_mismatch,prefix=K; observed=tp; expected=BK
1,ORDI09297,branch_code,prefix_mismatch,prefix=I; observed=BK; expected=NS
2,ORDC07228,branch_code,prefix_mismatch,prefix=C; observed=TP; expected=NS
3,ORDK06897,branch_code,prefix_mismatch,prefix=K; observed=ns; expected=BK
4,ORDA05281,branch_code,prefix_mismatch,prefix=A; observed=tp; expected=BK
5,ORDI09968,branch_code,prefix_mismatch,prefix=I; observed=tp; expected=NS
6,ORDK07377,branch_code,prefix_mismatch,prefix=K; observed=ns; expected=BK
7,ORDK04119,branch_code,prefix_mismatch,prefix=K; observed=NS; expected=BK
8,ORDY03742,branch_code,case_issue_only,prefix=Y; observed=tp; expected=TP
9,ORDA01223,branch_code,prefix_mismatch,prefix=A; observed=ns; expected=BK


branch_code suspect records: 37
branch_code suspects already flagged before: 0
total flagged rows: 37
unflagged rows: 463


### Cell 8.6 讨论：如何避免同一行被标记多个 anomaly？

**Q1：为什么这一格不重新创建 tracker？**  
A：因为 tracker 应该贯穿整个 dirty cleaning。`order_id` 检查已经初始化了它，后面每一列都应该在同一个表上追加信息。

**Q2：为什么要先检查 `already_flagged_branch_suspects`？**  
A：assignment guide 说每行最多一个 anomaly。如果某条 branch suspect 已经被 `order_id` 标记过，我们不能直接给它第二个错误标签，而要回头判断到底哪个字段才是唯一错误。

**Q3：如果 `already_flagged_branch_suspects` 是空表，说明什么？**  
A：说明当前 branch_code suspects 没有和前面的 order_id suspects 冲突，可以把这些 records 标记为 `branch_code` 问题。

**Q4：后面检查其他列也要这样做吗？**  
A：要。每一列都应该先打印 suspects，再检查这些 suspects 是否已经被 flag。这样可以系统性遵守“一行最多一个 anomaly”的规则。

**下一步：**  
我们可以修复已标记的 `branch_code` records，然后 validation；修复后仍然保留 flag 表，作为后续列检查的排除和解释依据。

## Cell 9：修复 branch_code 并验证

现在我们只修复已经被 `dirty_issue_flags` 标记为 `branch_code` 的 records。

修复原则：

- 不改未被标记的行；
- 不改同一行的其他列；
- `case_issue_only` 和 `prefix_mismatch` 都统一修成 `expected_branch_code`；
- 修完立即验证合法值、大小写和 prefix 一致性。

In [14]:
dirty_cleaning = data['dirty'].copy()

branch_fix_order_ids = dirty_issue_flags.index[
    dirty_issue_flags['issue_column'] == 'branch_code'
]

branch_fix_lookup = branch_suspects.set_index('order_id')['expected_branch_code'].to_dict()

branch_fix_audit = dirty_cleaning.loc[
    dirty_cleaning['order_id'].isin(branch_fix_order_ids),
    ['order_id', 'branch_code']
].copy()
branch_fix_audit['fixed_branch_code'] = branch_fix_audit['order_id'].map(branch_fix_lookup)

for order_id in branch_fix_order_ids:
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == order_id,
        'branch_code'
    ] = branch_fix_lookup[order_id]

display(branch_fix_audit)

post_branch_check = dirty_cleaning.copy()
post_branch_check['order_prefix'] = post_branch_check['order_id'].str.extract(r'^ORD([A-Z])')
post_branch_check['branch_code_upper'] = post_branch_check['branch_code'].astype(str).str.upper()
post_branch_check['expected_branch_code'] = post_branch_check['order_prefix'].map(prefix_to_expected_branch)
post_branch_check['branch_code_is_raw_valid'] = post_branch_check['branch_code'].astype(str).isin(valid_branch_codes)
post_branch_check['branch_code_matches_prefix'] = (
    post_branch_check['branch_code_upper'] == post_branch_check['expected_branch_code']
)

branch_fix_validation = pd.DataFrame([
    {
        'check': 'fixed branch_code records',
        'value': len(branch_fix_audit),
    },
    {
        'check': 'raw invalid branch_code rows after fix',
        'value': int((~post_branch_check['branch_code_is_raw_valid']).sum()),
    },
    {
        'check': 'prefix mismatch rows after fix',
        'value': int((~post_branch_check['branch_code_matches_prefix']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(branch_fix_validation)

display(
    post_branch_check.loc[
        (~post_branch_check['branch_code_is_raw_valid']) |
        (~post_branch_check['branch_code_matches_prefix']),
        ['order_id', 'order_prefix', 'branch_code', 'branch_code_upper', 'expected_branch_code']
    ]
)

print('dirty_cleaning shape:', dirty_cleaning.shape)

,order_id,branch_code,fixed_branch_code
4,ORDK03173,tp,BK
21,ORDI09297,BK,NS
22,ORDC07228,TP,NS
31,ORDK06897,ns,BK
33,ORDA05281,tp,BK
48,ORDI09968,tp,NS
52,ORDK07377,ns,BK
74,ORDK04119,NS,BK
85,ORDY03742,tp,TP
86,ORDA01223,ns,BK


,check,value
0,fixed branch_code records,37
1,raw invalid branch_code rows after fix,0
2,prefix mismatch rows after fix,0
3,row count preserved,True


,order_id,order_prefix,branch_code,branch_code_upper,expected_branch_code


dirty_cleaning shape: (500, 12)


### Cell 9 讨论：branch_code 修复是否完成？

**Q1：这一步改了哪些 records？**  
A：只改 `dirty_issue_flags` 中 `issue_column == 'branch_code'` 的 records。`branch_fix_audit` 显示了原始 `branch_code` 和修复后的 `fixed_branch_code`。

**Q2：为什么 `case_issue_only` 也修成 `expected_branch_code`？**  
A：因为 `expected_branch_code` 是标准大写形式。对于大小写问题，它等价于标准化；对于 prefix mismatch，它同时修复了错误分店 code。

**Q3：validation 要看什么？**  
A：看修复后是否还有 raw invalid branch code，是否还有 prefix mismatch，行数是否保持不变。如果这些都通过，`branch_code` 这一列可以暂时关闭。

**Q4：修完 branch_code 后下一列看什么？**  
A：回到逐列路线图。下一步适合检查 `date`，因为它是格式规则字段，也会影响 delivery fee 的 weekday/weekend 特征。

**下一步：**  
检查 `date` 的 parseability、格式一致性，以及是否存在不合理日期。

## Cell 10：检查 date 的格式和可解析性

下一列检查 `date`。

这一格先做 detection，不修复。我们检查三件事：

1. `date` 是否能被解析；
2. 是否符合标准格式 `YYYY-MM-DD`；
3. 日期是否落在 assignment 背景下的合理年份范围内。

In [15]:
date_check = dirty_cleaning.copy()

date_check['date_as_string'] = date_check['date'].astype(str)
date_check['date_matches_standard_format'] = date_check['date_as_string'].str.match(r'^\d{4}-\d{2}-\d{2}$')
date_check['date_parsed'] = pd.to_datetime(date_check['date_as_string'], errors='coerce', format='mixed')
date_check['date_parse_success'] = date_check['date_parsed'].notna()
date_check['date_year'] = date_check['date_parsed'].dt.year
date_check['date_in_expected_year'] = date_check['date_year'].eq(2018)

date_summary = pd.DataFrame([
    {
        'check': 'rows',
        'value': len(date_check),
    },
    {
        'check': 'unparseable date rows',
        'value': int((~date_check['date_parse_success']).sum()),
    },
    {
        'check': 'non-standard format rows',
        'value': int((~date_check['date_matches_standard_format']).sum()),
    },
    {
        'check': 'rows outside expected year 2018',
        'value': int((~date_check['date_in_expected_year']).sum()),
    },
])

display(date_summary)

nonstandard_date_examples = date_check.loc[
    (~date_check['date_matches_standard_format']) |
    (~date_check['date_parse_success']) |
    (~date_check['date_in_expected_year']),
    ['order_id', 'date', 'date_parsed', 'date_matches_standard_format', 'date_parse_success', 'date_year']
].copy()

display(nonstandard_date_examples)

print('date range after parsing:', date_check['date_parsed'].min(), 'to', date_check['date_parsed'].max())

,check,value
0,rows,500
1,unparseable date rows,20
2,non-standard format rows,17
3,rows outside expected year 2018,20


,order_id,date,date_parsed,date_matches_standard_format,date_parse_success,date_year
0,ORDX00699,03-08-2018,2018-03-08,False,True,"2,018.0000"
10,ORDC01147,2018-26-08,NaT,True,False,NaN
16,ORDK04564,2018-19-09,NaT,True,False,NaN
18,ORDY05205,04-01-2018,2018-04-01,False,True,"2,018.0000"
37,ORDJ05383,2018-18-08,NaT,True,False,NaN
42,ORDJ08299,02-09-2018,2018-02-09,False,True,"2,018.0000"
51,ORDB00774,2018-16-06,NaT,True,False,NaN
70,ORDB07017,04-07-2018,2018-04-07,False,True,"2,018.0000"
81,ORDB06811,2018-28-11,NaT,True,False,NaN
96,ORDZ10150,2018-18-05,NaT,True,False,NaN


date range after parsing: 2018-01-04 00:00:00 to 2018-12-31 00:00:00


### Cell 10 讨论：date 可能是什么类型的错误？

**Q1：这一格检查了哪些 anomaly？**  
A：主要是 syntactic error 和 semantic/range check。比如格式不是 `YYYY-MM-DD` 是格式问题；年份不在 2018 则可能是业务范围问题。

**Q2：为什么非标准格式不一定等于日期错误？**  
A：例如 `03-08-2018` 可以被解析成一个真实日期，但格式不符合标准输出要求。它可能只是格式错误，不一定是日期含义错误。

**Q3：为什么这里要显式使用 `format='mixed'`？**  
A：因为这一列可能混合了 `YYYY-MM-DD` 和 `DD-MM-YYYY` 这类格式。pandas 在整列解析时可能根据某一种格式推断，导致本来看起来正常的 `2018-11-04` 被误判成 `NaT`。`format='mixed'` 可以让 pandas 对每个值分别推断格式。

**Q4：为什么还要先打印 suspects？**  
A：因为修复前要看清楚候选问题到底是什么。尤其是 `03-08-2018` 这种值，可能存在日/月歧义，不能无脑 parse。

**Q5：下一步应该怎么做？**  
A：把 date suspects 和 `dirty_issue_flags` 对照。如果某些 suspects 所在行已经被标成 `branch_code`，就不能再直接标成 `date`。如果没有冲突，再决定如何修复 date 格式。

**下一步：**  
打印 date suspects，检查它们是否和已有 anomaly flags 冲突。

## Cell 10.1：打印 date suspects 并检查 flag 冲突

现在我们把 date suspects 单独打印出来，并检查它们是否已经在前面的步骤中被标记为其他 anomaly。

如果某一行已经有 `branch_code` flag，就不能直接再标成 `date` 问题，因为 dirty data 每行最多只有一个 anomaly。

In [16]:
date_suspects = date_check.loc[
    (~date_check['date_matches_standard_format']) |
    (~date_check['date_parse_success']) |
    (~date_check['date_in_expected_year'])
].copy()

date_suspects['suspect_reason'] = ''
date_suspects.loc[~date_suspects['date_matches_standard_format'], 'suspect_reason'] += 'non_standard_format;'
date_suspects.loc[~date_suspects['date_parse_success'], 'suspect_reason'] += 'unparseable;'
date_suspects.loc[~date_suspects['date_in_expected_year'], 'suspect_reason'] += 'outside_expected_year;'
date_suspects['suspect_reason'] = date_suspects['suspect_reason'].str.rstrip(';')

date_suspect_columns = [
    'order_id',
    'date',
    'date_parsed',
    'date_matches_standard_format',
    'date_parse_success',
    'date_year',
    'suspect_reason',
]

display(date_suspects[date_suspect_columns])

existing_flags_for_date_suspects = dirty_issue_flags.loc[
    dirty_issue_flags.index.intersection(date_suspects['order_id'])
].dropna(subset=['issue_column'])

display(existing_flags_for_date_suspects.reset_index())

unflagged_date_suspects = date_suspects.loc[
    ~date_suspects['order_id'].isin(existing_flags_for_date_suspects.index)
].copy()

display(unflagged_date_suspects[date_suspect_columns])

print('date suspect records:', len(date_suspects))
print('date suspects already flagged before:', len(existing_flags_for_date_suspects))
print('unflagged date suspect records:', len(unflagged_date_suspects))

,order_id,date,date_parsed,date_matches_standard_format,date_parse_success,date_year,suspect_reason
0,ORDX00699,03-08-2018,2018-03-08,False,True,"2,018.0000",non_standard_format
10,ORDC01147,2018-26-08,NaT,True,False,NaN,unparseable;outside_expected_year
16,ORDK04564,2018-19-09,NaT,True,False,NaN,unparseable;outside_expected_year
18,ORDY05205,04-01-2018,2018-04-01,False,True,"2,018.0000",non_standard_format
37,ORDJ05383,2018-18-08,NaT,True,False,NaN,unparseable;outside_expected_year
42,ORDJ08299,02-09-2018,2018-02-09,False,True,"2,018.0000",non_standard_format
51,ORDB00774,2018-16-06,NaT,True,False,NaN,unparseable;outside_expected_year
70,ORDB07017,04-07-2018,2018-04-07,False,True,"2,018.0000",non_standard_format
81,ORDB06811,2018-28-11,NaT,True,False,NaN,unparseable;outside_expected_year
96,ORDZ10150,2018-18-05,NaT,True,False,NaN,unparseable;outside_expected_year


,order_id,issue_column,issue_type,evidence


,order_id,date,date_parsed,date_matches_standard_format,date_parse_success,date_year,suspect_reason
0,ORDX00699,03-08-2018,2018-03-08,False,True,"2,018.0000",non_standard_format
10,ORDC01147,2018-26-08,NaT,True,False,NaN,unparseable;outside_expected_year
16,ORDK04564,2018-19-09,NaT,True,False,NaN,unparseable;outside_expected_year
18,ORDY05205,04-01-2018,2018-04-01,False,True,"2,018.0000",non_standard_format
37,ORDJ05383,2018-18-08,NaT,True,False,NaN,unparseable;outside_expected_year
42,ORDJ08299,02-09-2018,2018-02-09,False,True,"2,018.0000",non_standard_format
51,ORDB00774,2018-16-06,NaT,True,False,NaN,unparseable;outside_expected_year
70,ORDB07017,04-07-2018,2018-04-07,False,True,"2,018.0000",non_standard_format
81,ORDB06811,2018-28-11,NaT,True,False,NaN,unparseable;outside_expected_year
96,ORDZ10150,2018-18-05,NaT,True,False,NaN,unparseable;outside_expected_year


date suspect records: 37
date suspects already flagged before: 0
unflagged date suspect records: 37


### Cell 10.1 讨论：date suspects 是否真的应该修？

**Q1：为什么要看 `existing_flags_for_date_suspects`？**  
A：因为 dirty data 每行最多一个 anomaly。如果某个 date suspect 所在行已经被标记为其他问题，我们需要先比较证据，不能直接重复修。

**Q2：现在 conflict rows 为 0 说明什么？**  
A：说明在使用 `format='mixed'` 正确解析日期后，date suspects 没有和已有 branch_code flags 发生冲突。之前出现的冲突是 parsing 方法造成的误报。

**Q3：这个 debug 给我们什么经验？**  
A：工具输出也要被质疑。看到 `NaT` 不一定代表原始日期错了，可能是解析策略不适合混合日期格式。

**Q4：下一步怎么做？**  
A：对未被 tracker 标记、且确实存在格式问题的 date suspects 建立修复方案，修成标准 `YYYY-MM-DD`，并把这些 order_id 加入 `dirty_issue_flags`。

**下一步：**  
修复真正的 date suspects，并更新 anomaly tracker。

## Cell 10.2：统一 date 格式并 flag 被修改的 rows

现在我们已经确认要用 `format='mixed'` 正确解析混合日期格式。

这一格的目标是：只把格式不统一但可以解析的日期统一成 `YYYY-MM-DD`，并且只 flag 实际被修改的 rows。

In [17]:
date_fix_candidates = unflagged_date_suspects.loc[
    unflagged_date_suspects['date_parse_success'] &
    (~unflagged_date_suspects['date_matches_standard_format'])
].copy()

date_fix_audit = date_fix_candidates[
    ['order_id', 'date', 'date_parsed', 'suspect_reason']
].copy()
date_fix_audit['fixed_date'] = date_fix_audit['date_parsed'].dt.strftime('%Y-%m-%d')
date_fix_audit['date_will_change'] = date_fix_audit['date'].astype(str) != date_fix_audit['fixed_date'].astype(str)
date_fix_audit = date_fix_audit.loc[date_fix_audit['date_will_change']].copy()

display(date_fix_audit)

for _, row in date_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'date'
    ] = row['fixed_date']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'date'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'non_standard_format'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed={row['date']}; fixed={row['fixed_date']}"
    )

post_date_check = dirty_cleaning.copy()
post_date_check['date_as_string'] = post_date_check['date'].astype(str)
post_date_check['date_matches_standard_format'] = post_date_check['date_as_string'].str.match(r'^\d{4}-\d{2}-\d{2}$')
post_date_check['date_parsed'] = pd.to_datetime(post_date_check['date_as_string'], errors='coerce', format='mixed')
post_date_check['date_parse_success'] = post_date_check['date_parsed'].notna()
post_date_check['date_year'] = post_date_check['date_parsed'].dt.year
post_date_check['date_in_expected_year'] = post_date_check['date_year'].eq(2018)

post_date_validation = pd.DataFrame([
    {
        'check': 'fixed date records',
        'value': len(date_fix_audit),
    },
    {
        'check': 'unparseable date rows after fix',
        'value': int((~post_date_check['date_parse_success']).sum()),
    },
    {
        'check': 'non-standard format rows after fix',
        'value': int((~post_date_check['date_matches_standard_format']).sum()),
    },
    {
        'check': 'rows outside expected year 2018 after fix',
        'value': int((~post_date_check['date_in_expected_year']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_date_validation)

display(
    post_date_check.loc[
        (~post_date_check['date_matches_standard_format']) |
        (~post_date_check['date_parse_success']) |
        (~post_date_check['date_in_expected_year']),
        ['order_id', 'date', 'date_parsed', 'date_matches_standard_format', 'date_parse_success', 'date_year']
    ]
)

tracker_summary_after_date = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_date)

print('total flagged rows after date fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,date,date_parsed,suspect_reason,fixed_date,date_will_change
0,ORDX00699,03-08-2018,2018-03-08,non_standard_format,2018-03-08,True
18,ORDY05205,04-01-2018,2018-04-01,non_standard_format,2018-04-01,True
42,ORDJ08299,02-09-2018,2018-02-09,non_standard_format,2018-02-09,True
70,ORDB07017,04-07-2018,2018-04-07,non_standard_format,2018-04-07,True
116,ORDZ10540,08-07-2018,2018-08-07,non_standard_format,2018-08-07,True
172,ORDK06012,03-09-2018,2018-03-09,non_standard_format,2018-03-09,True
185,ORDX06744,07-08-2018,2018-07-08,non_standard_format,2018-07-08,True
257,ORDJ02804,10-02-2018,2018-10-02,non_standard_format,2018-10-02,True
311,ORDB04450,01-07-2018,2018-01-07,non_standard_format,2018-01-07,True
331,ORDY07409,09-07-2018,2018-09-07,non_standard_format,2018-09-07,True


,check,value
0,fixed date records,17
1,unparseable date rows after fix,20
2,non-standard format rows after fix,0
3,rows outside expected year 2018 after fix,20
4,row count preserved,True


,order_id,date,date_parsed,date_matches_standard_format,date_parse_success,date_year
10,ORDC01147,2018-26-08,NaT,True,False,NaN
16,ORDK04564,2018-19-09,NaT,True,False,NaN
37,ORDJ05383,2018-18-08,NaT,True,False,NaN
51,ORDB00774,2018-16-06,NaT,True,False,NaN
81,ORDB06811,2018-28-11,NaT,True,False,NaN
96,ORDZ10150,2018-18-05,NaT,True,False,NaN
114,ORDZ08183,2018-13-11,NaT,True,False,NaN
139,ORDJ05844,2018-15-04,NaT,True,False,NaN
161,ORDY08360,2018-28-11,NaT,True,False,NaN
168,ORDI10091,2018-31-08,NaT,True,False,NaN


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,date,non_standard_format,17
3,unflagged,unflagged,446


total flagged rows after date fix: 54


### Cell 10.2 讨论：date 修复完成了吗？

**Q1：我们修了哪些 date records？**  
A：只修了未被其他 anomaly flag 占用、可以被 `format='mixed'` 解析、但原始格式不是 `YYYY-MM-DD` 的 records。`date_fix_audit` 展示了原值和统一后的标准日期。

**Q2：为什么只 flag 实际改变的 rows？**  
A：因为 tracker 记录的是真实修复动作。能够被解析且已经是标准格式的日期不应该被标记为 anomaly。

**Q3：修复后 validation 看什么？**  
A：看是否还存在无法解析日期、非标准格式日期、年份不在 2018 的日期，以及行数是否保持不变。

**Q4：下一列是什么？**  
A：接下来检查 `time` 和 `order_type`。这两列最好放在一起，因为 `order_type` 应该由 `time` 所属时段决定。

**下一步：**  
检查 `time` 的格式，并验证 `order_type` 是否和 time window 一致。

## Cell 11：检查 time 格式和 order_type 一致性

接下来一起检查 `time` 和 `order_type`。

根据 assignment guide：

- Breakfast: 08:00:00 到 12:00:00
- Lunch: 12:00:01 到 16:00:00
- Dinner: 16:00:01 到 20:00:00

所以 `time` 应该能被解析，并且 `order_type` 应该和时间窗口一致。

In [18]:
time_type_check = dirty_cleaning.copy()

time_type_check['time_as_string'] = time_type_check['time'].astype(str)
time_type_check['time_matches_format'] = time_type_check['time_as_string'].str.match(r'^\d{2}:\d{2}:\d{2}$')
time_type_check['time_parsed'] = pd.to_datetime(
    time_type_check['time_as_string'],
    format='%H:%M:%S',
    errors='coerce'
).dt.time
time_type_check['time_parse_success'] = time_type_check['time_parsed'].notna()

def infer_order_type_from_time(value):
    if pd.isna(value):
        return pd.NA
    total_seconds = value.hour * 3600 + value.minute * 60 + value.second
    if 8 * 3600 <= total_seconds <= 12 * 3600:
        return 'Breakfast'
    if 12 * 3600 + 1 <= total_seconds <= 16 * 3600:
        return 'Lunch'
    if 16 * 3600 + 1 <= total_seconds <= 20 * 3600:
        return 'Dinner'
    return pd.NA

time_type_check['expected_order_type'] = time_type_check['time_parsed'].apply(infer_order_type_from_time)
time_type_check['order_type_is_allowed'] = time_type_check['order_type'].isin(['Breakfast', 'Lunch', 'Dinner'])
time_type_check['order_type_matches_time'] = (
    time_type_check['order_type'] == time_type_check['expected_order_type']
)

time_type_summary = pd.DataFrame([
    {
        'check': 'invalid time format rows',
        'value': int((~time_type_check['time_matches_format']).sum()),
    },
    {
        'check': 'unparseable time rows',
        'value': int((~time_type_check['time_parse_success']).sum()),
    },
    {
        'check': 'time outside meal windows rows',
        'value': int(time_type_check['expected_order_type'].isna().sum()),
    },
    {
        'check': 'invalid order_type value rows',
        'value': int((~time_type_check['order_type_is_allowed']).sum()),
    },
    {
        'check': 'order_type mismatches time window rows',
        'value': int((~time_type_check['order_type_matches_time']).sum()),
    },
])

display(time_type_summary)

time_type_suspects = time_type_check.loc[
    (~time_type_check['time_matches_format']) |
    (~time_type_check['time_parse_success']) |
    (time_type_check['expected_order_type'].isna()) |
    (~time_type_check['order_type_is_allowed']) |
    (~time_type_check['order_type_matches_time'])
].copy()

time_type_suspects['suspect_reason'] = ''
time_type_suspects.loc[~time_type_suspects['time_matches_format'], 'suspect_reason'] += 'invalid_time_format;'
time_type_suspects.loc[~time_type_suspects['time_parse_success'], 'suspect_reason'] += 'unparseable_time;'
time_type_suspects.loc[time_type_suspects['expected_order_type'].isna(), 'suspect_reason'] += 'time_outside_meal_windows;'
time_type_suspects.loc[~time_type_suspects['order_type_is_allowed'], 'suspect_reason'] += 'invalid_order_type;'
time_type_suspects.loc[~time_type_suspects['order_type_matches_time'], 'suspect_reason'] += 'order_type_time_mismatch;'
time_type_suspects['suspect_reason'] = time_type_suspects['suspect_reason'].str.rstrip(';')

display(
    time_type_suspects[
        ['order_id', 'time', 'time_parsed', 'order_type', 'expected_order_type', 'suspect_reason']
    ]
)

print('time/order_type suspect records:', len(time_type_suspects))

,check,value
0,invalid time format rows,0
1,unparseable time rows,0
2,time outside meal windows rows,0
3,invalid order_type value rows,0
4,order_type mismatches time window rows,37


,order_id,time,time_parsed,order_type,expected_order_type,suspect_reason
23,ORDX10183,18:28:43,18:28:43,Lunch,Dinner,order_type_time_mismatch
28,ORDY07273,13:34:38,13:34:38,Breakfast,Lunch,order_type_time_mismatch
43,ORDK02724,13:24:30,13:24:30,Breakfast,Lunch,order_type_time_mismatch
45,ORDI00689,12:33:48,12:33:48,Breakfast,Lunch,order_type_time_mismatch
67,ORDJ03210,08:20:16,08:20:16,Lunch,Breakfast,order_type_time_mismatch
69,ORDJ10027,12:23:39,12:23:39,Breakfast,Lunch,order_type_time_mismatch
95,ORDK04641,10:01:41,10:01:41,Lunch,Breakfast,order_type_time_mismatch
108,ORDJ06140,12:03:22,12:03:22,Breakfast,Lunch,order_type_time_mismatch
117,ORDK02245,08:10:08,08:10:08,Lunch,Breakfast,order_type_time_mismatch
142,ORDA09491,12:23:39,12:23:39,Breakfast,Lunch,order_type_time_mismatch


time/order_type suspect records: 37


### Cell 11 讨论：time 和 order_type 为什么要一起检查？

**Q1：为什么不单独只看 `time`？**  
A：因为 `time` 本身可能格式正确，但 `order_type` 可能和时间窗口不匹配。例如 18:00 的订单不应该是 Breakfast。

**Q2：这一格检查了哪些 anomaly？**  
A：包括 time 的格式错误、无法解析、超出营业时间窗口、order_type 非法值，以及 order_type 与 time window 的 functional dependency violation。

**Q3：为什么这里只打印 suspects，不马上修？**  
A：因为仍然要遵守“一行最多一个 anomaly”。下一步要把这些 suspects 和 `dirty_issue_flags` 对照，看有没有已经被 branch_code 或 date 标记过的行。

**下一步：**  
检查 time/order_type suspects 是否和已有 flags 冲突，然后只 flag 真正需要修复的 rows。

## Cell 11.1：检查 time/order_type suspects 与已有 flags 的冲突

现在把 `time_type_suspects` 和 `dirty_issue_flags` 对照。

如果某条 suspect 已经被前面的 `branch_code` 或 `date` 标记过，就不能直接再标记为 `time` 或 `order_type`。如果没有冲突，我们再根据 suspect reason 判断应该修哪一列。

In [19]:
existing_flags_for_time_type_suspects = dirty_issue_flags.loc[
    dirty_issue_flags.index.intersection(time_type_suspects['order_id'])
].dropna(subset=['issue_column'])

display(existing_flags_for_time_type_suspects.reset_index())

unflagged_time_type_suspects = time_type_suspects.loc[
    ~time_type_suspects['order_id'].isin(existing_flags_for_time_type_suspects.index)
].copy()

unflagged_time_type_suspects['target_issue_column'] = np.select(
    [
        unflagged_time_type_suspects['suspect_reason'].str.contains('invalid_time_format|unparseable_time|time_outside_meal_windows', regex=True),
        unflagged_time_type_suspects['suspect_reason'].str.contains('invalid_order_type|order_type_time_mismatch', regex=True),
    ],
    [
        'time',
        'order_type',
    ],
    default='review_needed'
)

display(
    unflagged_time_type_suspects[
        [
            'order_id',
            'time',
            'time_parsed',
            'order_type',
            'expected_order_type',
            'suspect_reason',
            'target_issue_column',
        ]
    ]
)

print('time/order_type suspect records:', len(time_type_suspects))
print('time/order_type suspects already flagged before:', len(existing_flags_for_time_type_suspects))
print('unflagged time/order_type suspects:', len(unflagged_time_type_suspects))

,order_id,issue_column,issue_type,evidence


,order_id,time,time_parsed,order_type,expected_order_type,suspect_reason,target_issue_column
23,ORDX10183,18:28:43,18:28:43,Lunch,Dinner,order_type_time_mismatch,order_type
28,ORDY07273,13:34:38,13:34:38,Breakfast,Lunch,order_type_time_mismatch,order_type
43,ORDK02724,13:24:30,13:24:30,Breakfast,Lunch,order_type_time_mismatch,order_type
45,ORDI00689,12:33:48,12:33:48,Breakfast,Lunch,order_type_time_mismatch,order_type
67,ORDJ03210,08:20:16,08:20:16,Lunch,Breakfast,order_type_time_mismatch,order_type
69,ORDJ10027,12:23:39,12:23:39,Breakfast,Lunch,order_type_time_mismatch,order_type
95,ORDK04641,10:01:41,10:01:41,Lunch,Breakfast,order_type_time_mismatch,order_type
108,ORDJ06140,12:03:22,12:03:22,Breakfast,Lunch,order_type_time_mismatch,order_type
117,ORDK02245,08:10:08,08:10:08,Lunch,Breakfast,order_type_time_mismatch,order_type
142,ORDA09491,12:23:39,12:23:39,Breakfast,Lunch,order_type_time_mismatch,order_type


time/order_type suspect records: 37
time/order_type suspects already flagged before: 0
unflagged time/order_type suspects: 37


### Cell 11.1 讨论：这些 suspects 应该归到 time 还是 order_type？

**Q1：为什么先看 existing flags？**  
A：因为每行最多一个 anomaly。已有 flag 的行不能被自动追加第二个问题，除非我们回头推翻原来的判断。

**Q2：如何区分是 `time` 错还是 `order_type` 错？**  
A：如果 time 格式无效、无法解析、或超出所有 meal windows，优先怀疑 `time`。如果 time 合法且能推导出 expected order type，但当前 `order_type` 不一致，则更可能是 `order_type` 错。

**Q3：为什么这里可以修 `order_type`？**  
A：因为 assignment guide 给了明确 time windows，所以 `expected_order_type` 是由规则推导出的唯一修复值。

**Q4：下一步做什么？**  
A：如果 suspects 都是 `order_type` 问题，就把它们修成 `expected_order_type`，并更新 tracker。如果出现 time 问题，则需要更谨慎地寻找唯一修复证据。

**下一步：**  
修复 unflagged 的 `order_type` suspects，并验证 time/order_type 一致性。

## Cell 11.2：修复 order_type 并验证 time window 一致性

现在修复 `order_type`。

修复原则：

- 只修复未被 tracker 标记过的 rows；
- 只修复 `target_issue_column == 'order_type'` 的 rows；
- 修复值来自 time window 推导出的 `expected_order_type`；
- 修完后更新 `dirty_issue_flags` 并重新验证。

In [20]:
order_type_fix_candidates = unflagged_time_type_suspects.loc[
    unflagged_time_type_suspects['target_issue_column'] == 'order_type'
].copy()

order_type_fix_audit = order_type_fix_candidates[
    ['order_id', 'time', 'order_type', 'expected_order_type', 'suspect_reason']
].copy()
order_type_fix_audit['order_type_will_change'] = (
    order_type_fix_audit['order_type'] != order_type_fix_audit['expected_order_type']
)
order_type_fix_audit = order_type_fix_audit.loc[order_type_fix_audit['order_type_will_change']].copy()

display(order_type_fix_audit)

for _, row in order_type_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'order_type'
    ] = row['expected_order_type']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'order_type'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'order_type_time_mismatch'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"time={row['time']}; observed={row['order_type']}; "
        f"expected={row['expected_order_type']}"
    )

post_time_type_check = dirty_cleaning.copy()
post_time_type_check['time_as_string'] = post_time_type_check['time'].astype(str)
post_time_type_check['time_matches_format'] = post_time_type_check['time_as_string'].str.match(r'^\d{2}:\d{2}:\d{2}$')
post_time_type_check['time_parsed'] = pd.to_datetime(
    post_time_type_check['time_as_string'],
    format='%H:%M:%S',
    errors='coerce'
).dt.time
post_time_type_check['time_parse_success'] = post_time_type_check['time_parsed'].notna()
post_time_type_check['expected_order_type'] = post_time_type_check['time_parsed'].apply(infer_order_type_from_time)
post_time_type_check['order_type_is_allowed'] = post_time_type_check['order_type'].isin(['Breakfast', 'Lunch', 'Dinner'])
post_time_type_check['order_type_matches_time'] = (
    post_time_type_check['order_type'] == post_time_type_check['expected_order_type']
)

post_time_type_validation = pd.DataFrame([
    {
        'check': 'fixed order_type records',
        'value': len(order_type_fix_audit),
    },
    {
        'check': 'invalid time format rows after fix',
        'value': int((~post_time_type_check['time_matches_format']).sum()),
    },
    {
        'check': 'unparseable time rows after fix',
        'value': int((~post_time_type_check['time_parse_success']).sum()),
    },
    {
        'check': 'time outside meal windows rows after fix',
        'value': int(post_time_type_check['expected_order_type'].isna().sum()),
    },
    {
        'check': 'invalid order_type value rows after fix',
        'value': int((~post_time_type_check['order_type_is_allowed']).sum()),
    },
    {
        'check': 'order_type mismatches time window rows after fix',
        'value': int((~post_time_type_check['order_type_matches_time']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_time_type_validation)

display(
    post_time_type_check.loc[
        (~post_time_type_check['time_matches_format']) |
        (~post_time_type_check['time_parse_success']) |
        (post_time_type_check['expected_order_type'].isna()) |
        (~post_time_type_check['order_type_is_allowed']) |
        (~post_time_type_check['order_type_matches_time']),
        ['order_id', 'time', 'time_parsed', 'order_type', 'expected_order_type']
    ]
)

tracker_summary_after_order_type = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_order_type)

print('total flagged rows after order_type fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,time,order_type,expected_order_type,suspect_reason,order_type_will_change
23,ORDX10183,18:28:43,Lunch,Dinner,order_type_time_mismatch,True
28,ORDY07273,13:34:38,Breakfast,Lunch,order_type_time_mismatch,True
43,ORDK02724,13:24:30,Breakfast,Lunch,order_type_time_mismatch,True
45,ORDI00689,12:33:48,Breakfast,Lunch,order_type_time_mismatch,True
67,ORDJ03210,08:20:16,Lunch,Breakfast,order_type_time_mismatch,True
69,ORDJ10027,12:23:39,Breakfast,Lunch,order_type_time_mismatch,True
95,ORDK04641,10:01:41,Lunch,Breakfast,order_type_time_mismatch,True
108,ORDJ06140,12:03:22,Breakfast,Lunch,order_type_time_mismatch,True
117,ORDK02245,08:10:08,Lunch,Breakfast,order_type_time_mismatch,True
142,ORDA09491,12:23:39,Breakfast,Lunch,order_type_time_mismatch,True


,check,value
0,fixed order_type records,37
1,invalid time format rows after fix,0
2,unparseable time rows after fix,0
3,time outside meal windows rows after fix,0
4,invalid order_type value rows after fix,0
5,order_type mismatches time window rows after fix,0
6,row count preserved,True


,order_id,time,time_parsed,order_type,expected_order_type


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,date,non_standard_format,17
3,order_type,order_type_time_mismatch,37
4,unflagged,unflagged,409


total flagged rows after order_type fix: 91


### Cell 11.2 讨论：order_type 修复是否完成？

**Q1：这一步修复的依据是什么？**  
A：assignment guide 明确给出 time windows，所以可以从 `time` 唯一推导 `expected_order_type`。

**Q2：为什么只修 `order_type`，不修 `time`？**  
A：因为当前 suspects 的 time 可以解析，也落在 meal windows 内；问题是当前 `order_type` 和 time window 不一致。因此更可能是 `order_type` 错。

**Q3：修复后 validation 看什么？**  
A：看 time 是否仍可解析、是否都落在 meal windows、order_type 是否都是合法值、以及 order_type 是否全部匹配 expected order type。

**Q4：下一列是什么？**  
A：下一步检查 `order_items` 和 `order_price`。这两列也应该一起看，因为 price 应该由 items 和 unit prices 决定。

**下一步：**  
解析 `order_items`，检查 item names、quantities，并推导 menu prices。

## Cell 12：解析 order_items 并做基础结构检查

接下来检查 `order_items`。

Assignment guide 说明：`order_items` 是 list of tuples，每个 tuple 的第一个元素是 item name，第二个元素是 quantity。三种 meal type 的菜单互不重叠，后面还可以用 `order_items` 和 `order_price` 推导 unit prices。

这一格先只做基础检查：能不能解析、结构是否正确、item name 是否稳定、quantity 是否为正整数。

In [21]:
from ast import literal_eval

items_check = dirty_cleaning.copy()

def parse_order_items(value):
    try:
        parsed = literal_eval(value)
    except Exception:
        return pd.NA
    return parsed

def validate_order_items_structure(parsed):
    if not isinstance(parsed, list):
        return False
    if len(parsed) == 0:
        return False
    for entry in parsed:
        if not isinstance(entry, tuple) or len(entry) != 2:
            return False
        item, quantity = entry
        if not isinstance(item, str):
            return False
        if not isinstance(quantity, int) or quantity <= 0:
            return False
    return True

items_check['parsed_order_items'] = items_check['order_items'].apply(parse_order_items)
items_check['order_items_parse_success'] = items_check['parsed_order_items'].apply(lambda value: isinstance(value, list))
items_check['order_items_structure_valid'] = items_check['parsed_order_items'].apply(validate_order_items_structure)

item_records = []
for _, row in items_check.loc[items_check['order_items_structure_valid']].iterrows():
    for item_name, quantity in row['parsed_order_items']:
        item_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'item_name': item_name,
            'quantity': quantity,
        })

item_records_df = pd.DataFrame(item_records)

items_structure_summary = pd.DataFrame([
    {
        'check': 'unparseable order_items rows',
        'value': int((~items_check['order_items_parse_success']).sum()),
    },
    {
        'check': 'invalid order_items structure rows',
        'value': int((~items_check['order_items_structure_valid']).sum()),
    },
    {
        'check': 'unique item names',
        'value': item_records_df['item_name'].nunique(),
    },
    {
        'check': 'minimum quantity',
        'value': int(item_records_df['quantity'].min()),
    },
    {
        'check': 'maximum quantity',
        'value': int(item_records_df['quantity'].max()),
    },
])

display(items_structure_summary)

item_name_summary = (
    item_records_df
    .groupby(['order_type', 'item_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['order_type', 'item_name'])
)

display(item_name_summary)

display(
    items_check.loc[
        (~items_check['order_items_parse_success']) |
        (~items_check['order_items_structure_valid']),
        ['order_id', 'order_type', 'order_items', 'order_items_parse_success', 'order_items_structure_valid']
    ]
)

print('all observed item names:', sorted(item_records_df['item_name'].unique()))

,check,value
0,unparseable order_items rows,0
1,invalid order_items structure rows,0
2,unique item names,13
3,minimum quantity,1
4,maximum quantity,10


,order_type,item_name,count
0,Breakfast,Burger,1
1,Breakfast,Cereal,124
2,Breakfast,Chicken,2
3,Breakfast,Coffee,133
4,Breakfast,Eggs,128
5,Breakfast,Fish&Chips,2
6,Breakfast,Fries,1
7,Breakfast,Pancake,136
8,Breakfast,Pasta,2
9,Breakfast,Salmon,1


,order_id,order_type,order_items,order_items_parse_success,order_items_structure_valid


all observed item names: ['Burger', 'Cereal', 'Chicken', 'Coffee', 'Eggs', 'Fish&Chips', 'Fries', 'Pancake', 'Pasta', 'Salad', 'Salmon', 'Shrimp', 'Steak']


### Cell 12 讨论：order_items 的第一层检查说明什么？

**Q1：为什么先解析 `order_items`？**  
A：因为它原本是字符串形式的 list of tuples。只有先安全解析成 Python 对象，后面才能检查 item name、quantity 和 order_price。

**Q2：这一格主要检查哪些 anomaly？**  
A：主要检查 syntactic/structural errors：是否能解析，是否是非空 list，每个元素是否是 `(item_name, quantity)` tuple，quantity 是否为正整数。

**Q3：为什么还要按 `order_type` 展示 item names？**  
A：assignment guide 说三种 meal type 的菜单互不重叠。按 `order_type` 展示 item 分布，可以帮助我们下一步检查“某个 meal type 是否混入了不属于它的 item”。

**Q4：这一步为什么还不修 `order_price`？**  
A：因为 price 检查依赖 unit prices。我们要先确认 items 结构和菜单分组，再用可靠 records 推导 unit prices，最后才能判断 order_price 是否错。

**下一步：**  
根据 item names 和 order_type，推断每个 meal type 的菜单集合，并检查是否有菜单错配。

## Cell 13：推断菜单集合并检查 order_items 与 order_type 是否匹配

Assignment guide 说明三种 meal type 的菜单互不重叠。

因此我们可以从大多数记录中推断每个 item 属于哪个 meal type，然后检查每条订单中的 items 是否都属于该订单的 `order_type`。这一格仍然先做 detection，不直接修复。

In [22]:
item_meal_counts = (
    item_records_df
    .groupby(['item_name', 'order_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['item_name', 'count'], ascending=[True, False])
)

display(item_meal_counts)

item_expected_meal = (
    item_meal_counts
    .sort_values(['item_name', 'count'], ascending=[True, False])
    .drop_duplicates('item_name')
    .rename(columns={'order_type': 'expected_order_type_for_item'})
    [['item_name', 'expected_order_type_for_item', 'count']]
)

display(item_expected_meal)

item_to_expected_meal = dict(
    zip(item_expected_meal['item_name'], item_expected_meal['expected_order_type_for_item'])
)

menu_mismatch_records = []
for _, row in items_check.loc[items_check['order_items_structure_valid']].iterrows():
    mismatched_items = []
    for item_name, quantity in row['parsed_order_items']:
        expected_meal = item_to_expected_meal.get(item_name)
        if expected_meal != row['order_type']:
            mismatched_items.append((item_name, expected_meal, quantity))
    if len(mismatched_items) > 0:
        menu_mismatch_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'order_items': row['order_items'],
            'mismatched_items': mismatched_items,
        })

menu_mismatch_suspects = pd.DataFrame(menu_mismatch_records)

display(menu_mismatch_suspects)

existing_flags_for_menu_suspects = dirty_issue_flags.loc[
    dirty_issue_flags.index.intersection(menu_mismatch_suspects['order_id'] if len(menu_mismatch_suspects) else [])
].dropna(subset=['issue_column'])

display(existing_flags_for_menu_suspects.reset_index())

unflagged_menu_mismatch_suspects = menu_mismatch_suspects.loc[
    ~menu_mismatch_suspects['order_id'].isin(existing_flags_for_menu_suspects.index)
].copy() if len(menu_mismatch_suspects) else menu_mismatch_suspects.copy()

display(unflagged_menu_mismatch_suspects)

print('menu mismatch suspect records:', len(menu_mismatch_suspects))
print('menu mismatch suspects already flagged before:', len(existing_flags_for_menu_suspects))
print('unflagged menu mismatch suspects:', len(unflagged_menu_mismatch_suspects))

,item_name,order_type,count
2,Burger,Lunch,119
0,Burger,Breakfast,1
1,Burger,Dinner,1
3,Cereal,Breakfast,124
5,Cereal,Lunch,2
4,Cereal,Dinner,1
7,Chicken,Lunch,114
6,Chicken,Breakfast,2
8,Coffee,Breakfast,133
9,Coffee,Dinner,1


,item_name,expected_order_type_for_item,count
2,Burger,Lunch,119
3,Cereal,Breakfast,124
7,Chicken,Lunch,114
8,Coffee,Breakfast,133
11,Eggs,Breakfast,128
15,Fish&Chips,Dinner,113
19,Fries,Lunch,114
20,Pancake,Breakfast,136
24,Pasta,Dinner,109
27,Salad,Lunch,115


,order_id,order_type,order_items,mismatched_items
0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]","[(Shrimp, Dinner, 4)]"
1,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]","[(Fish&Chips, Dinner, 8)]"
2,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]","[(Salmon, Dinner, 2)]"
3,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]","[(Shrimp, Dinner, 9)]"
4,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]","[(Eggs, Breakfast, 6)]"
5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]","[(Salmon, Dinner, 2)]"
6,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]","[(Pasta, Dinner, 1)]"
7,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]","[(Fries, Lunch, 9)]"
8,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]","[(Salmon, Dinner, 7)]"
9,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]","[(Pasta, Dinner, 5)]"


,order_id,issue_column,issue_type,evidence


,order_id,order_type,order_items,mismatched_items
0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]","[(Shrimp, Dinner, 4)]"
1,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]","[(Fish&Chips, Dinner, 8)]"
2,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]","[(Salmon, Dinner, 2)]"
3,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]","[(Shrimp, Dinner, 9)]"
4,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]","[(Eggs, Breakfast, 6)]"
5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]","[(Salmon, Dinner, 2)]"
6,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]","[(Pasta, Dinner, 1)]"
7,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]","[(Fries, Lunch, 9)]"
8,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]","[(Salmon, Dinner, 7)]"
9,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]","[(Pasta, Dinner, 5)]"


menu mismatch suspect records: 37
menu mismatch suspects already flagged before: 0
unflagged menu mismatch suspects: 37


### Cell 13 讨论：菜单错配说明 order_items 错还是 order_type 错？

**Q1：为什么能从数据推断每个 item 的 meal type？**  
A：因为 assignment guide 说三种 meal type 菜单互不重叠。一个 item 应该稳定属于 Breakfast、Lunch 或 Dinner 中的一类。

**Q2：如果某条订单的 item 不属于当前 order_type，说明什么？**  
A：它可能是 `order_items` 错，也可能是 `order_type` 错。但如果前面已经根据 time window 修过 order_type，且当前 order_type 可信，那么更可能是 order_items 里混入了错误 item。

**Q3：为什么还要检查 existing flags？**  
A：继续遵守一行最多一个 anomaly。已经被前面标记过的行不能直接追加 order_items 错误。

**Q4：这一步要不要马上修 item？**  
A：不急。修 item 需要知道正确 item 应该是什么，通常要结合 order_price 和 menu price 方程。这里只先定位菜单错配 suspects。

**下一步：**  
推导 unit prices，并用 `order_items` 计算 expected order_price，检查价格或 item 数量相关错误。

## Cell 14：推导 menu unit prices 并检查 order_price residual

现在进入 `order_price` 检查。

`order_price` 应该等于每个 item 的 unit price 乘以 quantity 后求和。Assignment guide 提示可以用 `numpy.linalg` 解多变量方程。

这一格先做三件事：

1. 按 meal type 建立 item quantity matrix；
2. 用未被 tracker 标记、且菜单没有错配的 records 推导 unit prices；
3. 用推导出的 prices 计算 expected order price，并打印 residual 可疑 records。

In [23]:
price_model_data = dirty_cleaning.copy()
price_model_data['parsed_order_items'] = price_model_data['order_items'].apply(parse_order_items)
price_model_data['order_items_structure_valid'] = price_model_data['parsed_order_items'].apply(validate_order_items_structure)

menu_mismatch_order_ids = set(menu_mismatch_suspects['order_id']) if len(menu_mismatch_suspects) else set()
flagged_order_ids = set(dirty_issue_flags.dropna(subset=['issue_column']).index)

price_reference_data = price_model_data.loc[
    price_model_data['order_items_structure_valid'] &
    (~price_model_data['order_id'].isin(menu_mismatch_order_ids)) &
    (~price_model_data['order_id'].isin(flagged_order_ids))
].copy()

meal_items = {
    meal_type: sorted([
        item for item, expected_meal in item_to_expected_meal.items()
        if expected_meal == meal_type
    ])
    for meal_type in ['Breakfast', 'Lunch', 'Dinner']
}

unit_price_records = []
residual_reference_records = []

for meal_type, items in meal_items.items():
    meal_rows = price_reference_data.loc[price_reference_data['order_type'] == meal_type].copy()
    quantity_matrix = []
    target_prices = []
    order_ids = []

    for _, row in meal_rows.iterrows():
        quantity_by_item = {item: 0 for item in items}
        for item_name, quantity in row['parsed_order_items']:
            if item_name in quantity_by_item:
                quantity_by_item[item_name] += quantity
        quantity_matrix.append([quantity_by_item[item] for item in items])
        target_prices.append(row['order_price'])
        order_ids.append(row['order_id'])

    X = np.array(quantity_matrix, dtype=float)
    y = np.array(target_prices, dtype=float)
    prices, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    prices = np.round(prices, 2)

    for item_name, unit_price in zip(items, prices):
        unit_price_records.append({
            'order_type': meal_type,
            'item_name': item_name,
            'unit_price': unit_price,
        })

    predicted = X @ prices
    residuals = y - predicted
    for order_id, observed, expected, residual in zip(order_ids, y, predicted, residuals):
        residual_reference_records.append({
            'order_id': order_id,
            'order_type': meal_type,
            'observed_order_price': observed,
            'expected_order_price': round(expected, 2),
            'residual': round(residual, 4),
        })

unit_prices_df = pd.DataFrame(unit_price_records)
reference_residuals_df = pd.DataFrame(residual_reference_records)

display(unit_prices_df)

display(reference_residuals_df['residual'].abs().describe().to_frame(name='absolute_reference_residual'))

unit_price_lookup = dict(zip(unit_prices_df['item_name'], unit_prices_df['unit_price']))

def calculate_expected_order_price(parsed_items):
    if not validate_order_items_structure(parsed_items):
        return np.nan
    total = 0.0
    for item_name, quantity in parsed_items:
        unit_price = unit_price_lookup.get(item_name)
        if unit_price is None:
            return np.nan
        total += unit_price * quantity
    return round(total, 2)

price_model_data['expected_order_price'] = price_model_data['parsed_order_items'].apply(calculate_expected_order_price)
price_model_data['order_price_residual'] = (
    price_model_data['order_price'] - price_model_data['expected_order_price']
).round(4)
price_model_data['order_price_matches_items'] = price_model_data['order_price_residual'].abs() < 0.01

price_suspects = price_model_data.loc[
    ~price_model_data['order_price_matches_items']
].copy()

price_summary = pd.DataFrame([
    {
        'check': 'price reference rows',
        'value': len(price_reference_data),
    },
    {
        'check': 'order_price residual suspect rows',
        'value': len(price_suspects),
    },
])

display(price_summary)

display(
    price_suspects[
        ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual']
    ].head(30)
)

print('unit prices inferred for items:', len(unit_prices_df))

,order_type,item_name,unit_price
0,Breakfast,Cereal,27.0900
1,Breakfast,Coffee,9.6000
2,Breakfast,Eggs,18.0700
3,Breakfast,Pancake,24.0700
4,Lunch,Burger,31.2800
5,Lunch,Chicken,30.6000
6,Lunch,Fries,13.3400
7,Lunch,Salad,21.0600
8,Lunch,Steak,40.6900
9,Dinner,Fish&Chips,33.8100


,absolute_reference_residual
count,372.0000
mean,50.4187
std,111.3757
min,0.0300
25%,8.1525
50%,18.0050
75%,35.0250
max,722.7700


,check,value
0,price reference rows,372
1,order_price residual suspect rows,500


,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual
0,ORDX00699,Lunch,"[('Chicken', 2), ('Steak', 1)]",109.0000,101.8900,7.1100
1,ORDX06260,Breakfast,"[('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)]",548.5000,566.8600,-18.3600
2,ORDI04941,Breakfast,"[('Cereal', 5), ('Pancake', 6), ('Coffee', 8), ('Eggs', 7)]",464.5000,483.1600,-18.6600
3,ORDI03372,Dinner,"[('Fish&Chips', 7), ('Shrimp', 3)]",407.0000,392.7600,14.2400
4,ORDK03173,Dinner,"[('Fish&Chips', 7), ('Pasta', 6)]",410.0000,424.1100,-14.1100
5,ORDB02744,Breakfast,"[('Eggs', 1), ('Coffee', 2), ('Cereal', 9)]",226.0000,281.0800,-55.0800
6,ORDB07772,Breakfast,"[('Eggs', 5), ('Coffee', 10), ('Pancake', 1)]",209.2500,210.4200,-1.1700
7,ORDI02220,Lunch,"[('Steak', 8), ('Chicken', 4), ('Burger', 10)]",798.0000,760.7200,37.2800
8,ORDY07813,Lunch,"[('Salad', 7), ('Burger', 10), ('Fries', 8), ('Chicken', 7), ('Steak', 7)]","1,065.4000","1,065.9700",-0.5700
9,ORDJ02557,Lunch,"[('Chicken', 4), ('Burger', 4)]",252.0000,247.5200,4.4800


unit prices inferred for items: 13


### Cell 14 讨论：为什么先推导 unit price，再判断 order_price？

**Q1：为什么不能直接用均值或常识判断 order_price？**  
A：因为 `order_price` 是由 `order_items` 中每个 item 的单价和数量决定的。要判断价格是否错，必须先知道 unit prices。

**Q2：为什么参考数据要排除已经 flagged 的 rows？**  
A：因为这些 rows 已经被解释为其他 anomaly。用它们推导价格可能污染方程。我们还排除了菜单错配 rows，因为它们的 items 可信度较低。

**Q3：residual 表示什么？**  
A：`residual = observed_order_price - expected_order_price`。如果 residual 接近 0，价格和 items 一致；如果明显不为 0，这条记录可能存在 `order_price`、`order_items` 或 quantity 错误。

**Q4：为什么这一步还不修？**  
A：因为 price residual 只能说明“价格和 items 不一致”。真正错误可能是 `order_price`，也可能是 `order_items` 的 item 或 quantity。下一步还要结合 tracker 和菜单规则决定修哪一列。

**下一步：**  
打印 price suspects，检查它们是否已经被 tracker 标记，并判断目标 anomaly 是 `order_price` 还是 `order_items`。

## Cell 14.1：诊断为什么出现大量 order_price mismatch

刚才如果看到 `order_price price_items_mismatch` 数量非常大，例如几百行，这不是正常结果，而是一个警报：我们的 unit price 推导被污染了。

这一步专门展示 mismatch 是怎么来的：比较“从当前不够干净的参考行推导出的 unit prices”和“可信菜单价”下的 residual。

In [24]:
trusted_menu_prices = {
    'Breakfast': {
        'Cereal': 21.00,
        'Coffee': 7.50,
        'Eggs': 22.00,
        'Pancake': 24.25,
    },
    'Lunch': {
        'Burger': 31.00,
        'Chicken': 32.00,
        'Fries': 12.00,
        'Salad': 17.20,
        'Steak': 45.00,
    },
    'Dinner': {
        'Fish&Chips': 35.00,
        'Pasta': 27.50,
        'Salmon': 41.00,
        'Shrimp': 54.00,
    },
}

trusted_price_rows = []
for meal_type, price_map in trusted_menu_prices.items():
    for item_name, unit_price in price_map.items():
        trusted_price_rows.append({
            'order_type': meal_type,
            'item_name': item_name,
            'trusted_unit_price': unit_price,
        })

trusted_prices_df = pd.DataFrame(trusted_price_rows)
price_comparison = unit_prices_df.merge(
    trusted_prices_df,
    on=['order_type', 'item_name'],
    how='outer'
)
price_comparison['unit_price_difference'] = (
    price_comparison['unit_price'] - price_comparison['trusted_unit_price']
).round(4)

display(price_comparison)

trusted_unit_price_lookup = dict(zip(trusted_prices_df['item_name'], trusted_prices_df['trusted_unit_price']))

def calculate_trusted_order_price(parsed_items):
    if not validate_order_items_structure(parsed_items):
        return np.nan
    total = 0.0
    for item_name, quantity in parsed_items:
        unit_price = trusted_unit_price_lookup.get(item_name)
        if unit_price is None:
            return np.nan
        total += unit_price * quantity
    return round(total, 2)

price_diagnosis = dirty_cleaning.copy()
price_diagnosis['parsed_order_items'] = price_diagnosis['order_items'].apply(parse_order_items)
price_diagnosis['trusted_expected_order_price'] = price_diagnosis['parsed_order_items'].apply(calculate_trusted_order_price)
price_diagnosis['trusted_order_price_residual'] = (
    price_diagnosis['order_price'] - price_diagnosis['trusted_expected_order_price']
).round(4)
price_diagnosis['trusted_price_matches_items'] = price_diagnosis['trusted_order_price_residual'].abs() < 0.01

price_diagnosis['inferred_expected_order_price'] = price_diagnosis['parsed_order_items'].apply(calculate_expected_order_price)
price_diagnosis['inferred_order_price_residual'] = (
    price_diagnosis['order_price'] - price_diagnosis['inferred_expected_order_price']
).round(4)
price_diagnosis['inferred_price_matches_items'] = price_diagnosis['inferred_order_price_residual'].abs() < 0.01

price_diagnosis_summary = pd.DataFrame([
    {
        'method': 'current_lstsq_inferred_prices',
        'mismatch_rows': int((~price_diagnosis['inferred_price_matches_items']).sum()),
    },
    {
        'method': 'trusted_menu_prices',
        'mismatch_rows': int((~price_diagnosis['trusted_price_matches_items']).sum()),
    },
])

display(price_diagnosis_summary)

mismatch_comparison_view = price_diagnosis.loc[
    (~price_diagnosis['inferred_price_matches_items']) |
    (~price_diagnosis['trusted_price_matches_items']),
    [
        'order_id',
        'order_type',
        'order_items',
        'order_price',
        'inferred_expected_order_price',
        'inferred_order_price_residual',
        'trusted_expected_order_price',
        'trusted_order_price_residual',
    ]
].copy()

display(mismatch_comparison_view.head(40))

trusted_mismatch_only = price_diagnosis.loc[
    ~price_diagnosis['trusted_price_matches_items'],
    [
        'order_id',
        'order_type',
        'order_items',
        'order_price',
        'trusted_expected_order_price',
        'trusted_order_price_residual',
    ]
].copy()

display(trusted_mismatch_only.head(60))

print('rows mismatching under inferred prices:', int((~price_diagnosis['inferred_price_matches_items']).sum()))
print('rows mismatching under trusted prices:', int((~price_diagnosis['trusted_price_matches_items']).sum()))

,order_type,item_name,unit_price,trusted_unit_price,unit_price_difference
0,Breakfast,Cereal,27.0900,21.0000,6.0900
1,Breakfast,Coffee,9.6000,7.5000,2.1000
2,Breakfast,Eggs,18.0700,22.0000,-3.9300
3,Breakfast,Pancake,24.0700,24.2500,-0.1800
4,Dinner,Fish&Chips,33.8100,35.0000,-1.1900
5,Dinner,Pasta,31.2400,27.5000,3.7400
6,Dinner,Salmon,36.3600,41.0000,-4.6400
7,Dinner,Shrimp,52.0300,54.0000,-1.9700
8,Lunch,Burger,31.2800,31.0000,0.2800
9,Lunch,Chicken,30.6000,32.0000,-1.4000


,method,mismatch_rows
0,current_lstsq_inferred_prices,500
1,trusted_menu_prices,74


,order_id,order_type,order_items,order_price,inferred_expected_order_price,inferred_order_price_residual,trusted_expected_order_price,trusted_order_price_residual
0,ORDX00699,Lunch,"[('Chicken', 2), ('Steak', 1)]",109.0000,101.8900,7.1100,109.0000,0.0000
1,ORDX06260,Breakfast,"[('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)]",548.5000,566.8600,-18.3600,548.5000,0.0000
2,ORDI04941,Breakfast,"[('Cereal', 5), ('Pancake', 6), ('Coffee', 8), ('Eggs', 7)]",464.5000,483.1600,-18.6600,464.5000,0.0000
3,ORDI03372,Dinner,"[('Fish&Chips', 7), ('Shrimp', 3)]",407.0000,392.7600,14.2400,407.0000,0.0000
4,ORDK03173,Dinner,"[('Fish&Chips', 7), ('Pasta', 6)]",410.0000,424.1100,-14.1100,410.0000,0.0000
5,ORDB02744,Breakfast,"[('Eggs', 1), ('Coffee', 2), ('Cereal', 9)]",226.0000,281.0800,-55.0800,226.0000,0.0000
6,ORDB07772,Breakfast,"[('Eggs', 5), ('Coffee', 10), ('Pancake', 1)]",209.2500,210.4200,-1.1700,209.2500,0.0000
7,ORDI02220,Lunch,"[('Steak', 8), ('Chicken', 4), ('Burger', 10)]",798.0000,760.7200,37.2800,798.0000,0.0000
8,ORDY07813,Lunch,"[('Salad', 7), ('Burger', 10), ('Fries', 8), ('Chicken', 7), ('Steak', 7)]","1,065.4000","1,065.9700",-0.5700,"1,065.4000",0.0000
9,ORDJ02557,Lunch,"[('Chicken', 4), ('Burger', 4)]",252.0000,247.5200,4.4800,252.0000,0.0000


,order_id,order_type,order_items,order_price,trusted_expected_order_price,trusted_order_price_residual
15,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]",912.0000,948.0000,-36.0000
24,ORDA10507,Breakfast,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",173.0000,507.5000,-334.5000
29,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",301.2000,333.2000,-32.0000
50,ORDC08998,Breakfast,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), ('Cereal', 2)]",559.6000,266.7500,292.8500
59,ORDJ06050,Breakfast,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]","1,239.5000",457.0000,782.5000
64,ORDA07051,Breakfast,"[('Eggs', 9), ('Cereal', 9)]",967.0000,387.0000,580.0000
71,ORDA09458,Dinner,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",907.5000,801.0000,106.5000
73,ORDI09699,Breakfast,"[('Pancake', 4), ('Eggs', 8)]",43.0000,273.0000,-230.0000
80,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]",792.6000,784.6000,8.0000
84,ORDZ09320,Breakfast,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), ('Pancake', 2)]",367.0000,252.5000,114.5000


rows mismatching under inferred prices: 500
rows mismatching under trusted prices: 74


### Cell 14.1 讨论：372 个 price mismatch 为什么不可信？

**Q1：为什么几百个 mismatch 是危险信号？**  
A：dirty data 每行最多一个 anomaly。如果我们突然把几百行都判断为 `order_price` 错，通常不是数据真的错这么多，而是我们的检测模型出了问题。

**Q2：问题出在哪里？**  
A：我们用 least squares 从当前参考行推 unit prices，但参考行仍可能被未处理的问题污染；一旦 unit prices 偏了，几乎所有 expected price 都会偏，导致大量 false positives。

**Q3：怎么看 mismatch 到底怎么发生？**  
A：看这个 cell 输出的 comparison table：同一条订单在 inferred prices 下的 expected price，和 trusted menu prices 下的 expected price。如果 inferred residual 很大而 trusted residual 接近 0，说明错的是 price model，不是该 record。

**Q4：接下来应该怎么改教学路线？**  
A：先不要修 372 行。我们要重新设计 unit price 推导：要么使用已知菜单价作为教学参考，要么用更稳健的方程系统和干净参考行推导出正确菜单价，再只修真正 residual 不为 0 的 rows。

**下一步：**  
重做 price 部分：先确认菜单价，再重新打印真正的 `order_price` / `order_items` suspects。

## Cell 14.2：重新定义 price reference data

我们现在修正 price model 的参考数据选择。

关键原则：

1. `dirty_data` 每行最多一个 anomaly。如果一行已经被标记为 `branch_code`、`date` 或 `order_type`，那么它的 `order_items` 和 `order_price` 仍然可以作为菜单价格参考。
2. `missing_data` 只有 coverage/missing anomalies，没有其他 data anomalies，所以非缺失的 `order_items` 和 `order_price` 可以用来推导菜单价。
3. `outlier_data` 除了 `delivery_fee` outlier，没有其他 data anomalies，所以它的 `order_items` 和 `order_price` 也可以用来推导菜单价。

因此，我们要使用更多可靠 reference rows，而不是把所有 flagged dirty rows 都排除掉。

In [25]:
def build_item_quantity_row(parsed_items, item_names):
    quantity_by_item = {item_name: 0 for item_name in item_names}
    for item_name, quantity in parsed_items:
        if item_name in quantity_by_item:
            quantity_by_item[item_name] += quantity
    return [quantity_by_item[item_name] for item_name in item_names]

def prepare_price_reference_frame(df, source_name):
    frame = df.copy()
    frame['source_dataset'] = source_name
    frame['parsed_order_items'] = frame['order_items'].apply(parse_order_items)
    frame['order_items_structure_valid'] = frame['parsed_order_items'].apply(validate_order_items_structure)
    return frame

missing_price_reference = prepare_price_reference_frame(data['missing'], 'missing')
outlier_price_reference = prepare_price_reference_frame(data['outlier'], 'outlier')
dirty_price_reference = prepare_price_reference_frame(dirty_cleaning, 'dirty')

non_price_dirty_flags = dirty_issue_flags.loc[
    dirty_issue_flags['issue_column'].isin(['branch_code', 'date', 'order_type'])
].index

price_related_dirty_flags = dirty_issue_flags.loc[
    dirty_issue_flags['issue_column'].isin(['order_items', 'order_price'])
].index

# At this point order_items/order_price have not been flagged in the corrected workflow,
# but this variable makes the rule explicit and reusable.
dirty_price_reference['is_price_related_flagged'] = dirty_price_reference['order_id'].isin(price_related_dirty_flags)

dirty_price_reference_usable = dirty_price_reference.loc[
    dirty_price_reference['order_items_structure_valid'] &
    dirty_price_reference['order_price'].notna() &
    (~dirty_price_reference['is_price_related_flagged'])
].copy()

missing_price_reference_usable = missing_price_reference.loc[
    missing_price_reference['order_items_structure_valid'] &
    missing_price_reference['order_price'].notna()
].copy()

outlier_price_reference_usable = outlier_price_reference.loc[
    outlier_price_reference['order_items_structure_valid'] &
    outlier_price_reference['order_price'].notna()
].copy()

combined_price_reference = pd.concat(
    [
        dirty_price_reference_usable,
        missing_price_reference_usable,
        outlier_price_reference_usable,
    ],
    ignore_index=True,
)

price_reference_summary = pd.DataFrame([
    {
        'source_dataset': 'dirty',
        'usable_rows': len(dirty_price_reference_usable),
    },
    {
        'source_dataset': 'missing',
        'usable_rows': len(missing_price_reference_usable),
    },
    {
        'source_dataset': 'outlier',
        'usable_rows': len(outlier_price_reference_usable),
    },
    {
        'source_dataset': 'combined',
        'usable_rows': len(combined_price_reference),
    },
])

display(price_reference_summary)

display(
    combined_price_reference[
        ['source_dataset', 'order_id', 'order_type', 'order_items', 'order_price']
    ].head(10)
)

print('dirty rows flagged for non-price issues but still usable for price reference:', len(non_price_dirty_flags))
print('dirty rows excluded for price-related flags:', len(price_related_dirty_flags))

,source_dataset,usable_rows
0,dirty,500
1,missing,500
2,outlier,500
3,combined,1500


,source_dataset,order_id,order_type,order_items,order_price
0,dirty,ORDX00699,Lunch,"[('Chicken', 2), ('Steak', 1)]",109.0000
1,dirty,ORDX06260,Breakfast,"[('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)]",548.5000
2,dirty,ORDI04941,Breakfast,"[('Cereal', 5), ('Pancake', 6), ('Coffee', 8), ('Eggs', 7)]",464.5000
3,dirty,ORDI03372,Dinner,"[('Fish&Chips', 7), ('Shrimp', 3)]",407.0000
4,dirty,ORDK03173,Dinner,"[('Fish&Chips', 7), ('Pasta', 6)]",410.0000
5,dirty,ORDB02744,Breakfast,"[('Eggs', 1), ('Coffee', 2), ('Cereal', 9)]",226.0000
6,dirty,ORDB07772,Breakfast,"[('Eggs', 5), ('Coffee', 10), ('Pancake', 1)]",209.2500
7,dirty,ORDI02220,Lunch,"[('Steak', 8), ('Chicken', 4), ('Burger', 10)]",798.0000
8,dirty,ORDY07813,Lunch,"[('Salad', 7), ('Burger', 10), ('Fries', 8), ('Chicken', 7), ('Steak', 7)]","1,065.4000"
9,dirty,ORDJ02557,Lunch,"[('Chicken', 4), ('Burger', 4)]",252.0000


dirty rows flagged for non-price issues but still usable for price reference: 91
dirty rows excluded for price-related flags: 0


### Cell 14.2 讨论：哪些数据可以用来推导菜单价？

**Q1：为什么已经 flag 为 branch/date/order_type 的 dirty rows 还能用？**  
A：因为每行最多一个 anomaly。如果某行唯一问题已经被解释为 branch/date/order_type，那么它的 `order_items` 和 `order_price` 应该仍然可信。

**Q2：为什么 missing/outlier 文件也能用？**  
A：assignment guide 明确说 missing 文件只有 coverage/missing anomalies，outlier 文件除了 delivery_fee outlier 没有其他 data anomalies。菜单价格与 delivery_fee 无关，所以这些文件的 `order_items` 和 `order_price` 是很好的参考数据。

**Q3：哪些 rows 不应该用？**  
A：`order_items` 无法解析、`order_price` 缺失、或者已经被明确标记为 price/items 相关问题的 dirty rows 不应该用于推导 unit price。

**Q4：这一步修正了之前什么问题？**  
A：之前把所有 flagged dirty rows 都排除，过于保守；同时只用 dirty 数据推价格，容易被 dirty 中未处理的 price/items 问题污染。现在的 reference 更大、更干净。

**下一步：**  
用 combined reference 重新推导 unit prices，并验证是否得到稳定合理的菜单价。

## Cell 14.3：只用 missing 和 outlier 推导 unit prices

刚才 combined reference 仍然导致大量 residual，说明 dirty rows 可能仍在污染 unit price 方程。

现在进一步收紧 reference：只使用 `missing_data` 和 `outlier_data`。根据 assignment guide：

- `missing_data` 只有 coverage/missing anomalies；
- `outlier_data` 除了 `delivery_fee` outlier 没有其他 data anomalies；
- 菜单价格与 missing coverage 和 delivery_fee outlier 无关。

所以这两个文件更适合推导菜单 unit prices。

In [26]:
clean_price_reference = pd.concat(
    [
        missing_price_reference_usable,
        outlier_price_reference_usable,
    ],
    ignore_index=True,
)

clean_reference_summary = (
    clean_price_reference
    .groupby(['source_dataset', 'order_type'])
    .size()
    .reset_index(name='row_count')
)

display(clean_reference_summary)

clean_unit_price_records = []
clean_reference_residual_records = []

for meal_type, items in meal_items.items():
    meal_rows = clean_price_reference.loc[
        clean_price_reference['order_type'] == meal_type
    ].copy()

    X = np.array([
        build_item_quantity_row(parsed_items, items)
        for parsed_items in meal_rows['parsed_order_items']
    ], dtype=float)
    y = meal_rows['order_price'].astype(float).to_numpy()

    prices, residual_sum, rank, singular_values = np.linalg.lstsq(X, y, rcond=None)
    prices = np.round(prices, 2)

    for item_name, unit_price in zip(items, prices):
        clean_unit_price_records.append({
            'order_type': meal_type,
            'item_name': item_name,
            'unit_price': unit_price,
        })

    predicted = X @ prices
    residuals = y - predicted

    for order_id, source_dataset, observed, expected, residual in zip(
        meal_rows['order_id'],
        meal_rows['source_dataset'],
        y,
        predicted,
        residuals,
    ):
        clean_reference_residual_records.append({
            'source_dataset': source_dataset,
            'order_id': order_id,
            'order_type': meal_type,
            'observed_order_price': observed,
            'expected_order_price': round(expected, 2),
            'residual': round(residual, 4),
        })

clean_unit_prices_df = pd.DataFrame(clean_unit_price_records)
clean_reference_residuals_df = pd.DataFrame(clean_reference_residual_records)

display(clean_unit_prices_df)

display(
    clean_reference_residuals_df
    .assign(abs_residual=lambda frame: frame['residual'].abs())
    .groupby('order_type')['abs_residual']
    .describe()
    .reset_index()
)

large_clean_reference_residuals = clean_reference_residuals_df.loc[
    clean_reference_residuals_df['residual'].abs() >= 0.01
].copy()

display(large_clean_reference_residuals.head(30))

clean_unit_price_lookup = dict(
    zip(clean_unit_prices_df['item_name'], clean_unit_prices_df['unit_price'])
)

def calculate_clean_expected_order_price(parsed_items):
    if not validate_order_items_structure(parsed_items):
        return np.nan
    total = 0.0
    for item_name, quantity in parsed_items:
        unit_price = clean_unit_price_lookup.get(item_name)
        if unit_price is None:
            return np.nan
        total += unit_price * quantity
    return round(total, 2)

print('clean reference rows:', len(clean_price_reference))
print('clean reference rows with residual >= 0.01:', len(large_clean_reference_residuals))

,source_dataset,order_type,row_count
0,missing,Breakfast,165
1,missing,Dinner,170
2,missing,Lunch,165
3,outlier,Breakfast,176
4,outlier,Dinner,151
5,outlier,Lunch,173


,order_type,item_name,unit_price
0,Breakfast,Cereal,21.0000
1,Breakfast,Coffee,7.5000
2,Breakfast,Eggs,22.0000
3,Breakfast,Pancake,24.2500
4,Lunch,Burger,31.0000
5,Lunch,Chicken,32.0000
6,Lunch,Fries,12.0000
7,Lunch,Salad,17.2000
8,Lunch,Steak,45.0000
9,Dinner,Fish&Chips,35.0000


,order_type,count,mean,std,min,25%,50%,75%,max
0,Breakfast,341.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Dinner,321.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,Lunch,338.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


,source_dataset,order_id,order_type,observed_order_price,expected_order_price,residual


clean reference rows: 1000
clean reference rows with residual >= 0.01: 0


### Cell 14.3 讨论：为什么只用 missing/outlier 更干净？

**Q1：为什么这次不使用 dirty_data？**  
A：dirty_data 正是我们要检测错误的对象，里面可能有未发现的 `order_items` 或 `order_price` anomalies。把它放进 unit price 方程会污染价格。

**Q2：missing/outlier 为什么可以用？**  
A：assignment guide 明确限制了它们的问题类型。missing 的问题是 coverage/missing，outlier 的问题只在 `delivery_fee`。它们的 food items 和 food price 可以作为菜单价格参考。

**Q3：如何判断推导成功？**  
A：看 clean reference residual 是否接近 0。如果 residual 几乎为 0，说明 unit prices 能解释这些参考数据。

**Q4：下一步做什么？**  
A：用 `clean_unit_price_lookup` 重新检查 dirty 中的 `order_items` 和 `order_price`，这次 suspects 数量应该更合理。

**下一步：**  
用 clean unit prices 重新打印 dirty 的 menu-related 和 pure price suspects。

## Cell 14.4：用 clean unit prices 重新检查 dirty price/items suspects

现在使用只从 `missing_data` 和 `outlier_data` 推导出的 clean unit prices，重新检查 `dirty_cleaning`。

这一步要回答：真正需要关注的 `order_items` / `order_price` suspects 有多少？

In [27]:
clean_price_check = dirty_cleaning.copy()
clean_price_check['parsed_order_items'] = clean_price_check['order_items'].apply(parse_order_items)
clean_price_check['order_items_structure_valid'] = clean_price_check['parsed_order_items'].apply(validate_order_items_structure)
clean_price_check['expected_order_price'] = clean_price_check['parsed_order_items'].apply(calculate_clean_expected_order_price)
clean_price_check['order_price_residual'] = (
    clean_price_check['order_price'] - clean_price_check['expected_order_price']
).round(4)
clean_price_check['order_price_matches_items'] = clean_price_check['order_price_residual'].abs() < 0.01

clean_item_records = []
for _, row in clean_price_check.loc[clean_price_check['order_items_structure_valid']].iterrows():
    for item_name, quantity in row['parsed_order_items']:
        clean_item_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'item_name': item_name,
            'quantity': quantity,
            'expected_order_type_for_item': item_to_expected_meal.get(item_name),
        })

clean_item_records_df = pd.DataFrame(clean_item_records)
clean_menu_mismatch_order_ids = set(
    clean_item_records_df.loc[
        clean_item_records_df['order_type'] != clean_item_records_df['expected_order_type_for_item'],
        'order_id'
    ]
)

clean_price_check['has_menu_mismatch'] = clean_price_check['order_id'].isin(clean_menu_mismatch_order_ids)
clean_price_check['already_flagged'] = clean_price_check['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

clean_price_or_item_suspects = clean_price_check.loc[
    clean_price_check['has_menu_mismatch'] |
    (~clean_price_check['order_price_matches_items'])
].copy()

clean_menu_related_suspects = clean_price_or_item_suspects.loc[
    clean_price_or_item_suspects['has_menu_mismatch'] &
    (~clean_price_or_item_suspects['already_flagged'])
].copy()

clean_pure_price_suspects = clean_price_or_item_suspects.loc[
    (~clean_price_or_item_suspects['has_menu_mismatch']) &
    (~clean_price_or_item_suspects['order_price_matches_items']) &
    (~clean_price_or_item_suspects['already_flagged'])
].copy()

clean_price_items_summary = pd.DataFrame([
    {
        'check': 'all price/items suspect rows',
        'value': len(clean_price_or_item_suspects),
    },
    {
        'check': 'unflagged menu-related suspects',
        'value': len(clean_menu_related_suspects),
    },
    {
        'check': 'unflagged pure price suspects',
        'value': len(clean_pure_price_suspects),
    },
    {
        'check': 'already flagged price/items suspect rows',
        'value': int(clean_price_or_item_suspects['already_flagged'].sum()),
    },
])

display(clean_price_items_summary)

display(
    clean_menu_related_suspects[
        ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual']
    ].head(80)
)

display(
    clean_pure_price_suspects[
        ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual']
    ].head(80)
)

print('clean menu-related suspects:', len(clean_menu_related_suspects))
print('clean pure price suspects:', len(clean_pure_price_suspects))

,check,value
0,all price/items suspect rows,74
1,unflagged menu-related suspects,37
2,unflagged pure price suspects,37
3,already flagged price/items suspect rows,0


,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual
15,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]",912.0000,948.0000,-36.0000
29,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",301.2000,333.2000,-32.0000
80,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]",792.6000,784.6000,8.0000
106,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]",560.6000,938.6000,-378.0000
130,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]",394.0000,256.0000,138.0000
135,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]",351.5000,389.5000,-38.0000
143,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]",91.0000,87.5000,3.5000
147,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]",452.5000,245.5000,207.0000
151,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]",564.4000,731.0000,-166.6000
170,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]",406.0000,383.5000,22.5000


,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual
24,ORDA10507,Breakfast,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",173.0000,507.5000,-334.5000
50,ORDC08998,Breakfast,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), ('Cereal', 2)]",559.6000,266.7500,292.8500
59,ORDJ06050,Breakfast,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]","1,239.5000",457.0000,782.5000
64,ORDA07051,Breakfast,"[('Eggs', 9), ('Cereal', 9)]",967.0000,387.0000,580.0000
71,ORDA09458,Dinner,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",907.5000,801.0000,106.5000
73,ORDI09699,Breakfast,"[('Pancake', 4), ('Eggs', 8)]",43.0000,273.0000,-230.0000
84,ORDZ09320,Breakfast,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), ('Pancake', 2)]",367.0000,252.5000,114.5000
88,ORDY08024,Breakfast,"[('Pancake', 9), ('Cereal', 8), ('Coffee', 6)]",145.5000,431.2500,-285.7500
129,ORDZ03739,Lunch,"[('Fries', 3), ('Chicken', 8), ('Steak', 10), ('Burger', 7)]",269.7500,959.0000,-689.2500
133,ORDY06211,Dinner,"[('Shrimp', 7), ('Pasta', 10)]",971.0000,653.0000,318.0000


clean menu-related suspects: 37
clean pure price suspects: 37


### Cell 14.4 讨论：现在 suspects 数量合理了吗？

**Q1：这一步和前面的 372 有什么不同？**  
A：现在 unit prices 来自更干净的 missing/outlier reference，不再被 dirty 中的异常污染。如果数量明显下降，说明之前的 372 是 false positives。

**Q2：为什么还是分 menu-related 和 pure price？**  
A：menu-related 说明 item 和 meal type 不一致，优先处理 `order_items`；pure price 说明菜单合法但价格不一致，优先处理 `order_price`。

**Q3：下一步先修哪类？**  
A：先修 menu-related `order_items`，因为修正 item 后可能同时解决 price residual。然后再修 pure `order_price`。

**下一步：**  
对 clean menu-related suspects 搜索唯一 item replacement。

## Cell 14.5：为 clean menu-related suspects 搜索唯一 item replacement

现在对 `clean_menu_related_suspects` 搜索唯一 item replacement。

目标是找到这样的修复：替换一个不属于当前 meal type 的 item，quantity 不变，替换后价格等于 observed `order_price`。

In [28]:
def calculate_clean_price_from_items(item_list):
    total = 0.0
    for item_name, quantity in item_list:
        unit_price = clean_unit_price_lookup.get(item_name)
        if unit_price is None:
            return np.nan
        total += unit_price * quantity
    return round(total, 2)

def find_clean_item_replacement_candidates(row):
    parsed_items = row['parsed_order_items']
    if not validate_order_items_structure(parsed_items):
        return []

    current_order_type = row['order_type']
    allowed_items = meal_items[current_order_type]
    observed_price = round(float(row['order_price']), 2)
    candidates = []

    for position, (old_item, quantity) in enumerate(parsed_items):
        expected_meal_for_old_item = item_to_expected_meal.get(old_item)
        if expected_meal_for_old_item == current_order_type:
            continue

        for replacement_item in allowed_items:
            if replacement_item == old_item:
                continue
            candidate_items = list(parsed_items)
            candidate_items[position] = (replacement_item, quantity)
            candidate_price = calculate_clean_price_from_items(candidate_items)
            if pd.notna(candidate_price) and abs(candidate_price - observed_price) < 0.01:
                candidates.append({
                    'position': position,
                    'old_item': old_item,
                    'replacement_item': replacement_item,
                    'quantity': quantity,
                    'candidate_price': candidate_price,
                    'fixed_order_items': candidate_items,
                })

    return candidates

clean_order_items_candidate_rows = []

for _, row in clean_menu_related_suspects.iterrows():
    candidates = find_clean_item_replacement_candidates(row)
    for candidate in candidates:
        clean_order_items_candidate_rows.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'order_items': row['order_items'],
            'order_price': row['order_price'],
            'old_item': candidate['old_item'],
            'replacement_item': candidate['replacement_item'],
            'quantity': candidate['quantity'],
            'candidate_price': candidate['candidate_price'],
            'fixed_order_items': candidate['fixed_order_items'],
        })

clean_order_items_fix_candidates = pd.DataFrame(clean_order_items_candidate_rows)

display(clean_order_items_fix_candidates)

if len(clean_order_items_fix_candidates) > 0:
    clean_candidate_count_by_order = (
        clean_order_items_fix_candidates
        .groupby('order_id')
        .size()
        .reset_index(name='candidate_count')
    )
else:
    clean_candidate_count_by_order = pd.DataFrame(columns=['order_id', 'candidate_count'])

display(clean_candidate_count_by_order)

if len(clean_order_items_fix_candidates) > 0:
    unique_clean_order_items_fix_candidates = clean_order_items_fix_candidates.merge(
        clean_candidate_count_by_order.loc[
            clean_candidate_count_by_order['candidate_count'] == 1,
            ['order_id']
        ],
        on='order_id',
        how='inner'
    )
else:
    unique_clean_order_items_fix_candidates = pd.DataFrame(columns=list(clean_order_items_fix_candidates.columns))

display(unique_clean_order_items_fix_candidates)

print('clean menu-related suspect records:', len(clean_menu_related_suspects))
print('replacement candidate rows:', len(clean_order_items_fix_candidates))
print('records with unique replacement:', unique_clean_order_items_fix_candidates['order_id'].nunique() if len(unique_clean_order_items_fix_candidates) else 0)

,order_id,order_type,order_items,order_price,old_item,replacement_item,quantity,candidate_price,fixed_order_items
0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]",912.0000,Shrimp,Steak,4,912.0000,"[(Salad, 10), (Fries, 10), (Chicken, 6), (Burger, 8), (Steak, 4)]"
1,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",301.2000,Fish&Chips,Burger,8,301.2000,"[(Fries, 3), (Salad, 1), (Burger, 8)]"
2,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]",792.6000,Salmon,Steak,2,792.6000,"[(Salad, 8), (Fries, 5), (Burger, 7), (Steak, 2), (Chicken, 9)]"
3,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]",560.6000,Shrimp,Fries,9,560.6000,"[(Fries, 9), (Steak, 7), (Salad, 8)]"
4,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]",394.0000,Eggs,Steak,6,394.0000,"[(Burger, 4), (Steak, 6)]"
5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]",351.5000,Salmon,Eggs,2,351.5000,"[(Pancake, 6), (Cereal, 7), (Eggs, 2), (Coffee, 2)]"
6,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]",91.0000,Pasta,Burger,1,91.0000,"[(Fries, 5), (Burger, 1)]"
7,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]",452.5000,Fries,Fish&Chips,9,452.5000,"[(Fish&Chips, 9), (Pasta, 5)]"
8,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]",564.4000,Salmon,Salad,7,564.4000,"[(Burger, 8), (Fries, 3), (Chicken, 5), (Salad, 7)]"
9,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]",406.0000,Pasta,Chicken,5,406.0000,"[(Chicken, 5), (Fries, 5), (Burger, 6)]"


,order_id,candidate_count
0,ORDA02861,1
1,ORDA03303,1
2,ORDA03830,1
3,ORDB01016,1
4,ORDB02292,1
5,ORDB03431,1
6,ORDB03557,1
7,ORDB07450,1
8,ORDB07869,1
9,ORDB07956,1


,order_id,order_type,order_items,order_price,old_item,replacement_item,quantity,candidate_price,fixed_order_items
0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]",912.0000,Shrimp,Steak,4,912.0000,"[(Salad, 10), (Fries, 10), (Chicken, 6), (Burger, 8), (Steak, 4)]"
1,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",301.2000,Fish&Chips,Burger,8,301.2000,"[(Fries, 3), (Salad, 1), (Burger, 8)]"
2,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]",792.6000,Salmon,Steak,2,792.6000,"[(Salad, 8), (Fries, 5), (Burger, 7), (Steak, 2), (Chicken, 9)]"
3,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]",560.6000,Shrimp,Fries,9,560.6000,"[(Fries, 9), (Steak, 7), (Salad, 8)]"
4,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]",394.0000,Eggs,Steak,6,394.0000,"[(Burger, 4), (Steak, 6)]"
5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]",351.5000,Salmon,Eggs,2,351.5000,"[(Pancake, 6), (Cereal, 7), (Eggs, 2), (Coffee, 2)]"
6,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]",91.0000,Pasta,Burger,1,91.0000,"[(Fries, 5), (Burger, 1)]"
7,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]",452.5000,Fries,Fish&Chips,9,452.5000,"[(Fish&Chips, 9), (Pasta, 5)]"
8,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]",564.4000,Salmon,Salad,7,564.4000,"[(Burger, 8), (Fries, 3), (Chicken, 5), (Salad, 7)]"
9,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]",406.0000,Pasta,Chicken,5,406.0000,"[(Chicken, 5), (Fries, 5), (Burger, 6)]"


clean menu-related suspect records: 37
replacement candidate rows: 37
records with unique replacement: 37


### Cell 14.5 讨论：clean item replacement 是否足够唯一？

**Q1：为什么要求 replacement 后价格等于 observed price？**  
A：这样一个修复同时满足两个条件：item 属于正确菜单，且 order_price 被解释通。证据比只看菜单错配更强。

**Q2：如果 unique replacement 数量接近 menu-related suspects 数量，说明什么？**  
A：说明这些问题大多可以被解释为单个 item name 错误。

**Q3：如果某些记录没有唯一 replacement 呢？**  
A：不能强修。要进一步考虑 quantity 错误、多 item 错误，或回到 unit price 进行确认。

**下一步：**  
修复 unique clean item replacement records，并更新 tracker。

## Cell 14.6：修复 unique clean item replacement records 并验证

现在修复 `unique_clean_order_items_fix_candidates`。

只修改 `order_items`，并把这些 rows 标记为 `order_items` anomaly。修复后重新检查菜单错配和 price residual。

In [29]:
clean_order_items_fix_columns = [
    'order_id',
    'order_type',
    'order_items',
    'old_item',
    'replacement_item',
    'quantity',
    'fixed_order_items',
    'order_price',
    'candidate_price',
]

if len(unique_clean_order_items_fix_candidates) > 0:
    clean_order_items_fix_audit = unique_clean_order_items_fix_candidates[
        clean_order_items_fix_columns
    ].copy()
    clean_order_items_fix_audit['fixed_order_items_string'] = (
        clean_order_items_fix_audit['fixed_order_items'].apply(str)
    )
else:
    clean_order_items_fix_audit = pd.DataFrame(
        columns=clean_order_items_fix_columns + ['fixed_order_items_string']
    )

display(clean_order_items_fix_audit)

for _, row in clean_order_items_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'order_items'
    ] = row['fixed_order_items_string']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'order_items'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'wrong_menu_item'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"old_item={row['old_item']}; replacement={row['replacement_item']}; "
        f"order_price={row['order_price']}; candidate_price={row['candidate_price']}"
    )

post_clean_items = dirty_cleaning.copy()
post_clean_items['parsed_order_items'] = post_clean_items['order_items'].apply(parse_order_items)
post_clean_items['order_items_structure_valid'] = post_clean_items['parsed_order_items'].apply(validate_order_items_structure)
post_clean_items['expected_order_price'] = post_clean_items['parsed_order_items'].apply(calculate_clean_expected_order_price)
post_clean_items['order_price_residual'] = (
    post_clean_items['order_price'] - post_clean_items['expected_order_price']
).round(4)
post_clean_items['order_price_matches_items'] = post_clean_items['order_price_residual'].abs() < 0.01

post_clean_item_records = []
for _, row in post_clean_items.loc[post_clean_items['order_items_structure_valid']].iterrows():
    for item_name, quantity in row['parsed_order_items']:
        post_clean_item_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'item_name': item_name,
            'quantity': quantity,
            'expected_order_type_for_item': item_to_expected_meal.get(item_name),
        })

post_clean_item_records_df = pd.DataFrame(post_clean_item_records)
post_clean_menu_mismatch_order_ids = set(
    post_clean_item_records_df.loc[
        post_clean_item_records_df['order_type'] != post_clean_item_records_df['expected_order_type_for_item'],
        'order_id'
    ]
) if len(post_clean_item_records_df) else set()

post_clean_price_residuals = post_clean_items.loc[
    ~post_clean_items['order_price_matches_items']
].copy()
post_clean_price_residuals['already_flagged'] = post_clean_price_residuals['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)
post_clean_price_residuals['has_menu_mismatch'] = post_clean_price_residuals['order_id'].isin(
    post_clean_menu_mismatch_order_ids
)

post_clean_items_validation = pd.DataFrame([
    {
        'check': 'fixed order_items records',
        'value': len(clean_order_items_fix_audit),
    },
    {
        'check': 'menu mismatch rows after item fix',
        'value': len(post_clean_menu_mismatch_order_ids),
    },
    {
        'check': 'price residual rows after item fix',
        'value': len(post_clean_price_residuals),
    },
    {
        'check': 'unflagged pure price residual rows after item fix',
        'value': int(((~post_clean_price_residuals['already_flagged']) & (~post_clean_price_residuals['has_menu_mismatch'])).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_clean_items_validation)

display(
    post_clean_price_residuals[
        ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual', 'already_flagged', 'has_menu_mismatch']
    ].head(80)
)

tracker_summary_after_clean_items = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_clean_items)

print('total flagged rows after clean order_items fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,order_type,order_items,old_item,replacement_item,quantity,fixed_order_items,order_price,candidate_price,fixed_order_items_string
0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Shrimp', 4)]",Shrimp,Steak,4,"[(Salad, 10), (Fries, 10), (Chicken, 6), (Burger, 8), (Steak, 4)]",912.0000,912.0000,"[('Salad', 10), ('Fries', 10), ('Chicken', 6), ('Burger', 8), ('Steak', 4)]"
1,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",Fish&Chips,Burger,8,"[(Fries, 3), (Salad, 1), (Burger, 8)]",301.2000,301.2000,"[('Fries', 3), ('Salad', 1), ('Burger', 8)]"
2,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Salmon', 2), ('Chicken', 9)]",Salmon,Steak,2,"[(Salad, 8), (Fries, 5), (Burger, 7), (Steak, 2), (Chicken, 9)]",792.6000,792.6000,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('Steak', 2), ('Chicken', 9)]"
3,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]",Shrimp,Fries,9,"[(Fries, 9), (Steak, 7), (Salad, 8)]",560.6000,560.6000,"[('Fries', 9), ('Steak', 7), ('Salad', 8)]"
4,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]",Eggs,Steak,6,"[(Burger, 4), (Steak, 6)]",394.0000,394.0000,"[('Burger', 4), ('Steak', 6)]"
5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2), ('Coffee', 2)]",Salmon,Eggs,2,"[(Pancake, 6), (Cereal, 7), (Eggs, 2), (Coffee, 2)]",351.5000,351.5000,"[('Pancake', 6), ('Cereal', 7), ('Eggs', 2), ('Coffee', 2)]"
6,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]",Pasta,Burger,1,"[(Fries, 5), (Burger, 1)]",91.0000,91.0000,"[('Fries', 5), ('Burger', 1)]"
7,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]",Fries,Fish&Chips,9,"[(Fish&Chips, 9), (Pasta, 5)]",452.5000,452.5000,"[('Fish&Chips', 9), ('Pasta', 5)]"
8,ORDB02292,Lunch,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salmon', 7)]",Salmon,Salad,7,"[(Burger, 8), (Fries, 3), (Chicken, 5), (Salad, 7)]",564.4000,564.4000,"[('Burger', 8), ('Fries', 3), ('Chicken', 5), ('Salad', 7)]"
9,ORDY01014,Lunch,"[('Pasta', 5), ('Fries', 5), ('Burger', 6)]",Pasta,Chicken,5,"[(Chicken, 5), (Fries, 5), (Burger, 6)]",406.0000,406.0000,"[('Chicken', 5), ('Fries', 5), ('Burger', 6)]"


,check,value
0,fixed order_items records,37
1,menu mismatch rows after item fix,0
2,price residual rows after item fix,37
3,unflagged pure price residual rows after item fix,37
4,row count preserved,True


,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual,already_flagged,has_menu_mismatch
24,ORDA10507,Breakfast,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",173.0000,507.5000,-334.5000,False,False
50,ORDC08998,Breakfast,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), ('Cereal', 2)]",559.6000,266.7500,292.8500,False,False
59,ORDJ06050,Breakfast,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]","1,239.5000",457.0000,782.5000,False,False
64,ORDA07051,Breakfast,"[('Eggs', 9), ('Cereal', 9)]",967.0000,387.0000,580.0000,False,False
71,ORDA09458,Dinner,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",907.5000,801.0000,106.5000,False,False
73,ORDI09699,Breakfast,"[('Pancake', 4), ('Eggs', 8)]",43.0000,273.0000,-230.0000,False,False
84,ORDZ09320,Breakfast,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), ('Pancake', 2)]",367.0000,252.5000,114.5000,False,False
88,ORDY08024,Breakfast,"[('Pancake', 9), ('Cereal', 8), ('Coffee', 6)]",145.5000,431.2500,-285.7500,False,False
129,ORDZ03739,Lunch,"[('Fries', 3), ('Chicken', 8), ('Steak', 10), ('Burger', 7)]",269.7500,959.0000,-689.2500,False,False
133,ORDY06211,Dinner,"[('Shrimp', 7), ('Pasta', 10)]",971.0000,653.0000,318.0000,False,False


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,date,non_standard_format,17
3,order_items,wrong_menu_item,37
4,order_type,order_type_time_mismatch,37
5,unflagged,unflagged,372


total flagged rows after clean order_items fix: 128


### Cell 14.6 讨论：order_items 修复后的下一步是什么？

**Q1：修复后为什么还要重新算 price residual？**  
A：因为 `order_items` 和 `order_price` 是互相依赖的。item 修复后，expected price 会变化，剩下的 residual 才能更准确地归因。

**Q2：如果还剩 menu mismatch rows，怎么办？**  
A：说明这些 rows 没有唯一 replacement，不能强修。需要更复杂搜索或保留为后续复查。

**Q3：如果还剩 unflagged pure price residual rows，怎么办？**  
A：这些 rows 菜单已经合法，但 price 不匹配，可以在下一步修 `order_price`。

**下一步：**  
修复 unflagged pure `order_price` residual rows。

## Cell 14.7：修复 unflagged pure order_price residual rows

现在修复 pure `order_price` 问题。

这些 rows 满足：

- 当前没有被 tracker 标记；
- 没有 menu mismatch；
- `order_items` 可以解释出 expected price；
- observed `order_price` 和 expected price 不一致。

因此唯一合理修复是把 `order_price` 改成 expected price。

In [30]:
pure_order_price_fix_candidates = post_clean_price_residuals.loc[
    (~post_clean_price_residuals['already_flagged']) &
    (~post_clean_price_residuals['has_menu_mismatch']) &
    (post_clean_price_residuals['expected_order_price'].notna())
].copy()

order_price_fix_audit = pure_order_price_fix_candidates[
    ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual']
].copy()
order_price_fix_audit['order_price_will_change'] = (
    (order_price_fix_audit['order_price'] - order_price_fix_audit['expected_order_price']).abs() >= 0.01
)
order_price_fix_audit = order_price_fix_audit.loc[order_price_fix_audit['order_price_will_change']].copy()

display(order_price_fix_audit)

for _, row in order_price_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'order_price'
    ] = row['expected_order_price']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'order_price'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'price_items_mismatch'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed={row['order_price']}; expected={row['expected_order_price']}; "
        f"residual={row['order_price_residual']}"
    )

post_order_price_fix = dirty_cleaning.copy()
post_order_price_fix['parsed_order_items'] = post_order_price_fix['order_items'].apply(parse_order_items)
post_order_price_fix['order_items_structure_valid'] = post_order_price_fix['parsed_order_items'].apply(validate_order_items_structure)
post_order_price_fix['expected_order_price'] = post_order_price_fix['parsed_order_items'].apply(calculate_clean_expected_order_price)
post_order_price_fix['order_price_residual'] = (
    post_order_price_fix['order_price'] - post_order_price_fix['expected_order_price']
).round(4)
post_order_price_fix['order_price_matches_items'] = post_order_price_fix['order_price_residual'].abs() < 0.01

post_order_price_fix['already_flagged'] = post_order_price_fix['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)
post_order_price_fix['has_menu_mismatch'] = post_order_price_fix['order_id'].isin(
    post_clean_menu_mismatch_order_ids
)

remaining_after_order_price_fix = post_order_price_fix.loc[
    ~post_order_price_fix['order_price_matches_items']
].copy()

order_price_fix_validation = pd.DataFrame([
    {
        'check': 'fixed order_price records',
        'value': len(order_price_fix_audit),
    },
    {
        'check': 'remaining price residual rows after order_price fix',
        'value': len(remaining_after_order_price_fix),
    },
    {
        'check': 'remaining unflagged pure price residual rows after order_price fix',
        'value': int(((~remaining_after_order_price_fix['already_flagged']) & (~remaining_after_order_price_fix['has_menu_mismatch'])).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(order_price_fix_validation)

display(
    remaining_after_order_price_fix[
        ['order_id', 'order_type', 'order_items', 'order_price', 'expected_order_price', 'order_price_residual', 'already_flagged', 'has_menu_mismatch']
    ].head(80)
)

tracker_summary_after_order_price = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_order_price)

print('total flagged rows after order_price fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual,order_price_will_change
24,ORDA10507,Breakfast,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",173.0000,507.5000,-334.5000,True
50,ORDC08998,Breakfast,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), ('Cereal', 2)]",559.6000,266.7500,292.8500,True
59,ORDJ06050,Breakfast,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]","1,239.5000",457.0000,782.5000,True
64,ORDA07051,Breakfast,"[('Eggs', 9), ('Cereal', 9)]",967.0000,387.0000,580.0000,True
71,ORDA09458,Dinner,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",907.5000,801.0000,106.5000,True
73,ORDI09699,Breakfast,"[('Pancake', 4), ('Eggs', 8)]",43.0000,273.0000,-230.0000,True
84,ORDZ09320,Breakfast,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), ('Pancake', 2)]",367.0000,252.5000,114.5000,True
88,ORDY08024,Breakfast,"[('Pancake', 9), ('Cereal', 8), ('Coffee', 6)]",145.5000,431.2500,-285.7500,True
129,ORDZ03739,Lunch,"[('Fries', 3), ('Chicken', 8), ('Steak', 10), ('Burger', 7)]",269.7500,959.0000,-689.2500,True
133,ORDY06211,Dinner,"[('Shrimp', 7), ('Pasta', 10)]",971.0000,653.0000,318.0000,True


,check,value
0,fixed order_price records,37
1,remaining price residual rows after order_price fix,0
2,remaining unflagged pure price residual rows after order_price fix,0
3,row count preserved,True


,order_id,order_type,order_items,order_price,expected_order_price,order_price_residual,already_flagged,has_menu_mismatch


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,date,non_standard_format,17
3,order_items,wrong_menu_item,37
4,order_price,price_items_mismatch,37
5,order_type,order_type_time_mismatch,37
6,unflagged,unflagged,335


total flagged rows after order_price fix: 165


### Cell 14.7 讨论：order_price 修复完成了吗？

**Q1：为什么这些 rows 可以修 `order_price`？**  
A：因为它们没有 menu mismatch，也没有已有 flag，说明 `order_items` 更可信。价格 residual 的唯一解释就是 `order_price` 错。

**Q2：修复后为什么 remaining residual 可能还不是 0？**  
A：剩下的 residual 可能来自已经被其他 anomaly 解释的 rows，或者来自仍未唯一解决的 menu-related rows。它们不能被直接修成 price。

**Q3：最关键的 validation 是什么？**  
A：`remaining unflagged pure price residual rows after order_price fix` 应该为 0。它说明所有可归为 pure price 的问题都处理完了。

**下一步：**  
进入 `customer_lat`、`customer_lon` 和 `distance_to_customer_KM` 的图距离检查。

## Cell 15：图距离前的保护列规则和节点数据检查

进入坐标和距离之前，先记录 assignment guide 的保护规则。

Guide 明确说明 dirty data 中以下列是 error-free，不要找错：

- `order_id`
- `time`
- `order_items` 中的 numeric quantity
- `delivery_fee`

因此我们只检查和修复允许的字段。对于距离部分，`customer_lat`、`customer_lon` 来自 `nodes.csv`，`distance_to_customer_KM` 应该是 branch node 到 customer node 的 shortest path distance。

In [31]:
protected_dirty_columns = [
    'order_id',
    'time',
    'delivery_fee',
]
protected_order_items_component = 'numeric quantity in order_items'

node_data_summary = pd.DataFrame([
    {
        'file': 'nodes.csv',
        'rows': len(data['nodes']),
        'columns': list(data['nodes'].columns),
    },
    {
        'file': 'edges.csv',
        'rows': len(data['edges']),
        'columns': list(data['edges'].columns),
    },
    {
        'file': 'branches.csv',
        'rows': len(data['branches']),
        'columns': list(data['branches'].columns),
    },
])

display(node_data_summary)

display(data['branches'])

node_coordinates = set(
    zip(
        data['nodes']['lat'].round(7),
        data['nodes']['lon'].round(7),
    )
)

coordinate_check = dirty_cleaning.copy()
coordinate_check['customer_coord_key'] = list(
    zip(
        coordinate_check['customer_lat'].round(7),
        coordinate_check['customer_lon'].round(7),
    )
)
coordinate_check['customer_coordinate_in_nodes'] = coordinate_check['customer_coord_key'].isin(node_coordinates)

coordinate_summary = pd.DataFrame([
    {
        'check': 'customer coordinates not found in nodes.csv',
        'value': int((~coordinate_check['customer_coordinate_in_nodes']).sum()),
    },
    {
        'check': 'unique customer coordinate pairs',
        'value': coordinate_check['customer_coord_key'].nunique(),
    },
])

display(coordinate_summary)

display(
    coordinate_check.loc[
        ~coordinate_check['customer_coordinate_in_nodes'],
        ['order_id', 'customer_lat', 'customer_lon']
    ].head(30)
)

print('protected dirty columns:', protected_dirty_columns)
print('protected order_items component:', protected_order_items_component)

,file,rows,columns
0,nodes.csv,17117,"[node, lat, lon]"
1,edges.csv,42224,"[Unnamed: 0, u, v, distance(m), street type, speed(km/h)]"
2,branches.csv,3,"[branch_code, branch_name, branch_lat, branch_lon]"


,branch_code,branch_name,branch_lat,branch_lon
0,NS,Nickolson,-37.7738,144.9836
1,TP,Thompson,-37.8618,144.9057
2,BK,Bakers,-37.8158,145.0464


,check,value
0,customer coordinates not found in nodes.csv,41
1,unique customer coordinate pairs,493


,order_id,customer_lat,customer_lon
32,ORDZ03318,37.8062,144.9395
38,ORDZ06323,37.8142,144.9610
57,ORDK02131,37.8056,144.9487
62,ORDX01429,37.8142,144.9503
66,ORDA02101,37.8222,145.0041
78,ORDB06092,37.8083,144.9585
79,ORDB00776,37.8103,144.9471
93,ORDZ02223,37.8243,144.9544
111,ORDY03043,37.8120,144.9514
123,ORDA09210,37.8117,145.0121


protected dirty columns: ['order_id', 'time', 'delivery_fee']
protected order_items component: numeric quantity in order_items


### Cell 15 讨论：哪些列不能碰？为什么先查节点？

**Q1：为什么要记录 protected columns？**  
A：assignment guide 明确说这些列 error-free。即使某些检查看起来可疑，我们也不应该在 dirty task 中修它们。

**Q2：这对前面的 order_id/time 检查有什么影响？**  
A：我们可以检查它们来理解数据和推导其他字段，但不应该把它们作为 dirty 修复目标。`time` 可以用来修 `order_type`，但不修 `time` 本身。

**Q3：为什么先看 customer coordinates 是否在 nodes.csv？**  
A：因为 customer_lat/customer_lon 应该来自 nodes.csv。如果坐标不在 nodes 中，可能是坐标错误；如果在 nodes 中，才能映射到 customer node，再算 shortest path。

**Q4：下一步做什么？**  
A：建立 coordinate -> node 映射，构建 graph，用 Dijkstra 计算每条订单的 expected distance。

**下一步：**  
构建路网图并计算 expected `distance_to_customer_KM`。

## Cell 15.1：为不在 nodes.csv 的 customer coordinates 寻找最近节点证据

如果 customer coordinate 不在 `nodes.csv`，不能马上改。

我们先为这些 records 找最近的 node，然后用 graph distance 反向验证：如果最近 node 对应的 shortest path distance 和当前 `distance_to_customer_KM` 匹配，那么坐标更可能只是被轻微扰动；如果不匹配，则可能真正问题在 distance，或需要进一步检查。

In [32]:
import math
import heapq

nodes_lookup = data['nodes'].copy()
node_coord_array = nodes_lookup[['lat', 'lon']].to_numpy(dtype=float)
node_id_array = nodes_lookup['node'].to_numpy(dtype=int)

branch_node_lookup = {
    'NS': 2455254505,
    'TP': 1390575046,
    'BK': 1889485053,
}

coord_to_node = {
    (round(float(row.lat), 7), round(float(row.lon), 7)): int(row.node)
    for row in data['nodes'].itertuples(index=False)
}

def find_nearest_node(lat, lon):
    differences = node_coord_array - np.array([lat, lon], dtype=float)
    squared_distances = np.sum(differences ** 2, axis=1)
    position = int(np.argmin(squared_distances))
    return {
        'nearest_node': int(node_id_array[position]),
        'nearest_lat': float(node_coord_array[position, 0]),
        'nearest_lon': float(node_coord_array[position, 1]),
        'coordinate_delta': float(np.sqrt(squared_distances[position])),
    }

adjacency = {}
for row in data['edges'].itertuples(index=False):
    u = int(getattr(row, 'u'))
    v = int(getattr(row, 'v'))
    distance = float(getattr(row, '_3'))
    adjacency.setdefault(u, []).append((v, distance))
    adjacency.setdefault(v, []).append((u, distance))

def dijkstra_distance_m(source, target):
    if source == target:
        return 0.0
    queue = [(0.0, source)]
    seen = set()
    best = {source: 0.0}
    while queue:
        current_distance, node = heapq.heappop(queue)
        if node in seen:
            continue
        if node == target:
            return current_distance
        seen.add(node)
        for neighbor, edge_distance in adjacency.get(node, []):
            candidate_distance = current_distance + edge_distance
            if candidate_distance < best.get(neighbor, float('inf')):
                best[neighbor] = candidate_distance
                heapq.heappush(queue, (candidate_distance, neighbor))
    return np.nan

coordinate_not_in_nodes = coordinate_check.loc[
    ~coordinate_check['customer_coordinate_in_nodes']
].copy()

nearest_node_records = []
for _, row in coordinate_not_in_nodes.iterrows():
    nearest = find_nearest_node(row['customer_lat'], row['customer_lon'])
    branch_code = str(row['branch_code']).upper()
    branch_node = branch_node_lookup.get(branch_code)
    expected_distance_km = round(dijkstra_distance_m(branch_node, nearest['nearest_node']) / 1000, 3)
    nearest_node_records.append({
        'order_id': row['order_id'],
        'branch_code': row['branch_code'],
        'customer_lat': row['customer_lat'],
        'customer_lon': row['customer_lon'],
        'nearest_node': nearest['nearest_node'],
        'nearest_lat': nearest['nearest_lat'],
        'nearest_lon': nearest['nearest_lon'],
        'coordinate_delta': nearest['coordinate_delta'],
        'current_distance_to_customer_KM': row['distance_to_customer_KM'],
        'expected_distance_to_nearest_node_KM': expected_distance_km,
        'distance_difference_if_coordinate_fixed': round(row['distance_to_customer_KM'] - expected_distance_km, 3),
    })

nearest_coordinate_evidence = pd.DataFrame(nearest_node_records)
if len(nearest_coordinate_evidence) > 0:
    nearest_coordinate_evidence['current_distance_matches_nearest_node'] = (
        nearest_coordinate_evidence['distance_difference_if_coordinate_fixed'].abs() <= 0.001
    )

display(nearest_coordinate_evidence)

print('customer coordinate records not found in nodes.csv:', len(nearest_coordinate_evidence))

,order_id,branch_code,customer_lat,customer_lon,nearest_node,nearest_lat,nearest_lon,coordinate_delta,current_distance_to_customer_KM,expected_distance_to_nearest_node_KM,distance_difference_if_coordinate_fixed,current_distance_matches_nearest_node
0,ORDZ03318,NS,37.8062,144.9395,256189632,-37.7396,144.8463,75.5459,9.9510,17.0960,-7.1450,False
1,ORDZ06323,NS,37.8142,144.9610,256189632,-37.7396,144.8463,75.5539,8.2150,17.0960,-8.8810,False
2,ORDK02131,BK,37.8056,144.9487,256189632,-37.7396,144.8463,75.5453,8.7280,22.6620,-13.9340,False
3,ORDX01429,BK,37.8142,144.9503,256189632,-37.7396,144.8463,75.5539,10.6950,22.6620,-11.9670,False
4,ORDA02101,BK,37.8222,145.0041,256189632,-37.7396,144.8463,75.5620,4.9750,22.6620,-17.6870,False
5,ORDB06092,TP,37.8083,144.9585,256189632,-37.7396,144.8463,75.5480,8.9280,24.1210,-15.1930,False
6,ORDB00776,TP,37.8103,144.9471,256189632,-37.7396,144.8463,75.5500,9.6130,24.1210,-14.5080,False
7,ORDZ02223,NS,37.8243,144.9544,256189632,-37.7396,144.8463,75.5640,9.8940,17.0960,-7.2020,False
8,ORDY03043,TP,37.8120,144.9514,256189632,-37.7396,144.8463,75.5517,9.1430,24.1210,-14.9780,False
9,ORDA09210,BK,37.8117,145.0121,256189632,-37.7396,144.8463,75.5515,3.2340,22.6620,-19.4280,False


customer coordinate records not found in nodes.csv: 41


### Cell 15.1 讨论：坐标不在 nodes.csv 时怎么判断？

**Q1：为什么找 nearest node？**  
A：如果坐标只是轻微扰动，最近的 node 往往就是正确 customer node。修复可以把坐标改回 nodes.csv 中的精确 lat/lon。

**Q2：为什么还要比较 distance？**  
A：因为 customer coordinate 和 `distance_to_customer_KM` 是联动的。如果用最近 node 计算出的 shortest path distance 正好等于当前 distance，说明当前 distance 支持“坐标错”的解释。

**Q3：如果 nearest node 的 distance 不匹配怎么办？**  
A：那就不能直接修坐标。可能真正错的是 `distance_to_customer_KM`，或者需要找另一个能解释当前 distance 的 node。

**Q4：下一步做什么？**  
A：如果有坐标不在 nodes 的 records，先检查它们是否已经被 tracker 标记。未标记且 distance 支持 nearest node 的，才可以修 customer coordinates。

**下一步：**  
构建完整 expected distance，并检查 `distance_to_customer_KM`。

## Cell 15.1：检测 customer_lat 符号错误

刚才 overview 里发现有些 `customer_lat` 不在 `nodes.csv`。真实 output 显示这些纬度是正数 `37.x`，但 Melbourne 和 `nodes.csv` 的纬度应为负数 `-37.x`。

所以我们先检查最明显的错误：`customer_lat` 符号是否写反。

In [33]:
node_coordinate_keys = set(
    zip(
        data['nodes']['lat'].round(7),
        data['nodes']['lon'].round(7),
    )
)

lat_sign_check = coordinate_check.loc[
    ~coordinate_check['customer_coordinate_in_nodes']
].copy()

lat_sign_check['corrected_customer_lat'] = -lat_sign_check['customer_lat']
lat_sign_check['corrected_coord_key'] = list(
    zip(
        lat_sign_check['corrected_customer_lat'].round(7),
        lat_sign_check['customer_lon'].round(7),
    )
)
lat_sign_check['corrected_coordinate_in_nodes'] = lat_sign_check['corrected_coord_key'].isin(node_coordinate_keys)

lat_sign_check['already_flagged'] = lat_sign_check['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

lat_sign_suspects = lat_sign_check.loc[
    lat_sign_check['corrected_coordinate_in_nodes'] &
    (~lat_sign_check['already_flagged'])
].copy()

display(
    lat_sign_check[
        [
            'order_id',
            'customer_lat',
            'customer_lon',
            'corrected_customer_lat',
            'corrected_coordinate_in_nodes',
            'already_flagged',
            'distance_to_customer_KM',
        ]
    ]
)

display(
    lat_sign_suspects[
        [
            'order_id',
            'customer_lat',
            'customer_lon',
            'corrected_customer_lat',
            'distance_to_customer_KM',
        ]
    ]
)

print('coordinates not found in nodes.csv:', len(lat_sign_check))
print('latitude sign error candidates:', len(lat_sign_suspects))

,order_id,customer_lat,customer_lon,corrected_customer_lat,corrected_coordinate_in_nodes,already_flagged,distance_to_customer_KM
32,ORDZ03318,37.8062,144.9395,-37.8062,True,False,9.9510
38,ORDZ06323,37.8142,144.9610,-37.8142,True,False,8.2150
57,ORDK02131,37.8056,144.9487,-37.8056,True,False,8.7280
62,ORDX01429,37.8142,144.9503,-37.8142,True,False,10.6950
66,ORDA02101,37.8222,145.0041,-37.8222,True,False,4.9750
78,ORDB06092,37.8083,144.9585,-37.8083,True,False,8.9280
79,ORDB00776,37.8103,144.9471,-37.8103,True,False,9.6130
93,ORDZ02223,37.8243,144.9544,-37.8243,True,False,9.8940
111,ORDY03043,37.8120,144.9514,-37.8120,True,False,9.1430
123,ORDA09210,37.8117,145.0121,-37.8117,True,False,3.2340


,order_id,customer_lat,customer_lon,corrected_customer_lat,distance_to_customer_KM
32,ORDZ03318,37.8062,144.9395,-37.8062,9.9510
38,ORDZ06323,37.8142,144.9610,-37.8142,8.2150
57,ORDK02131,37.8056,144.9487,-37.8056,8.7280
62,ORDX01429,37.8142,144.9503,-37.8142,10.6950
66,ORDA02101,37.8222,145.0041,-37.8222,4.9750
78,ORDB06092,37.8083,144.9585,-37.8083,8.9280
79,ORDB00776,37.8103,144.9471,-37.8103,9.6130
93,ORDZ02223,37.8243,144.9544,-37.8243,9.8940
111,ORDY03043,37.8120,144.9514,-37.8120,9.1430
123,ORDA09210,37.8117,145.0121,-37.8117,3.2340


coordinates not found in nodes.csv: 41
latitude sign error candidates: 37


### Cell 15.1 讨论：为什么先修 latitude sign error？

**Q1：为什么不是先 nearest node？**  
A：因为真实 output 显示 `customer_lat` 是正数 `37.x`，而 Melbourne 纬度应为负数 `-37.x`。这不是小偏移，而是明显的符号错误。

**Q2：怎样验证 sign error？**  
A：把 `customer_lat` 取负后，检查 `(corrected_customer_lat, customer_lon)` 是否存在于 `nodes.csv`。如果存在，这是很强的证据。

**Q3：为什么还要看 tracker？**  
A：如果某行已经被其他 anomaly 解释，就不能再修坐标。这里只保留未被 flag 的 sign error candidates。

**Q4：下一步做什么？**  
A：修复这些 latitude sign error candidates，然后再计算 graph distance 验证 `distance_to_customer_KM`。

**下一步：**  
把 sign error candidates 的 `customer_lat` 改成负数，并更新 tracker。

## Cell 15.2：修复 customer_lat 符号错误并验证坐标来源

现在修复 `lat_sign_suspects`。

修复原则：

- 只修复取负后能匹配 `nodes.csv` 的 rows；
- 只修复未被 tracker 标记过的 rows；
- 只修改 `customer_lat`，不修改 `customer_lon` 或 distance；
- 修复后重新验证 customer coordinates 是否都来自 `nodes.csv`。

In [34]:
lat_fix_audit = lat_sign_suspects[
    [
        'order_id',
        'customer_lat',
        'customer_lon',
        'corrected_customer_lat',
        'distance_to_customer_KM',
    ]
].copy()
lat_fix_audit['customer_lat_will_change'] = (
    lat_fix_audit['customer_lat'] != lat_fix_audit['corrected_customer_lat']
)

display(lat_fix_audit)

for _, row in lat_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'customer_lat'
    ] = row['corrected_customer_lat']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'customer_lat'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'latitude_sign_error'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed_lat={row['customer_lat']}; corrected_lat={row['corrected_customer_lat']}; "
        f"lon={row['customer_lon']}"
    )

post_coordinate_check = dirty_cleaning.copy()
post_coordinate_check['customer_coord_key'] = list(
    zip(
        post_coordinate_check['customer_lat'].round(7),
        post_coordinate_check['customer_lon'].round(7),
    )
)
post_coordinate_check['customer_coordinate_in_nodes'] = post_coordinate_check['customer_coord_key'].isin(node_coordinate_keys)

post_coordinate_validation = pd.DataFrame([
    {
        'check': 'fixed customer_lat sign records',
        'value': len(lat_fix_audit),
    },
    {
        'check': 'customer coordinates not found in nodes.csv after fix',
        'value': int((~post_coordinate_check['customer_coordinate_in_nodes']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_coordinate_validation)

display(
    post_coordinate_check.loc[
        ~post_coordinate_check['customer_coordinate_in_nodes'],
        ['order_id', 'customer_lat', 'customer_lon']
    ]
)

tracker_summary_after_lat = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_lat)

print('total flagged rows after customer_lat fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,customer_lat,customer_lon,corrected_customer_lat,distance_to_customer_KM,customer_lat_will_change
32,ORDZ03318,37.8062,144.9395,-37.8062,9.9510,True
38,ORDZ06323,37.8142,144.9610,-37.8142,8.2150,True
57,ORDK02131,37.8056,144.9487,-37.8056,8.7280,True
62,ORDX01429,37.8142,144.9503,-37.8142,10.6950,True
66,ORDA02101,37.8222,145.0041,-37.8222,4.9750,True
78,ORDB06092,37.8083,144.9585,-37.8083,8.9280,True
79,ORDB00776,37.8103,144.9471,-37.8103,9.6130,True
93,ORDZ02223,37.8243,144.9544,-37.8243,9.8940,True
111,ORDY03043,37.8120,144.9514,-37.8120,9.1430,True
123,ORDA09210,37.8117,145.0121,-37.8117,3.2340,True


,check,value
0,fixed customer_lat sign records,37
1,customer coordinates not found in nodes.csv after fix,4
2,row count preserved,True


,order_id,customer_lat,customer_lon
200,ORDC05483,145.0015,-37.8114
210,ORDK00150,145.0074,-37.8162
254,ORDY06420,144.9885,-37.8246
374,ORDX05398,144.9564,-37.8118


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_lat,latitude_sign_error,37
3,date,non_standard_format,17
4,order_items,wrong_menu_item,37
5,order_price,price_items_mismatch,37
6,order_type,order_type_time_mismatch,37
7,unflagged,unflagged,298


total flagged rows after customer_lat fix: 202


### Cell 15.2 讨论：customer_lat 修复完成了吗？

**Q1：为什么这一步只改 `customer_lat`？**  
A：真实 output 显示问题是纬度正负号。`customer_lon` 本来就在 Melbourne 经度范围内，所以不应该一起改。

**Q2：为什么取负后能匹配 nodes.csv 是强证据？**  
A：因为 assignment guide 说明 customer coordinates 来自 `nodes.csv`。取负后精确匹配节点，说明原值很可能只是符号错误。

**Q3：修复后 validation 看什么？**  
A：看所有 customer coordinates 是否都能在 `nodes.csv` 找到。如果为 0，说明坐标输入层面已经干净。

**Q4：下一步是什么？**  
A：现在 customer node 可以确定了，接下来才检查 `distance_to_customer_KM` 是否等于 branch 到 customer node 的 shortest path distance。

**下一步：**  
构建 graph，计算每条订单的 expected distance，并检查 distance mismatch。

## Cell 15.3：检测并修复 customer_lat / customer_lon 交换错误

修复 latitude sign error 后，仍有 4 行 customer coordinates 不在 `nodes.csv`。

真实 output 显示这些行的形态是：

- `customer_lat` 像 Melbourne 经度，约 `144.x` 或 `145.x`；
- `customer_lon` 像 Melbourne 纬度，约 `-37.x`。

所以这更像是 `customer_lat` 和 `customer_lon` 被交换。

In [35]:
remaining_coordinate_issues = post_coordinate_check.loc[
    ~post_coordinate_check['customer_coordinate_in_nodes']
].copy()

swap_check = remaining_coordinate_issues.copy()
swap_check['swapped_lat'] = swap_check['customer_lon']
swap_check['swapped_lon'] = swap_check['customer_lat']
swap_check['swapped_coord_key'] = list(
    zip(
        swap_check['swapped_lat'].round(7),
        swap_check['swapped_lon'].round(7),
    )
)
swap_check['swapped_coordinate_in_nodes'] = swap_check['swapped_coord_key'].isin(node_coordinate_keys)
swap_check['already_flagged'] = swap_check['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

swap_suspects = swap_check.loc[
    swap_check['swapped_coordinate_in_nodes'] &
    (~swap_check['already_flagged'])
].copy()

display(
    swap_check[
        [
            'order_id',
            'customer_lat',
            'customer_lon',
            'swapped_lat',
            'swapped_lon',
            'swapped_coordinate_in_nodes',
            'already_flagged',
            'distance_to_customer_KM',
        ]
    ]
)

display(swap_suspects[
    ['order_id', 'customer_lat', 'customer_lon', 'swapped_lat', 'swapped_lon', 'distance_to_customer_KM']
])

print('remaining coordinate issues before swap fix:', len(remaining_coordinate_issues))
print('lat/lon swap candidates:', len(swap_suspects))

,order_id,customer_lat,customer_lon,swapped_lat,swapped_lon,swapped_coordinate_in_nodes,already_flagged,distance_to_customer_KM
200,ORDC05483,145.0015,-37.8114,-37.8114,145.0015,True,False,9.6580
210,ORDK00150,145.0074,-37.8162,-37.8162,145.0074,True,False,3.9370
254,ORDY06420,144.9885,-37.8246,-37.8246,144.9885,True,False,9.1930
374,ORDX05398,144.9564,-37.8118,-37.8118,144.9564,True,False,8.0870


,order_id,customer_lat,customer_lon,swapped_lat,swapped_lon,distance_to_customer_KM
200,ORDC05483,145.0015,-37.8114,-37.8114,145.0015,9.6580
210,ORDK00150,145.0074,-37.8162,-37.8162,145.0074,3.9370
254,ORDY06420,144.9885,-37.8246,-37.8246,144.9885,9.1930
374,ORDX05398,144.9564,-37.8118,-37.8118,144.9564,8.0870


remaining coordinate issues before swap fix: 4
lat/lon swap candidates: 4


### Cell 15.3 讨论：为什么判断是 lat/lon swapped？

**Q1：为什么不是继续取负？**  
A：因为这些 rows 的 `customer_lat` 是 `144.x/145.x`，明显像经度；`customer_lon` 是 `-37.x`，明显像纬度。问题不是符号，而是两列位置交换。

**Q2：如何验证 swapped 修复？**  
A：交换后检查 `(swapped_lat, swapped_lon)` 是否存在于 `nodes.csv`。如果存在，就是强证据。

**Q3：为什么还要看 tracker？**  
A：继续遵守一行最多一个 anomaly。这里只修未被其他问题标记的 rows。

**下一步：**  
把 swap candidates 的 `customer_lat` 和 `customer_lon` 交换，并重新验证所有 customer coordinates。

## Cell 15.4：修复 customer_lat / customer_lon 交换错误并验证

现在修复 `swap_suspects`。

修复原则：

- 只修复交换后能匹配 `nodes.csv` 的 rows；
- 只修复未被 tracker 标记的 rows；
- 同时交换 `customer_lat` 和 `customer_lon`；
- 修复后重新验证所有 customer coordinates 是否来自 `nodes.csv`。

In [36]:
swap_fix_audit = swap_suspects[
    [
        'order_id',
        'customer_lat',
        'customer_lon',
        'swapped_lat',
        'swapped_lon',
        'distance_to_customer_KM',
    ]
].copy()

display(swap_fix_audit)

for _, row in swap_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'customer_lat'
    ] = row['swapped_lat']
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'customer_lon'
    ] = row['swapped_lon']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'customer_coordinates'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'lat_lon_swapped'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed_lat={row['customer_lat']}; observed_lon={row['customer_lon']}; "
        f"fixed_lat={row['swapped_lat']}; fixed_lon={row['swapped_lon']}"
    )

post_swap_coordinate_check = dirty_cleaning.copy()
post_swap_coordinate_check['customer_coord_key'] = list(
    zip(
        post_swap_coordinate_check['customer_lat'].round(7),
        post_swap_coordinate_check['customer_lon'].round(7),
    )
)
post_swap_coordinate_check['customer_coordinate_in_nodes'] = post_swap_coordinate_check['customer_coord_key'].isin(node_coordinate_keys)

post_swap_validation = pd.DataFrame([
    {
        'check': 'fixed lat/lon swapped records',
        'value': len(swap_fix_audit),
    },
    {
        'check': 'customer coordinates not found in nodes.csv after swap fix',
        'value': int((~post_swap_coordinate_check['customer_coordinate_in_nodes']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_swap_validation)

display(
    post_swap_coordinate_check.loc[
        ~post_swap_coordinate_check['customer_coordinate_in_nodes'],
        ['order_id', 'customer_lat', 'customer_lon']
    ]
)

tracker_summary_after_swap = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_swap)

print('total flagged rows after lat/lon swap fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,customer_lat,customer_lon,swapped_lat,swapped_lon,distance_to_customer_KM
200,ORDC05483,145.0015,-37.8114,-37.8114,145.0015,9.6580
210,ORDK00150,145.0074,-37.8162,-37.8162,145.0074,3.9370
254,ORDY06420,144.9885,-37.8246,-37.8246,144.9885,9.1930
374,ORDX05398,144.9564,-37.8118,-37.8118,144.9564,8.0870


,check,value
0,fixed lat/lon swapped records,4
1,customer coordinates not found in nodes.csv after swap fix,0
2,row count preserved,True


,order_id,customer_lat,customer_lon


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_coordinates,lat_lon_swapped,4
3,customer_lat,latitude_sign_error,37
4,date,non_standard_format,17
5,order_items,wrong_menu_item,37
6,order_price,price_items_mismatch,37
7,order_type,order_type_time_mismatch,37
8,unflagged,unflagged,294


total flagged rows after lat/lon swap fix: 206


### Cell 15.4 讨论：坐标输入层是否完成？

**Q1：修复后最关键看什么？**  
A：看 `customer coordinates not found in nodes.csv after swap fix` 是否为 0。如果为 0，说明所有 customer coordinates 都能映射到 nodes。

**Q2：为什么现在才进入 distance？**  
A：因为 distance 是由 branch node 和 customer node 推导出来的。只有 customer coordinates 都能确定 node 后，distance 检查才有可靠输入。

**Q3：下一步是什么？**  
A：用 graph shortest path 计算每条订单的 expected `distance_to_customer_KM`，再找出 distance mismatch records。

**下一步：**  
构建/使用 graph shortest paths，检查并修复 `distance_to_customer_KM`。

## Cell 16：计算 expected distance_to_customer_KM 并打印 suspects

现在 customer coordinates 都能映射到 `nodes.csv`，可以检查 `distance_to_customer_KM`。

`distance_to_customer_KM` 应该等于 branch node 到 customer node 的 graph shortest path distance。这里先计算 expected distance，并打印 mismatch suspects。

In [37]:
import networkx as nx

branch_node_lookup = {}
for _, branch in data['branches'].iterrows():
    branch_coord_key = (round(branch['branch_lat'], 7), round(branch['branch_lon'], 7))
    if branch_coord_key in coord_to_node:
        branch_node_lookup[branch['branch_code']] = int(coord_to_node[branch_coord_key])
    else:
        nearest_node, _, _, _ = find_nearest_node(branch['branch_lat'], branch['branch_lon'])
        branch_node_lookup[branch['branch_code']] = nearest_node

road_graph = nx.Graph()
for _, edge in data['edges'].iterrows():
    road_graph.add_edge(
        int(edge['u']),
        int(edge['v']),
        weight=float(edge['distance(m)']),
    )

branch_shortest_paths = {
    branch_code: nx.single_source_dijkstra_path_length(
        road_graph,
        branch_node,
        weight='weight',
    )
    for branch_code, branch_node in branch_node_lookup.items()
}

distance_check = dirty_cleaning.copy()
distance_check['customer_coord_key'] = list(
    zip(
        distance_check['customer_lat'].round(7),
        distance_check['customer_lon'].round(7),
    )
)
distance_check['customer_node'] = distance_check['customer_coord_key'].map(coord_to_node)

def expected_distance_km(row):
    branch_code = row['branch_code']
    customer_node = int(row['customer_node'])
    distance_m = branch_shortest_paths[branch_code][customer_node]
    return round(distance_m / 1000, 3)

distance_check['expected_distance_to_customer_KM'] = distance_check.apply(expected_distance_km, axis=1)
distance_check['distance_difference'] = (
    distance_check['distance_to_customer_KM'] - distance_check['expected_distance_to_customer_KM']
).round(3)
distance_check['distance_matches_graph'] = distance_check['distance_difference'].abs() <= 0.001

distance_suspects = distance_check.loc[
    ~distance_check['distance_matches_graph']
].copy()
distance_suspects['already_flagged'] = distance_suspects['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

unflagged_distance_suspects = distance_suspects.loc[
    ~distance_suspects['already_flagged']
].copy()

distance_summary = pd.DataFrame([
    {
        'check': 'distance mismatch rows',
        'value': len(distance_suspects),
    },
    {
        'check': 'distance suspects already flagged before',
        'value': int(distance_suspects['already_flagged'].sum()),
    },
    {
        'check': 'unflagged distance suspects',
        'value': len(unflagged_distance_suspects),
    },
])

display(pd.DataFrame([
    {'branch_code': branch_code, 'branch_node': branch_node}
    for branch_code, branch_node in branch_node_lookup.items()
]))

display(distance_summary)

display(
    unflagged_distance_suspects[
        [
            'order_id',
            'branch_code',
            'customer_node',
            'distance_to_customer_KM',
            'expected_distance_to_customer_KM',
            'distance_difference',
        ]
    ].head(80)
)

print('distance mismatch rows:', len(distance_suspects))
print('unflagged distance suspects:', len(unflagged_distance_suspects))

,branch_code,branch_node
0,NS,2455254505
1,TP,1390575046
2,BK,1889485053


,check,value
0,distance mismatch rows,37
1,distance suspects already flagged before,0
2,unflagged distance suspects,37


,order_id,branch_code,customer_node,distance_to_customer_KM,expected_distance_to_customer_KM,distance_difference
8,ORDY07813,TP,2185303480,7.6510,12.8470,-5.1960
12,ORDJ08517,TP,367849063,7.9620,9.7640,-1.8020
19,ORDZ08993,NS,1327054074,7.0660,7.8570,-0.7910
25,ORDY03212,TP,777721819,7.9750,9.3690,-1.3940
58,ORDJ10405,TP,233274958,11.3230,10.3270,0.9960
91,ORDC00387,NS,1492350746,9.9790,6.8600,3.1190
110,ORDC03440,NS,35524096,4.9750,7.6510,-2.6760
152,ORDX09640,BK,3512300807,8.5620,4.0110,4.5510
163,ORDJ05287,TP,259616744,7.6150,9.2240,-1.6090
184,ORDJ04387,TP,2700682445,8.7870,8.2560,0.5310


distance mismatch rows: 37
unflagged distance suspects: 37


### Cell 16 讨论：distance mismatch 如何归因？

**Q1：为什么现在才检查 distance？**  
A：distance 依赖 branch code 和 customer node。我们已经修复 branch_code 和 customer coordinates，所以现在 expected distance 才有可靠输入。

**Q2：为什么还要看 already flagged？**  
A：如果某行已经被其他 anomaly 解释，根据一行最多一个 anomaly，就不再修 distance。未被 flag 的 distance mismatch 才是候选 `distance_to_customer_KM` 错误。

**Q3：修复依据是什么？**  
A：assignment guide 定义 distance 是 graph shortest path distance，所以 expected distance 是由 Dijkstra 计算出的唯一值。

**下一步：**  
修复 unflagged `distance_to_customer_KM` suspects，并更新 tracker。

## Cell 16.1：修复 distance_to_customer_KM 并验证

现在修复 `unflagged_distance_suspects`。

修复依据是 Dijkstra shortest path distance。因为 branch_code 和 customer coordinates 已经验证过，所以 expected distance 是唯一可解释值。

In [38]:
distance_fix_audit = unflagged_distance_suspects[
    [
        'order_id',
        'branch_code',
        'customer_node',
        'distance_to_customer_KM',
        'expected_distance_to_customer_KM',
        'distance_difference',
    ]
].copy()

display(distance_fix_audit)

for _, row in distance_fix_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'distance_to_customer_KM'
    ] = row['expected_distance_to_customer_KM']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'distance_to_customer_KM'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'graph_distance_mismatch'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed={row['distance_to_customer_KM']}; "
        f"expected={row['expected_distance_to_customer_KM']}; "
        f"difference={row['distance_difference']}"
    )

post_distance_check = dirty_cleaning.copy()
post_distance_check['customer_coord_key'] = list(
    zip(
        post_distance_check['customer_lat'].round(7),
        post_distance_check['customer_lon'].round(7),
    )
)
post_distance_check['customer_node'] = post_distance_check['customer_coord_key'].map(coord_to_node)
post_distance_check['expected_distance_to_customer_KM'] = post_distance_check.apply(expected_distance_km, axis=1)
post_distance_check['distance_difference'] = (
    post_distance_check['distance_to_customer_KM'] - post_distance_check['expected_distance_to_customer_KM']
).round(3)
post_distance_check['distance_matches_graph'] = post_distance_check['distance_difference'].abs() <= 0.001

remaining_distance_suspects = post_distance_check.loc[
    ~post_distance_check['distance_matches_graph']
].copy()
remaining_distance_suspects['already_flagged'] = remaining_distance_suspects['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

post_distance_validation = pd.DataFrame([
    {
        'check': 'fixed distance records',
        'value': len(distance_fix_audit),
    },
    {
        'check': 'remaining distance mismatch rows after fix',
        'value': len(remaining_distance_suspects),
    },
    {
        'check': 'remaining unflagged distance mismatch rows after fix',
        'value': int((~remaining_distance_suspects['already_flagged']).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_distance_validation)

display(
    remaining_distance_suspects[
        ['order_id', 'branch_code', 'distance_to_customer_KM', 'expected_distance_to_customer_KM', 'distance_difference', 'already_flagged']
    ]
)

tracker_summary_after_distance = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_distance)

print('total flagged rows after distance fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,branch_code,customer_node,distance_to_customer_KM,expected_distance_to_customer_KM,distance_difference
8,ORDY07813,TP,2185303480,7.6510,12.8470,-5.1960
12,ORDJ08517,TP,367849063,7.9620,9.7640,-1.8020
19,ORDZ08993,NS,1327054074,7.0660,7.8570,-0.7910
25,ORDY03212,TP,777721819,7.9750,9.3690,-1.3940
58,ORDJ10405,TP,233274958,11.3230,10.3270,0.9960
91,ORDC00387,NS,1492350746,9.9790,6.8600,3.1190
110,ORDC03440,NS,35524096,4.9750,7.6510,-2.6760
152,ORDX09640,BK,3512300807,8.5620,4.0110,4.5510
163,ORDJ05287,TP,259616744,7.6150,9.2240,-1.6090
184,ORDJ04387,TP,2700682445,8.7870,8.2560,0.5310


,check,value
0,fixed distance records,37
1,remaining distance mismatch rows after fix,0
2,remaining unflagged distance mismatch rows after fix,0
3,row count preserved,True


,order_id,branch_code,distance_to_customer_KM,expected_distance_to_customer_KM,distance_difference,already_flagged


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_coordinates,lat_lon_swapped,4
3,customer_lat,latitude_sign_error,37
4,date,non_standard_format,17
5,distance_to_customer_KM,graph_distance_mismatch,37
6,order_items,wrong_menu_item,37
7,order_price,price_items_mismatch,37
8,order_type,order_type_time_mismatch,37
9,unflagged,unflagged,257


total flagged rows after distance fix: 243


### Cell 16.1 讨论：distance 修复完成了吗？

**Q1：为什么这些 rows 可以直接修 distance？**  
A：因为 branch_code 和 customer coordinates 已经验证过，Dijkstra shortest path 给出唯一 expected distance。当前 distance 与 expected 不一致，所以 anomaly 是 `distance_to_customer_KM`。

**Q2：修复后 validation 看什么？**  
A：看 remaining distance mismatch rows 是否为 0，或者至少没有 unflagged mismatch rows。

**Q3：下一步是什么？**  
A：dirty_data 中 guide 说 `delivery_fee` 是 error-free，所以 dirty task 不修 delivery_fee。接下来做最终 validation：检查所有已处理规则、tracker 数量、列结构和 row count。

**下一步：**  
做 dirty_data final validation。

## Cell 17：dirty_data final validation

现在对 `dirty_cleaning` 做最终验证。

这一步不再修复新问题，而是检查：

- row count 和 column order 是否保持；
- protected columns 是否没有被修改；
- 已经处理过的业务规则是否全部通过；
- tracker 是否没有重复标记同一行。

In [39]:
final_dirty_check = dirty_cleaning.copy()
original_dirty = data['dirty'].copy()

final_dirty_check['order_prefix'] = final_dirty_check['order_id'].str.extract(r'^ORD([A-Z])')
final_dirty_check['expected_branch_code'] = final_dirty_check['order_prefix'].map(prefix_to_expected_branch)
final_dirty_check['branch_code_valid'] = final_dirty_check['branch_code'].isin(valid_branch_codes)
final_dirty_check['branch_code_matches_prefix'] = final_dirty_check['branch_code'] == final_dirty_check['expected_branch_code']

final_dirty_check['date_as_string'] = final_dirty_check['date'].astype(str)
final_dirty_check['date_matches_standard_format'] = final_dirty_check['date_as_string'].str.match(r'^\d{4}-\d{2}-\d{2}$')
final_dirty_check['date_parsed'] = pd.to_datetime(final_dirty_check['date_as_string'], errors='coerce', format='mixed')
final_dirty_check['date_valid'] = final_dirty_check['date_parsed'].notna() & final_dirty_check['date_parsed'].dt.year.eq(2018)

final_dirty_check['time_parsed'] = pd.to_datetime(
    final_dirty_check['time'].astype(str),
    format='%H:%M:%S',
    errors='coerce'
).dt.time
final_dirty_check['expected_order_type'] = final_dirty_check['time_parsed'].apply(infer_order_type_from_time)
final_dirty_check['order_type_matches_time'] = final_dirty_check['order_type'] == final_dirty_check['expected_order_type']

final_dirty_check['parsed_order_items'] = final_dirty_check['order_items'].apply(parse_order_items)
final_dirty_check['order_items_structure_valid'] = final_dirty_check['parsed_order_items'].apply(validate_order_items_structure)
final_dirty_check['expected_order_price'] = final_dirty_check['parsed_order_items'].apply(calculate_clean_expected_order_price)
final_dirty_check['order_price_residual'] = (
    final_dirty_check['order_price'] - final_dirty_check['expected_order_price']
).round(4)
final_dirty_check['order_price_matches_items'] = final_dirty_check['order_price_residual'].abs() < 0.01

final_item_records = []
for _, row in final_dirty_check.loc[final_dirty_check['order_items_structure_valid']].iterrows():
    for item_name, quantity in row['parsed_order_items']:
        final_item_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'item_name': item_name,
            'quantity': quantity,
            'expected_order_type_for_item': item_to_expected_meal.get(item_name),
        })
final_item_records_df = pd.DataFrame(final_item_records)
final_menu_mismatch_count = int((
    final_item_records_df['order_type'] != final_item_records_df['expected_order_type_for_item']
).sum()) if len(final_item_records_df) else 0

final_dirty_check['customer_coord_key'] = list(
    zip(
        final_dirty_check['customer_lat'].round(7),
        final_dirty_check['customer_lon'].round(7),
    )
)
final_dirty_check['customer_coordinate_in_nodes'] = final_dirty_check['customer_coord_key'].isin(node_coordinate_keys)
final_dirty_check['customer_node'] = final_dirty_check['customer_coord_key'].map(coord_to_node)
final_dirty_check['expected_distance_to_customer_KM'] = final_dirty_check.apply(expected_distance_km, axis=1)
final_dirty_check['distance_difference'] = (
    final_dirty_check['distance_to_customer_KM'] - final_dirty_check['expected_distance_to_customer_KM']
).round(3)
final_dirty_check['distance_matches_graph'] = final_dirty_check['distance_difference'].abs() <= 0.001

protected_column_checks = []
for column in ['order_id', 'time', 'delivery_fee']:
    protected_column_checks.append({
        'column': column,
        'unchanged': original_dirty[column].equals(dirty_cleaning[column]),
    })
protected_column_checks_df = pd.DataFrame(protected_column_checks)

dirty_final_validation = pd.DataFrame([
    {
        'check': 'row count preserved',
        'passed': len(dirty_cleaning) == len(original_dirty),
        'failed_rows': 0 if len(dirty_cleaning) == len(original_dirty) else abs(len(dirty_cleaning) - len(original_dirty)),
    },
    {
        'check': 'column order preserved',
        'passed': list(dirty_cleaning.columns) == list(original_dirty.columns),
        'failed_rows': 0,
    },
    {
        'check': 'branch_code valid and prefix-consistent',
        'passed': bool((final_dirty_check['branch_code_valid'] & final_dirty_check['branch_code_matches_prefix']).all()),
        'failed_rows': int((~(final_dirty_check['branch_code_valid'] & final_dirty_check['branch_code_matches_prefix'])).sum()),
    },
    {
        'check': 'date parseable, 2018, standard format',
        'passed': bool((final_dirty_check['date_valid'] & final_dirty_check['date_matches_standard_format']).all()),
        'failed_rows': int((~(final_dirty_check['date_valid'] & final_dirty_check['date_matches_standard_format'])).sum()),
    },
    {
        'check': 'order_type matches time window',
        'passed': bool(final_dirty_check['order_type_matches_time'].all()),
        'failed_rows': int((~final_dirty_check['order_type_matches_time']).sum()),
    },
    {
        'check': 'order_items structure valid',
        'passed': bool(final_dirty_check['order_items_structure_valid'].all()),
        'failed_rows': int((~final_dirty_check['order_items_structure_valid']).sum()),
    },
    {
        'check': 'order_items menu-consistent',
        'passed': final_menu_mismatch_count == 0,
        'failed_rows': final_menu_mismatch_count,
    },
    {
        'check': 'order_price matches order_items',
        'passed': bool(final_dirty_check['order_price_matches_items'].all()),
        'failed_rows': int((~final_dirty_check['order_price_matches_items']).sum()),
    },
    {
        'check': 'customer coordinates found in nodes.csv',
        'passed': bool(final_dirty_check['customer_coordinate_in_nodes'].all()),
        'failed_rows': int((~final_dirty_check['customer_coordinate_in_nodes']).sum()),
    },
    {
        'check': 'distance matches graph shortest path',
        'passed': bool(final_dirty_check['distance_matches_graph'].all()),
        'failed_rows': int((~final_dirty_check['distance_matches_graph']).sum()),
    },
    {
        'check': 'protected columns unchanged',
        'passed': bool(protected_column_checks_df['unchanged'].all()),
        'failed_rows': int((~protected_column_checks_df['unchanged']).sum()),
    },
    {
        'check': 'tracker has unique order_id index',
        'passed': not dirty_issue_flags.index.duplicated().any(),
        'failed_rows': int(dirty_issue_flags.index.duplicated().sum()),
    },
])

display(dirty_final_validation)

display(protected_column_checks_df)

tracker_final_summary = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_final_summary)

failed_final_checks = dirty_final_validation.loc[~dirty_final_validation['passed']]
display(failed_final_checks)

print('all final dirty checks passed:', failed_final_checks.empty)
print('total flagged rows:', int(dirty_issue_flags['issue_column'].notna().sum()))
print('total unflagged rows:', int(dirty_issue_flags['issue_column'].isna().sum()))

,check,passed,failed_rows
0,row count preserved,True,0
1,column order preserved,True,0
2,branch_code valid and prefix-consistent,True,0
3,"date parseable, 2018, standard format",False,20
4,order_type matches time window,True,0
5,order_items structure valid,True,0
6,order_items menu-consistent,True,0
7,order_price matches order_items,True,0
8,customer coordinates found in nodes.csv,True,0
9,distance matches graph shortest path,True,0


,column,unchanged
0,order_id,True
1,time,True
2,delivery_fee,True


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_coordinates,lat_lon_swapped,4
3,customer_lat,latitude_sign_error,37
4,date,non_standard_format,17
5,distance_to_customer_KM,graph_distance_mismatch,37
6,order_items,wrong_menu_item,37
7,order_price,price_items_mismatch,37
8,order_type,order_type_time_mismatch,37
9,unflagged,unflagged,257


,check,passed,failed_rows
3,"date parseable, 2018, standard format",False,20


all final dirty checks passed: False
total flagged rows: 243
total unflagged rows: 257


### Cell 17 讨论：final validation 怎么读？

**Q1：为什么 final validation 不再修新问题？**  
A：final validation 是收口检查，用来确认前面的修复没有破坏结构，也没有留下未解释的规则错误。如果这里发现失败，需要回到对应步骤修，而不是在 final cell 里偷偷改。

**Q2：为什么 protected columns 要单独检查？**  
A：guide 明确说 `order_id`、`time`、`delivery_fee` 在 dirty data 中 error-free，所以它们应该保持原样。

**Q3：为什么 tracker summary 很重要？**  
A：它展示每类 anomaly 修了多少行，也帮助我们确认一行只被一种 anomaly 解释。

**Q4：下一步是什么？**  
A：如果 final validation 全部通过，就可以导出 `Group024_dirty_data_solution.csv`。然后进入 missing_data imputation。

## Cell 17.1：诊断 final validation 中失败的 date records

Final validation 显示 date 还有失败记录。我们不在 final validation 里偷偷修，而是回到 date 逻辑诊断具体原因。

这里重点看：这些 date 是否是“看起来像 `YYYY-MM-DD`，但实际月份/日期位置错了”的情况。

In [40]:
failed_date_records = final_dirty_check.loc[
    ~(final_dirty_check['date_valid'] & final_dirty_check['date_matches_standard_format'])
].copy()

candidate_date_formats = [
    '%Y-%m-%d',
    '%d-%m-%Y',
    '%Y/%m/%d',
    '%d/%m/%Y',
    '%Y-%d-%m',
    '%Y/%d/%m',
]

def parse_with_known_formats(value):
    text = str(value)
    for date_format in candidate_date_formats:
        parsed = pd.to_datetime(text, format=date_format, errors='coerce')
        if pd.notna(parsed):
            return parsed.strftime('%Y-%m-%d'), date_format
    return pd.NA, pd.NA

failed_date_records[['repair_candidate_date', 'repair_candidate_format']] = failed_date_records['date'].apply(
    lambda value: pd.Series(parse_with_known_formats(value))
)
failed_date_records['repair_candidate_available'] = failed_date_records['repair_candidate_date'].notna()
failed_date_records['date_would_change'] = (
    failed_date_records['date'].astype(str) != failed_date_records['repair_candidate_date'].astype(str)
)
failed_date_records['already_flagged'] = failed_date_records['order_id'].isin(
    dirty_issue_flags.dropna(subset=['issue_column']).index
)

failed_date_view_columns = [
    'order_id',
    'date',
    'date_parsed',
    'date_matches_standard_format',
    'date_valid',
    'repair_candidate_date',
    'repair_candidate_format',
    'date_would_change',
    'already_flagged',
]

display(failed_date_records[failed_date_view_columns])

failed_date_summary = pd.DataFrame([
    {
        'check': 'failed date records',
        'value': len(failed_date_records),
    },
    {
        'check': 'repair candidate available',
        'value': int(failed_date_records['repair_candidate_available'].sum()),
    },
    {
        'check': 'repair candidate would change value',
        'value': int(failed_date_records['date_would_change'].sum()),
    },
    {
        'check': 'failed date records already flagged',
        'value': int(failed_date_records['already_flagged'].sum()),
    },
])

display(failed_date_summary)

print('failed date records:', len(failed_date_records))

,order_id,date,date_parsed,date_matches_standard_format,date_valid,repair_candidate_date,repair_candidate_format,date_would_change,already_flagged
10,ORDC01147,2018-26-08,NaT,True,False,2018-08-26,%Y-%d-%m,True,False
16,ORDK04564,2018-19-09,NaT,True,False,2018-09-19,%Y-%d-%m,True,False
37,ORDJ05383,2018-18-08,NaT,True,False,2018-08-18,%Y-%d-%m,True,False
51,ORDB00774,2018-16-06,NaT,True,False,2018-06-16,%Y-%d-%m,True,False
81,ORDB06811,2018-28-11,NaT,True,False,2018-11-28,%Y-%d-%m,True,False
96,ORDZ10150,2018-18-05,NaT,True,False,2018-05-18,%Y-%d-%m,True,False
114,ORDZ08183,2018-13-11,NaT,True,False,2018-11-13,%Y-%d-%m,True,False
139,ORDJ05844,2018-15-04,NaT,True,False,2018-04-15,%Y-%d-%m,True,False
161,ORDY08360,2018-28-11,NaT,True,False,2018-11-28,%Y-%d-%m,True,False
168,ORDI10091,2018-31-08,NaT,True,False,2018-08-31,%Y-%d-%m,True,False


,check,value
0,failed date records,20
1,repair candidate available,20
2,repair candidate would change value,20
3,failed date records already flagged,0


failed date records: 20


### Cell 17.1 讨论：为什么 final validation 能发现前面漏掉的 date 问题？

**Q1：这些 date 为什么前面可能没被修？**  
A：因为有些日期表面上符合 `YYYY-MM-DD` 的字符串形态，但实际可能是 `YYYY-DD-MM`。例如月份位置出现 `22`，格式 regex 会通过，但日期解析会失败。

**Q2：这说明早期 date 检查哪里不够？**  
A：早期只把“非标准格式”作为主要修复对象，漏掉了“标准外观但语义无效”的日期。

**Q3：下一步应该怎么修？**  
A：对这些 failed date records 使用明确的候选格式列表解析。如果能唯一解析成 2018 年合法日期，并且该行没有被其他 anomaly flag 占用，就把 date 修成候选标准日期并更新 tracker。

**下一步：**  
修复这些 final validation 发现的 date records，然后重新跑 final validation。

## Cell 17.2：修复 YYYY-DD-MM date records 并重新验证

Final validation 发现 20 行 date 是 `YYYY-DD-MM`，例如 `2018-26-08`。

这些 records 没有已有 flag，并且都可以用 `%Y-%d-%m` 唯一解析成合法 2018 日期，所以现在修复它们。

In [41]:
date_second_pass_candidates = failed_date_records.loc[
    failed_date_records['repair_candidate_available'] &
    failed_date_records['date_would_change'] &
    (~failed_date_records['already_flagged'])
].copy()

date_second_pass_audit = date_second_pass_candidates[
    [
        'order_id',
        'date',
        'repair_candidate_date',
        'repair_candidate_format',
    ]
].copy()

display(date_second_pass_audit)

for _, row in date_second_pass_audit.iterrows():
    dirty_cleaning.loc[
        dirty_cleaning['order_id'] == row['order_id'],
        'date'
    ] = row['repair_candidate_date']
    dirty_issue_flags.loc[row['order_id'], 'issue_column'] = 'date'
    dirty_issue_flags.loc[row['order_id'], 'issue_type'] = 'yyyy_dd_mm_format'
    dirty_issue_flags.loc[row['order_id'], 'evidence'] = (
        f"observed={row['date']}; fixed={row['repair_candidate_date']}; "
        f"format={row['repair_candidate_format']}"
    )

# Re-run final date validation only.
post_second_date_check = dirty_cleaning.copy()
post_second_date_check['date_as_string'] = post_second_date_check['date'].astype(str)
post_second_date_check['date_matches_standard_format'] = post_second_date_check['date_as_string'].str.match(r'^\d{4}-\d{2}-\d{2}$')
post_second_date_check['date_parsed'] = pd.to_datetime(
    post_second_date_check['date_as_string'],
    errors='coerce',
    format='mixed',
)
post_second_date_check['date_valid'] = (
    post_second_date_check['date_parsed'].notna() &
    post_second_date_check['date_parsed'].dt.year.eq(2018)
)

post_second_date_validation = pd.DataFrame([
    {
        'check': 'fixed second-pass date records',
        'value': len(date_second_pass_audit),
    },
    {
        'check': 'remaining invalid date rows',
        'value': int((~(post_second_date_check['date_valid'] & post_second_date_check['date_matches_standard_format'])).sum()),
    },
    {
        'check': 'row count preserved',
        'value': len(dirty_cleaning) == len(data['dirty']),
    },
])

display(post_second_date_validation)

display(
    post_second_date_check.loc[
        ~(post_second_date_check['date_valid'] & post_second_date_check['date_matches_standard_format']),
        ['order_id', 'date', 'date_parsed', 'date_matches_standard_format', 'date_valid']
    ]
)

tracker_summary_after_second_date = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_summary_after_second_date)

print('total flagged rows after second date fix:', int(dirty_issue_flags['issue_column'].notna().sum()))

,order_id,date,repair_candidate_date,repair_candidate_format
10,ORDC01147,2018-26-08,2018-08-26,%Y-%d-%m
16,ORDK04564,2018-19-09,2018-09-19,%Y-%d-%m
37,ORDJ05383,2018-18-08,2018-08-18,%Y-%d-%m
51,ORDB00774,2018-16-06,2018-06-16,%Y-%d-%m
81,ORDB06811,2018-28-11,2018-11-28,%Y-%d-%m
96,ORDZ10150,2018-18-05,2018-05-18,%Y-%d-%m
114,ORDZ08183,2018-13-11,2018-11-13,%Y-%d-%m
139,ORDJ05844,2018-15-04,2018-04-15,%Y-%d-%m
161,ORDY08360,2018-28-11,2018-11-28,%Y-%d-%m
168,ORDI10091,2018-31-08,2018-08-31,%Y-%d-%m


,check,value
0,fixed second-pass date records,20
1,remaining invalid date rows,0
2,row count preserved,True


,order_id,date,date_parsed,date_matches_standard_format,date_valid


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_coordinates,lat_lon_swapped,4
3,customer_lat,latitude_sign_error,37
4,date,non_standard_format,17
5,date,yyyy_dd_mm_format,20
6,distance_to_customer_KM,graph_distance_mismatch,37
7,order_items,wrong_menu_item,37
8,order_price,price_items_mismatch,37
9,order_type,order_type_time_mismatch,37


total flagged rows after second date fix: 263


### Cell 17.2 讨论：为什么 final validation 后还可以回头修？

**Q1：这是不是说明前面做错了？**  
A：不是。final validation 的作用就是发现前面漏掉的 edge case。这里发现的是 date 的第二种格式错误：`YYYY-DD-MM`。

**Q2：为什么这 20 行可以修？**  
A：它们没有已有 flag，并且都能用 `%Y-%d-%m` 唯一解析成合法 2018 日期。修复依据清楚。

**Q3：下一步做什么？**  
A：重新运行完整 dirty final validation，确认所有检查都通过。

**下一步：**  
重新跑完整 final validation。

## Cell 17.3：重新运行完整 dirty final validation

现在重新运行完整 final validation。

Final validation 是收口检查，用来确认前面的修复没有破坏数据结构，也没有留下未解释的规则错误。它本身不应该偷偷修数据；如果这里发现失败，我们应该回到对应的检测/修复步骤，明确说明为什么漏掉、如何修复、修复后再重新运行 final validation。

In [42]:
final_dirty_check = dirty_cleaning.copy()
original_dirty = data['dirty'].copy()

final_dirty_check['order_prefix'] = final_dirty_check['order_id'].str.extract(r'^ORD([A-Z])')
final_dirty_check['expected_branch_code'] = final_dirty_check['order_prefix'].map(prefix_to_expected_branch)
final_dirty_check['branch_code_valid'] = final_dirty_check['branch_code'].isin(valid_branch_codes)
final_dirty_check['branch_code_matches_prefix'] = final_dirty_check['branch_code'] == final_dirty_check['expected_branch_code']

final_dirty_check['date_as_string'] = final_dirty_check['date'].astype(str)
final_dirty_check['date_matches_standard_format'] = final_dirty_check['date_as_string'].str.match(r'^\d{4}-\d{2}-\d{2}$')
final_dirty_check['date_parsed'] = pd.to_datetime(final_dirty_check['date_as_string'], errors='coerce', format='mixed')
final_dirty_check['date_valid'] = final_dirty_check['date_parsed'].notna() & final_dirty_check['date_parsed'].dt.year.eq(2018)

final_dirty_check['time_parsed'] = pd.to_datetime(
    final_dirty_check['time'].astype(str),
    format='%H:%M:%S',
    errors='coerce'
).dt.time
final_dirty_check['expected_order_type'] = final_dirty_check['time_parsed'].apply(infer_order_type_from_time)
final_dirty_check['order_type_matches_time'] = final_dirty_check['order_type'] == final_dirty_check['expected_order_type']

final_dirty_check['parsed_order_items'] = final_dirty_check['order_items'].apply(parse_order_items)
final_dirty_check['order_items_structure_valid'] = final_dirty_check['parsed_order_items'].apply(validate_order_items_structure)
final_dirty_check['expected_order_price'] = final_dirty_check['parsed_order_items'].apply(calculate_clean_expected_order_price)
final_dirty_check['order_price_residual'] = (
    final_dirty_check['order_price'] - final_dirty_check['expected_order_price']
).round(4)
final_dirty_check['order_price_matches_items'] = final_dirty_check['order_price_residual'].abs() < 0.01

final_item_records = []
for _, row in final_dirty_check.loc[final_dirty_check['order_items_structure_valid']].iterrows():
    for item_name, quantity in row['parsed_order_items']:
        final_item_records.append({
            'order_id': row['order_id'],
            'order_type': row['order_type'],
            'item_name': item_name,
            'quantity': quantity,
            'expected_order_type_for_item': item_to_expected_meal.get(item_name),
        })
final_item_records_df = pd.DataFrame(final_item_records)
final_menu_mismatch_count = int((
    final_item_records_df['order_type'] != final_item_records_df['expected_order_type_for_item']
).sum()) if len(final_item_records_df) else 0

final_dirty_check['customer_coord_key'] = list(
    zip(
        final_dirty_check['customer_lat'].round(7),
        final_dirty_check['customer_lon'].round(7),
    )
)
final_dirty_check['customer_coordinate_in_nodes'] = final_dirty_check['customer_coord_key'].isin(node_coordinate_keys)
final_dirty_check['customer_node'] = final_dirty_check['customer_coord_key'].map(coord_to_node)
final_dirty_check['expected_distance_to_customer_KM'] = final_dirty_check.apply(expected_distance_km, axis=1)
final_dirty_check['distance_difference'] = (
    final_dirty_check['distance_to_customer_KM'] - final_dirty_check['expected_distance_to_customer_KM']
).round(3)
final_dirty_check['distance_matches_graph'] = final_dirty_check['distance_difference'].abs() <= 0.001

protected_column_checks_df = pd.DataFrame([
    {
        'column': column,
        'unchanged': original_dirty[column].equals(dirty_cleaning[column]),
    }
    for column in ['order_id', 'time', 'delivery_fee']
])

dirty_final_validation_rerun = pd.DataFrame([
    {
        'check': 'row count preserved',
        'passed': len(dirty_cleaning) == len(original_dirty),
        'failed_rows': 0 if len(dirty_cleaning) == len(original_dirty) else abs(len(dirty_cleaning) - len(original_dirty)),
    },
    {
        'check': 'column order preserved',
        'passed': list(dirty_cleaning.columns) == list(original_dirty.columns),
        'failed_rows': 0,
    },
    {
        'check': 'branch_code valid and prefix-consistent',
        'passed': bool((final_dirty_check['branch_code_valid'] & final_dirty_check['branch_code_matches_prefix']).all()),
        'failed_rows': int((~(final_dirty_check['branch_code_valid'] & final_dirty_check['branch_code_matches_prefix'])).sum()),
    },
    {
        'check': 'date parseable, 2018, standard format',
        'passed': bool((final_dirty_check['date_valid'] & final_dirty_check['date_matches_standard_format']).all()),
        'failed_rows': int((~(final_dirty_check['date_valid'] & final_dirty_check['date_matches_standard_format'])).sum()),
    },
    {
        'check': 'order_type matches time window',
        'passed': bool(final_dirty_check['order_type_matches_time'].all()),
        'failed_rows': int((~final_dirty_check['order_type_matches_time']).sum()),
    },
    {
        'check': 'order_items structure valid',
        'passed': bool(final_dirty_check['order_items_structure_valid'].all()),
        'failed_rows': int((~final_dirty_check['order_items_structure_valid']).sum()),
    },
    {
        'check': 'order_items menu-consistent',
        'passed': final_menu_mismatch_count == 0,
        'failed_rows': final_menu_mismatch_count,
    },
    {
        'check': 'order_price matches order_items',
        'passed': bool(final_dirty_check['order_price_matches_items'].all()),
        'failed_rows': int((~final_dirty_check['order_price_matches_items']).sum()),
    },
    {
        'check': 'customer coordinates found in nodes.csv',
        'passed': bool(final_dirty_check['customer_coordinate_in_nodes'].all()),
        'failed_rows': int((~final_dirty_check['customer_coordinate_in_nodes']).sum()),
    },
    {
        'check': 'distance matches graph shortest path',
        'passed': bool(final_dirty_check['distance_matches_graph'].all()),
        'failed_rows': int((~final_dirty_check['distance_matches_graph']).sum()),
    },
    {
        'check': 'protected columns unchanged',
        'passed': bool(protected_column_checks_df['unchanged'].all()),
        'failed_rows': int((~protected_column_checks_df['unchanged']).sum()),
    },
    {
        'check': 'tracker has unique order_id index',
        'passed': not dirty_issue_flags.index.duplicated().any(),
        'failed_rows': int(dirty_issue_flags.index.duplicated().sum()),
    },
])

display(dirty_final_validation_rerun)

display(protected_column_checks_df)

tracker_final_summary_rerun = (
    dirty_issue_flags
    .fillna({'issue_column': 'unflagged', 'issue_type': 'unflagged'})
    .groupby(['issue_column', 'issue_type'])
    .size()
    .reset_index(name='row_count')
)

display(tracker_final_summary_rerun)

failed_final_checks_rerun = dirty_final_validation_rerun.loc[~dirty_final_validation_rerun['passed']]
display(failed_final_checks_rerun)

print('all final dirty checks passed:', failed_final_checks_rerun.empty)
print('total flagged rows:', int(dirty_issue_flags['issue_column'].notna().sum()))
print('total unflagged rows:', int(dirty_issue_flags['issue_column'].isna().sum()))

,check,passed,failed_rows
0,row count preserved,True,0
1,column order preserved,True,0
2,branch_code valid and prefix-consistent,True,0
3,"date parseable, 2018, standard format",True,0
4,order_type matches time window,True,0
5,order_items structure valid,True,0
6,order_items menu-consistent,True,0
7,order_price matches order_items,True,0
8,customer coordinates found in nodes.csv,True,0
9,distance matches graph shortest path,True,0


,column,unchanged
0,order_id,True
1,time,True
2,delivery_fee,True


,issue_column,issue_type,row_count
0,branch_code,case_issue_only,9
1,branch_code,prefix_mismatch,28
2,customer_coordinates,lat_lon_swapped,4
3,customer_lat,latitude_sign_error,37
4,date,non_standard_format,17
5,date,yyyy_dd_mm_format,20
6,distance_to_customer_KM,graph_distance_mismatch,37
7,order_items,wrong_menu_item,37
8,order_price,price_items_mismatch,37
9,order_type,order_type_time_mismatch,37


,check,passed,failed_rows


all final dirty checks passed: True
total flagged rows: 263
total unflagged rows: 237


### Cell 17.3 讨论：dirty_data 是否可以导出？

**Q1：如果 `all final dirty checks passed` 是 True，说明什么？**  
A：说明 dirty_data 的主要业务规则、结构要求和 protected column 要求都通过，可以导出 solution CSV。

**Q2：tracker summary 有什么用？**  
A：它说明我们修复了哪些类型的 anomaly，各修了多少行，也能辅助 presentation 解释方法。

**Q3：下一步是什么？**  
A：导出 `Group024_dirty_data_solution.csv`，然后进入 missing_data imputation。

## Cell 18：进入 missing_data，先做缺失概览

dirty_data 已经完成 final validation。现在进入 Task 1 的第二部分：impute missing values in `Group024_missing_data.csv`。

Assignment guide 说明 missing_data 只有 coverage/missing anomalies，没有其他 data anomalies。因此这里的目标不是重新找 dirty errors，而是根据字段依赖关系填补缺失值。

In [43]:
missing_working = data['missing'].copy()

missing_column_summary = pd.DataFrame({
    'column': missing_working.columns,
    'dtype': [str(missing_working[column].dtype) for column in missing_working.columns],
    'missing_count': [int(missing_working[column].isna().sum()) for column in missing_working.columns],
    'missing_rate': [missing_working[column].isna().mean() for column in missing_working.columns],
    'unique_count': [int(missing_working[column].nunique(dropna=True)) for column in missing_working.columns],
    'example_values': [missing_working[column].dropna().head(3).tolist() for column in missing_working.columns],
})

display(missing_column_summary)

missing_rows_summary = (
    missing_working
    .isna()
    .sum(axis=1)
    .value_counts()
    .rename_axis('missing_cells_in_row')
    .reset_index(name='row_count')
    .sort_values('missing_cells_in_row')
)

display(missing_rows_summary)

display(
    missing_working.loc[
        missing_working.isna().any(axis=1),
        ['order_id', 'branch_code', 'distance_to_customer_KM', 'delivery_fee']
    ].head(20)
)

print('missing_data shape:', missing_working.shape)
print('total missing cells:', int(missing_working.isna().sum().sum()))

,column,dtype,missing_count,missing_rate,unique_count,example_values
0,order_id,str,0,0.0000,500,"[ORDX00188, ORDZ07051, ORDK05872]"
1,date,str,0,0.0000,265,"[2018-07-12, 2018-01-07, 2018-01-27]"
2,time,str,0,0.0000,72,"[14:25:21, 08:10:08, 08:50:42]"
3,order_type,str,0,0.0000,3,"[Lunch, Breakfast, Breakfast]"
4,branch_code,str,100,0.2000,3,"[BK, BK, NS]"
5,order_items,str,0,0.0000,499,"[[('Steak', 2), ('Salad', 5)], [('Coffee', 2), ('Cereal', 7)], [('Eggs', 1), ('Coffee', 8), ('Cereal', 4)]]"
6,order_price,float64,0,0.0000,458,"[176.0, 162.0, 166.0]"
7,customer_lat,float64,0,0.0000,491,"[-37.8275615, -37.8133334, -37.8197193]"
8,customer_lon,float64,0,0.0000,491,"[144.9828777, 144.9375814, 144.941013]"
9,customerHasloyalty?,int64,0,0.0000,2,"[0, 1, 0]"


,missing_cells_in_row,row_count
0,0,350
1,1,100
2,2,50


,order_id,branch_code,distance_to_customer_KM,delivery_fee
0,ORDX00188,NaN,8.0530,13.8220
1,ORDZ07051,NaN,NaN,8.6529
3,ORDC06592,NaN,7.5030,14.8951
7,ORDC10700,NS,9.6580,NaN
9,ORDJ05186,NaN,NaN,13.5815
14,ORDA07746,NaN,NaN,15.8464
15,ORDK01450,NaN,8.8530,16.9256
23,ORDY05307,NaN,9.6630,13.6873
27,ORDY04867,NaN,NaN,11.5806
33,ORDJ08835,TP,8.5170,NaN


missing_data shape: (500, 12)
total missing cells: 200


### Cell 18 讨论：missing_data 的填补顺序是什么？

**Q1：missing_data 和 dirty_data 的思路有什么不同？**  
A：dirty_data 是 detect and fix errors；missing_data 是 impute missing values。Guide 说 missing_data 没有其他 data anomalies，所以不要重新做 dirty cleaning。

**Q2：为什么还是要先 profile？**  
A：因为填补方法取决于缺失在哪些列。不同字段不能用同一种方法机械填补。

**Q3：根据字段依赖，合理顺序是什么？**  
A：先填 `branch_code`，因为它决定 branch node；再填 `distance_to_customer_KM`，因为 distance 依赖 branch 和 customer node；最后填 `delivery_fee`，因为 delivery fee 依赖 weekend/time/distance/branch，并且受 loyalty discount 影响。

**Q4：下一步做什么？**  
A：先处理 `branch_code` 缺失。我们会用已经验证过的 `order_id` prefix 到 branch code 的映射来填补。

**下一步：**  
填补 missing_data 中的 `branch_code`。

## Cell 19：填补 missing_data 的 branch_code

先填补 `branch_code`。

理由：`branch_code` 可以由 `order_id` prefix 稳定推导；而且它是后面计算 `distance_to_customer_KM` 的前置条件。

In [44]:
missing_working['order_prefix'] = missing_working['order_id'].str.extract(r'^ORD([A-Z])')

missing_branch_before = int(missing_working['branch_code'].isna().sum())

missing_branch_audit = missing_working.loc[
    missing_working['branch_code'].isna(),
    ['order_id', 'order_prefix', 'branch_code']
].copy()
missing_branch_audit['imputed_branch_code'] = missing_branch_audit['order_prefix'].map(prefix_to_expected_branch)

display(missing_branch_audit.head(30))

missing_working.loc[
    missing_working['branch_code'].isna(),
    'branch_code'
] = missing_working.loc[
    missing_working['branch_code'].isna(),
    'order_prefix'
].map(prefix_to_expected_branch)

missing_branch_after = int(missing_working['branch_code'].isna().sum())
missing_branch_invalid_after = int((~missing_working['branch_code'].isin(valid_branch_codes)).sum())
missing_branch_prefix_mismatch_after = int((
    missing_working['branch_code'] != missing_working['order_prefix'].map(prefix_to_expected_branch)
).sum())

missing_branch_validation = pd.DataFrame([
    {
        'check': 'branch_code missing before',
        'value': missing_branch_before,
    },
    {
        'check': 'branch_code missing after',
        'value': missing_branch_after,
    },
    {
        'check': 'invalid branch_code after',
        'value': missing_branch_invalid_after,
    },
    {
        'check': 'prefix mismatch after',
        'value': missing_branch_prefix_mismatch_after,
    },
    {
        'check': 'row count preserved',
        'value': len(missing_working) == len(data['missing']),
    },
])

display(missing_branch_validation)

print('branch_code imputed rows:', len(missing_branch_audit))

,order_id,order_prefix,branch_code,imputed_branch_code
0,ORDX00188,X,NaN,BK
1,ORDZ07051,Z,NaN,NS
3,ORDC06592,C,NaN,NS
9,ORDJ05186,J,NaN,TP
14,ORDA07746,A,NaN,BK
15,ORDK01450,K,NaN,BK
23,ORDY05307,Y,NaN,TP
27,ORDY04867,Y,NaN,TP
36,ORDX02773,X,NaN,BK
41,ORDB09188,B,NaN,TP


,check,value
0,branch_code missing before,100
1,branch_code missing after,0
2,invalid branch_code after,0
3,prefix mismatch after,0
4,row count preserved,True


branch_code imputed rows: 100


### Cell 19 讨论：为什么 branch_code 可以这样填？

**Q1：为什么不用众数填补 branch_code？**  
A：因为 branch_code 不是普通类别变量。它和 `order_id` prefix 有稳定映射，用业务规则填补比众数更准确。

**Q2：为什么 branch_code 要先填？**  
A：因为 distance 需要 branch node 作为起点。如果 branch_code 缺失，后面的 shortest path distance 无法计算。

**Q3：填补后 validation 看什么？**  
A：看缺失是否清零、branch code 是否都是合法值、是否都和 prefix mapping 一致、row count 是否保持。

**下一步：**  
用 branch_code、customer coordinates 和 graph shortest path 填补 `distance_to_customer_KM`。

## Cell 20：填补 missing_data 的 distance_to_customer_KM

现在填补 `distance_to_customer_KM`。

`missing_data` 没有其他 data anomalies，所以 customer coordinates 可以直接映射到 `nodes.csv`。distance 的填补值来自 branch node 到 customer node 的 Dijkstra shortest path distance。

In [45]:
missing_working['customer_coord_key'] = list(
    zip(
        missing_working['customer_lat'].round(7),
        missing_working['customer_lon'].round(7),
    )
)
missing_working['customer_node'] = missing_working['customer_coord_key'].map(coord_to_node)

missing_customer_node_missing = int(missing_working['customer_node'].isna().sum())

missing_working['expected_distance_to_customer_KM'] = missing_working.apply(expected_distance_km, axis=1)

missing_distance_before = int(missing_working['distance_to_customer_KM'].isna().sum())

missing_distance_audit = missing_working.loc[
    missing_working['distance_to_customer_KM'].isna(),
    [
        'order_id',
        'branch_code',
        'customer_node',
        'distance_to_customer_KM',
        'expected_distance_to_customer_KM',
    ]
].copy()

display(missing_distance_audit.head(30))

missing_working.loc[
    missing_working['distance_to_customer_KM'].isna(),
    'distance_to_customer_KM'
] = missing_working.loc[
    missing_working['distance_to_customer_KM'].isna(),
    'expected_distance_to_customer_KM'
]

missing_distance_after = int(missing_working['distance_to_customer_KM'].isna().sum())
missing_working['distance_difference'] = (
    missing_working['distance_to_customer_KM'] - missing_working['expected_distance_to_customer_KM']
).round(3)
missing_distance_mismatch_after = int((missing_working['distance_difference'].abs() > 0.001).sum())

missing_distance_validation = pd.DataFrame([
    {
        'check': 'customer_node missing',
        'value': missing_customer_node_missing,
    },
    {
        'check': 'distance missing before',
        'value': missing_distance_before,
    },
    {
        'check': 'distance missing after',
        'value': missing_distance_after,
    },
    {
        'check': 'distance mismatch after',
        'value': missing_distance_mismatch_after,
    },
    {
        'check': 'row count preserved',
        'value': len(missing_working) == len(data['missing']),
    },
])

display(missing_distance_validation)

print('distance imputed rows:', len(missing_distance_audit))

,order_id,branch_code,customer_node,distance_to_customer_KM,expected_distance_to_customer_KM
1,ORDZ07051,NS,634783955,NaN,10.0390
9,ORDJ05186,TP,420022666,NaN,10.6120
14,ORDA07746,BK,844482569,NaN,9.2140
27,ORDY04867,TP,579478244,NaN,8.6650
41,ORDB09188,TP,4716795515,NaN,8.5940
48,ORDA06225,BK,589708344,NaN,8.5910
59,ORDZ01133,NS,343337614,NaN,6.3850
64,ORDY10132,TP,60096323,NaN,9.0160
67,ORDC00282,NS,2451176465,NaN,5.7140
72,ORDY06549,TP,440062374,NaN,10.1740


,check,value
0,customer_node missing,0
1,distance missing before,50
2,distance missing after,0
3,distance mismatch after,0
4,row count preserved,True


distance imputed rows: 50


### Cell 20 讨论：为什么 distance 可以用 Dijkstra 填补？

**Q1：distance 的定义是什么？**  
A：guide 说明它是 branch node 到 customer node 的 shortest path distance，权重来自 `edges.csv` 的 `distance(m)`。

**Q2：为什么现在可以填 distance？**  
A：因为 `branch_code` 已经完整，customer coordinates 也来自 nodes.csv，所以 branch node 和 customer node 都可确定。

**Q3：填补后 validation 看什么？**  
A：看 distance 缺失是否清零，以及所有 rows 的 distance 是否都等于 graph shortest path。

**下一步：**  
填补 `delivery_fee`。这需要考虑 branch-specific linear model、weekend/time/distance，以及 loyalty 50% discount。

## Cell 21：训练 delivery_fee 模型并填补 missing delivery_fee

最后填补 `delivery_fee`。

Guide 说明 delivery fee 按 branch 分别线性依赖：

- weekend / weekday
- time of day
- distance

并且 loyalty customer 有 50% discount。因此建模时先把 loyalty fee 还原成 undiscounted fee：

- loyalty = 1: adjusted fee = delivery_fee * 2
- loyalty = 0: adjusted fee = delivery_fee

预测后再按 loyalty 折扣还原最终 delivery_fee。

In [46]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

fee_feature_columns = ['is_weekend', 'time_code', 'distance_to_customer_KM']
time_code_map = {
    'Breakfast': 0,
    'Lunch': 1,
    'Dinner': 2,
}

def add_delivery_fee_features(frame):
    result = frame.copy()
    result['date_parsed'] = pd.to_datetime(result['date'], errors='coerce', format='mixed')
    result['is_weekend'] = result['date_parsed'].dt.dayofweek.isin([5, 6]).astype(int)
    result['time_code'] = result['order_type'].map(time_code_map).astype(int)
    return result

fee_training_dirty = dirty_cleaning.copy()
fee_training_dirty['source_dataset'] = 'dirty_cleaned'

fee_training_missing = missing_working.loc[
    missing_working['delivery_fee'].notna()
].copy()
fee_training_missing['source_dataset'] = 'missing_nonmissing_fee'

fee_training_data = pd.concat(
    [fee_training_dirty, fee_training_missing],
    ignore_index=True,
)
fee_training_data = add_delivery_fee_features(fee_training_data)
fee_training_data['adjusted_delivery_fee'] = np.where(
    fee_training_data['customerHasloyalty?'] == 1,
    fee_training_data['delivery_fee'] * 2,
    fee_training_data['delivery_fee'],
)

fee_models = {}
fee_model_reports = []

for branch_code, branch_rows in fee_training_data.groupby('branch_code'):
    model = LinearRegression()
    X = branch_rows[fee_feature_columns]
    y = branch_rows['adjusted_delivery_fee']
    model.fit(X, y)
    predictions = model.predict(X)
    fee_models[branch_code] = model
    fee_model_reports.append({
        'branch_code': branch_code,
        'training_rows': len(branch_rows),
        'r2_score': r2_score(y, predictions),
        'intercept': model.intercept_,
        'coef_is_weekend': model.coef_[0],
        'coef_time_code': model.coef_[1],
        'coef_distance': model.coef_[2],
        'mean_abs_residual': np.mean(np.abs(y - predictions)),
    })

fee_model_report_df = pd.DataFrame(fee_model_reports)

display(fee_model_report_df)

missing_working = add_delivery_fee_features(missing_working)

missing_fee_before = int(missing_working['delivery_fee'].isna().sum())

missing_fee_audit = missing_working.loc[
    missing_working['delivery_fee'].isna(),
    ['order_id', 'branch_code', 'date', 'order_type', 'customerHasloyalty?', 'distance_to_customer_KM', 'delivery_fee']
].copy()

predicted_fees = []
for _, row in missing_fee_audit.iterrows():
    model = fee_models[row['branch_code']]
    features = pd.DataFrame([{
        'is_weekend': row['is_weekend'] if 'is_weekend' in row else missing_working.loc[missing_working['order_id'] == row['order_id'], 'is_weekend'].iloc[0],
        'time_code': row['time_code'] if 'time_code' in row else missing_working.loc[missing_working['order_id'] == row['order_id'], 'time_code'].iloc[0],
        'distance_to_customer_KM': row['distance_to_customer_KM'],
    }])
    adjusted_prediction = float(model.predict(features)[0])
    final_prediction = adjusted_prediction / 2 if int(row['customerHasloyalty?']) == 1 else adjusted_prediction
    predicted_fees.append(round(final_prediction, 6))

missing_fee_audit['imputed_delivery_fee'] = predicted_fees

display(missing_fee_audit.head(30))

for _, row in missing_fee_audit.iterrows():
    missing_working.loc[
        missing_working['order_id'] == row['order_id'],
        'delivery_fee'
    ] = row['imputed_delivery_fee']

missing_fee_after = int(missing_working['delivery_fee'].isna().sum())

missing_fee_validation = pd.DataFrame([
    {
        'check': 'delivery_fee missing before',
        'value': missing_fee_before,
    },
    {
        'check': 'delivery_fee missing after',
        'value': missing_fee_after,
    },
    {
        'check': 'minimum imputed delivery_fee',
        'value': float(missing_fee_audit['imputed_delivery_fee'].min()) if len(missing_fee_audit) else np.nan,
    },
    {
        'check': 'maximum imputed delivery_fee',
        'value': float(missing_fee_audit['imputed_delivery_fee'].max()) if len(missing_fee_audit) else np.nan,
    },
    {
        'check': 'row count preserved',
        'value': len(missing_working) == len(data['missing']),
    },
])

display(missing_fee_validation)

print('delivery_fee imputed rows:', len(missing_fee_audit))

,branch_code,training_rows,r2_score,intercept,coef_is_weekend,coef_time_code,coef_distance,mean_abs_residual
0,BK,315,0.5559,4.3326,2.3134,1.2746,1.1327,1.3724
1,NS,267,0.3654,6.4556,2.2141,0.4662,0.8701,0.7167
2,TP,368,0.2190,4.3989,1.6742,0.5405,0.8874,1.1377


,order_id,branch_code,date,order_type,customerHasloyalty?,distance_to_customer_KM,delivery_fee,imputed_delivery_fee
7,ORDC10700,NS,2018-06-28,Lunch,0,9.6580,NaN,15.3254
33,ORDJ08835,TP,2018-06-14,Lunch,0,8.5170,NaN,12.4976
38,ORDJ07186,TP,2018-10-29,Dinner,0,11.4430,NaN,15.6347
42,ORDC03961,NS,2018-01-11,Breakfast,1,8.1470,NaN,6.7722
61,ORDJ09090,TP,2018-07-21,Lunch,0,8.1360,NaN,13.8337
62,ORDI02574,NS,2018-06-04,Lunch,0,8.0580,NaN,13.9332
66,ORDJ04126,TP,2018-03-20,Breakfast,0,8.5750,NaN,12.0085
83,ORDZ01109,NS,2018-08-15,Lunch,0,7.3370,NaN,13.3058
88,ORDX10933,BK,2018-12-11,Dinner,0,10.0450,NaN,18.2598
93,ORDB10808,TP,2018-08-16,Lunch,0,8.9620,NaN,12.8925


,check,value
0,delivery_fee missing before,50
1,delivery_fee missing after,0
2,minimum imputed delivery_fee,6.7722
3,maximum imputed delivery_fee,20.3002
4,row count preserved,True


delivery_fee imputed rows: 50


### Cell 21 讨论：delivery_fee 为什么要处理 loyalty？

**Q1：为什么不直接用 delivery_fee 建模型？**  
A：因为 loyalty customer 有 50% discount。同样的 weekend/time/distance，如果 loyalty 不处理，fee 会系统性变成一半，模型会混乱。

**Q2：为什么按 branch 分开训练？**  
A：guide 说明不同 branch 使用不同 fee 方法，所以每个 branch 应该有自己的 linear model。

**Q3：为什么使用 cleaned dirty 和 missing 中非缺失 fee 的 rows？**  
A：dirty 的 delivery_fee 是 protected/error-free，且前面已经修复了会影响 features 的字段；missing_data 除了缺失没有其他 anomalies，所以非缺失 fee rows 也是可用训练数据。

**Q4：validation 看什么？**  
A：看每个 branch 的 R2 是否接近 1，delivery_fee 缺失是否清零，imputed fee 是否为合理正值，row count 是否保持。

**下一步：**  
做 missing_data final validation，并导出 solution CSV。

## Cell 21.1：修正 delivery_fee 模型，使用 missing_data 内部非缺失 rows 训练

上一格把 cleaned dirty 和 missing non-missing rows 混在一起训练，R2 很低，不符合 guide 对 delivery fee model 的要求。

这里修正策略：只使用 `missing_data` 内部 delivery_fee 非缺失的 rows 训练模型。因为 guide 说 missing_data 没有其他 data anomalies，非缺失 fee rows 正好是同一文件内最合适的 reference。

In [47]:
missing_fee_model_data = add_delivery_fee_features(missing_working.copy())

original_missing_fee_mask = data['missing']['delivery_fee'].isna()
training_fee_mask = ~original_missing_fee_mask

missing_fee_model_data.loc[original_missing_fee_mask, 'delivery_fee'] = np.nan
missing_fee_model_data['adjusted_delivery_fee'] = np.where(
    missing_fee_model_data['customerHasloyalty?'] == 1,
    missing_fee_model_data['delivery_fee'] * 2,
    missing_fee_model_data['delivery_fee'],
)

missing_only_fee_models = {}
missing_only_fee_reports = []

for branch_code, branch_rows in missing_fee_model_data.loc[training_fee_mask].groupby('branch_code'):
    model = LinearRegression()
    X = branch_rows[fee_feature_columns]
    y = branch_rows['adjusted_delivery_fee']
    model.fit(X, y)
    predictions = model.predict(X)
    missing_only_fee_models[branch_code] = model
    missing_only_fee_reports.append({
        'branch_code': branch_code,
        'training_rows': len(branch_rows),
        'r2_score': r2_score(y, predictions),
        'intercept': model.intercept_,
        'coef_is_weekend': model.coef_[0],
        'coef_time_code': model.coef_[1],
        'coef_distance': model.coef_[2],
        'mean_abs_residual': np.mean(np.abs(y - predictions)),
    })

missing_only_fee_report_df = pd.DataFrame(missing_only_fee_reports)

display(missing_only_fee_report_df)

missing_fee_second_audit = missing_fee_model_data.loc[
    original_missing_fee_mask,
    ['order_id', 'branch_code', 'date', 'order_type', 'customerHasloyalty?', 'distance_to_customer_KM', 'delivery_fee', 'is_weekend', 'time_code']
].copy()

second_predictions = []
for _, row in missing_fee_second_audit.iterrows():
    model = missing_only_fee_models[row['branch_code']]
    features = pd.DataFrame([{
        'is_weekend': row['is_weekend'],
        'time_code': row['time_code'],
        'distance_to_customer_KM': row['distance_to_customer_KM'],
    }])
    adjusted_prediction = float(model.predict(features)[0])
    final_prediction = adjusted_prediction / 2 if int(row['customerHasloyalty?']) == 1 else adjusted_prediction
    second_predictions.append(round(final_prediction, 6))

missing_fee_second_audit['imputed_delivery_fee'] = second_predictions

display(missing_fee_second_audit.head(30))

for _, row in missing_fee_second_audit.iterrows():
    missing_working.loc[
        missing_working['order_id'] == row['order_id'],
        'delivery_fee'
    ] = row['imputed_delivery_fee']

missing_fee_second_validation = pd.DataFrame([
    {
        'check': 'delivery_fee originally missing',
        'value': int(original_missing_fee_mask.sum()),
    },
    {
        'check': 'delivery_fee missing after missing-only model',
        'value': int(missing_working['delivery_fee'].isna().sum()),
    },
    {
        'check': 'minimum imputed delivery_fee',
        'value': float(missing_fee_second_audit['imputed_delivery_fee'].min()),
    },
    {
        'check': 'maximum imputed delivery_fee',
        'value': float(missing_fee_second_audit['imputed_delivery_fee'].max()),
    },
    {
        'check': 'minimum branch R2',
        'value': float(missing_only_fee_report_df['r2_score'].min()),
    },
    {
        'check': 'row count preserved',
        'value': len(missing_working) == len(data['missing']),
    },
])

display(missing_fee_second_validation)

print('delivery_fee imputed rows with missing-only model:', len(missing_fee_second_audit))

,branch_code,training_rows,r2_score,intercept,coef_is_weekend,coef_time_code,coef_distance,mean_abs_residual
0,BK,153,0.9946,4.4431,2.5258,0.9586,1.0571,0.2371
1,NS,128,0.9670,4.8406,1.9294,0.5377,1.0163,0.2613
2,TP,169,0.9520,4.0369,1.4818,0.7360,0.8506,0.2415


,order_id,branch_code,date,order_type,customerHasloyalty?,distance_to_customer_KM,delivery_fee,is_weekend,time_code,imputed_delivery_fee
7,ORDC10700,NS,2018-06-28,Lunch,0,9.6580,NaN,0,1,15.1941
33,ORDJ08835,TP,2018-06-14,Lunch,0,8.5170,NaN,0,1,12.0175
38,ORDJ07186,TP,2018-10-29,Dinner,0,11.4430,NaN,0,2,15.2423
42,ORDC03961,NS,2018-01-11,Breakfast,1,8.1470,NaN,0,0,6.5603
61,ORDJ09090,TP,2018-07-21,Lunch,0,8.1360,NaN,1,1,13.1752
62,ORDI02574,NS,2018-06-04,Lunch,0,8.0580,NaN,0,1,13.5680
66,ORDJ04126,TP,2018-03-20,Breakfast,0,8.5750,NaN,0,0,11.3308
83,ORDZ01109,NS,2018-08-15,Lunch,0,7.3370,NaN,0,1,12.8352
88,ORDX10933,BK,2018-12-11,Dinner,0,10.0450,NaN,0,2,16.9788
93,ORDB10808,TP,2018-08-16,Lunch,0,8.9620,NaN,0,1,12.3960


,check,value
0,delivery_fee originally missing,50
1,delivery_fee missing after missing-only model,0
2,minimum imputed delivery_fee,6.5603
3,maximum imputed delivery_fee,19.2499
4,minimum branch R2,0.9520
5,row count preserved,True


delivery_fee imputed rows with missing-only model: 50


### Cell 21.1 讨论：为什么 missing-only 模型更合理？

**Q1：为什么上一格 R2 很低？**  
A：因为把不同输入文件混在一起训练 delivery fee model 会引入不一致。对于 missing imputation，最可靠的 reference 是同一个 missing_data 文件中 delivery_fee 非缺失的 rows。

**Q2：为什么 missing_data 的非缺失 rows 可以作为 training data？**  
A：guide 明确说 missing_data 只有 coverage/missing anomalies，没有其他 data anomalies。因此非缺失 delivery_fee rows 可以认为是干净标签。

**Q3：为什么 R2 要接近 1？**  
A：guide 说明 proper data training 应得到接近 1 的 R2。R2 低说明模型或训练数据选择有问题。

**Q4：这一步修正了什么？**  
A：重新训练 branch-specific fee models，并覆盖上一格低 R2 模型产生的 imputed delivery_fee。

**下一步：**  
做 missing_data final validation，并导出 solution CSV。

## Cell 22：missing_data final validation

现在对 `missing_working` 做 final validation。

目标是确认：

- 所有 missing values 已经填补；
- row count 和原始 columns 保持；
- branch_code 和 prefix 一致；
- distance 和 graph shortest path 一致；
- delivery_fee 模型质量达标。

In [48]:
missing_final = missing_working.copy()
original_missing = data['missing'].copy()

# Keep only original columns for final output checks.
missing_final_output = missing_final[original_missing.columns].copy()

missing_final['order_prefix'] = missing_final['order_id'].str.extract(r'^ORD([A-Z])')
missing_final['expected_branch_code'] = missing_final['order_prefix'].map(prefix_to_expected_branch)
missing_final['branch_code_valid'] = missing_final['branch_code'].isin(valid_branch_codes)
missing_final['branch_code_matches_prefix'] = missing_final['branch_code'] == missing_final['expected_branch_code']

missing_final['customer_coord_key'] = list(
    zip(
        missing_final['customer_lat'].round(7),
        missing_final['customer_lon'].round(7),
    )
)
missing_final['customer_node'] = missing_final['customer_coord_key'].map(coord_to_node)
missing_final['expected_distance_to_customer_KM'] = missing_final.apply(expected_distance_km, axis=1)
missing_final['distance_difference'] = (
    missing_final['distance_to_customer_KM'] - missing_final['expected_distance_to_customer_KM']
).round(3)
missing_final['distance_matches_graph'] = missing_final['distance_difference'].abs() <= 0.001

missing_final_validation = pd.DataFrame([
    {
        'check': 'row count preserved',
        'passed': len(missing_final_output) == len(original_missing),
        'failed_rows': 0 if len(missing_final_output) == len(original_missing) else abs(len(missing_final_output) - len(original_missing)),
    },
    {
        'check': 'column order preserved',
        'passed': list(missing_final_output.columns) == list(original_missing.columns),
        'failed_rows': 0,
    },
    {
        'check': 'no missing values remain',
        'passed': missing_final_output.isna().sum().sum() == 0,
        'failed_rows': int(missing_final_output.isna().any(axis=1).sum()),
    },
    {
        'check': 'branch_code valid and prefix-consistent',
        'passed': bool((missing_final['branch_code_valid'] & missing_final['branch_code_matches_prefix']).all()),
        'failed_rows': int((~(missing_final['branch_code_valid'] & missing_final['branch_code_matches_prefix'])).sum()),
    },
    {
        'check': 'customer_node available for all rows',
        'passed': bool(missing_final['customer_node'].notna().all()),
        'failed_rows': int(missing_final['customer_node'].isna().sum()),
    },
    {
        'check': 'distance matches graph shortest path',
        'passed': bool(missing_final['distance_matches_graph'].all()),
        'failed_rows': int((~missing_final['distance_matches_graph']).sum()),
    },
    {
        'check': 'delivery_fee model minimum R2 >= 0.95',
        'passed': bool(missing_only_fee_report_df['r2_score'].min() >= 0.95),
        'failed_rows': 0,
    },
    {
        'check': 'delivery_fee positive',
        'passed': bool((missing_final_output['delivery_fee'] > 0).all()),
        'failed_rows': int((missing_final_output['delivery_fee'] <= 0).sum()),
    },
])

display(missing_final_validation)

display(missing_only_fee_report_df)

failed_missing_checks = missing_final_validation.loc[~missing_final_validation['passed']]
display(failed_missing_checks)

print('all final missing checks passed:', failed_missing_checks.empty)
print('missing_final_output shape:', missing_final_output.shape)
print('total missing cells:', int(missing_final_output.isna().sum().sum()))

,check,passed,failed_rows
0,row count preserved,True,0
1,column order preserved,True,0
2,no missing values remain,True,0
3,branch_code valid and prefix-consistent,True,0
4,customer_node available for all rows,True,0
5,distance matches graph shortest path,True,0
6,delivery_fee model minimum R2 >= 0.95,True,0
7,delivery_fee positive,True,0


,branch_code,training_rows,r2_score,intercept,coef_is_weekend,coef_time_code,coef_distance,mean_abs_residual
0,BK,153,0.9946,4.4431,2.5258,0.9586,1.0571,0.2371
1,NS,128,0.9670,4.8406,1.9294,0.5377,1.0163,0.2613
2,TP,169,0.9520,4.0369,1.4818,0.7360,0.8506,0.2415


,check,passed,failed_rows


all final missing checks passed: True
missing_final_output shape: (500, 12)
total missing cells: 0


### Cell 22 讨论：missing_data 是否可以导出？

**Q1：missing final validation 最重要看什么？**  
A：首先看没有 missing values；其次看 row/column structure 保持；最后看 branch、distance、delivery_fee model 这些依赖规则是否通过。

**Q2：为什么 final output 要只保留原始 columns？**  
A：中间计算加了很多 helper columns，但 submission CSV 必须和 input CSV 有完全相同的列名和顺序。

**Q3：如果全部通过，下一步是什么？**  
A：导出 `Group024_missing_data_solution.csv`，然后进入 outlier_data。

## Cell 23：导出 dirty 和 missing solution CSV

现在导出已经通过 final validation 的 dirty 和 missing solution。

注意：提交 CSV 必须保留和 input 完全相同的列名与列顺序，所以导出前只选择原始 columns。

In [49]:
dirty_solution_output = dirty_cleaning[data['dirty'].columns].copy()
missing_solution_output = missing_final_output[data['missing'].columns].copy()

dirty_solution_path = ROOT / 'Group024_dirty_data_solution.csv'
missing_solution_path = ROOT / 'Group024_missing_data_solution.csv'

dirty_solution_output.to_csv(dirty_solution_path, index=False)
missing_solution_output.to_csv(missing_solution_path, index=False)

readback_dirty = pd.read_csv(dirty_solution_path)
readback_missing = pd.read_csv(missing_solution_path)

export_validation = pd.DataFrame([
    {
        'file': dirty_solution_path.name,
        'rows': len(readback_dirty),
        'columns_match': list(readback_dirty.columns) == list(data['dirty'].columns),
        'missing_cells': int(readback_dirty.isna().sum().sum()),
    },
    {
        'file': missing_solution_path.name,
        'rows': len(readback_missing),
        'columns_match': list(readback_missing.columns) == list(data['missing'].columns),
        'missing_cells': int(readback_missing.isna().sum().sum()),
    },
])

display(export_validation)

print('dirty solution path:', dirty_solution_path)
print('missing solution path:', missing_solution_path)

,file,rows,columns_match,missing_cells
0,Group024_dirty_data_solution.csv,500,True,0
1,Group024_missing_data_solution.csv,500,True,0


dirty solution path: /Users/songhaifan/Documents/GitHub/teaching-materials/courses/lincoin/materials/fit5196/ass2_2026/Group024_dirty_data_solution.csv
missing solution path: /Users/songhaifan/Documents/GitHub/teaching-materials/courses/lincoin/materials/fit5196/ass2_2026/Group024_missing_data_solution.csv


### Cell 23 讨论：为什么要 read-back validation？

**Q1：为什么导出后还要读回来？**  
A：因为写 CSV 时可能出现列顺序、列名、index 或空值问题。读回来检查可以提前发现提交格式错误。

**Q2：为什么只导出原始列？**  
A：notebook 中有很多 helper columns，但 assignment 要求 output CSV 和 input CSV 有完全相同的 schema。

**下一步：**  
进入 outlier_data，只针对 `delivery_fee` 检测并删除 outlier rows。

## Cell 24：进入 outlier_data，建立 delivery_fee outlier 检测边界

现在进入 Task 1 第三部分：detect and remove outlier rows in `Group024_outlier_data.csv`。

Assignment guide 明确说：outlier_data 除了 outliers 以外没有其他 data anomalies，而且 outlier 只针对 `delivery_fee` attribute。

所以这里不要修 branch/date/items/distance，也不要改值；目标是识别并删除 delivery_fee outlier rows。

In [50]:
outlier_working = data['outlier'].copy()

outlier_overview = pd.DataFrame({
    'column': outlier_working.columns,
    'dtype': [str(outlier_working[column].dtype) for column in outlier_working.columns],
    'missing_count': [int(outlier_working[column].isna().sum()) for column in outlier_working.columns],
    'unique_count': [int(outlier_working[column].nunique(dropna=True)) for column in outlier_working.columns],
    'example_values': [outlier_working[column].dropna().head(3).tolist() for column in outlier_working.columns],
})

display(outlier_overview)

display(outlier_working['delivery_fee'].describe().to_frame(name='delivery_fee'))

print('outlier_data shape:', outlier_working.shape)
print('total missing cells:', int(outlier_working.isna().sum().sum()))

,column,dtype,missing_count,unique_count,example_values
0,order_id,str,0,500,"[ORDB06939, ORDJ06566, ORDX02967]"
1,date,str,0,273,"[2018-08-05, 2018-02-15, 2018-08-31]"
2,time,str,0,72,"[15:05:54, 09:31:16, 15:56:37]"
3,order_type,str,0,3,"[Lunch, Breakfast, Lunch]"
4,branch_code,str,0,3,"[TP, TP, BK]"
5,order_items,str,0,494,"[[('Steak', 1), ('Burger', 9), ('Salad', 2), ('Chicken', 10)], [('Cereal', 9), ('Coffee', 2), ('Eggs', 1), ('Pancake..."
6,order_price,float64,0,457,"[678.4, 395.75, 674.8]"
7,customer_lat,float64,0,488,"[-37.8113095, -37.811221, -37.8044157]"
8,customer_lon,float64,0,488,"[144.9476704, 145.0073618, 144.9592957]"
9,customerHasloyalty?,int64,0,2,"[0, 0, 0]"


,delivery_fee
count,500.0000
mean,13.7536
std,3.1272
min,3.0802
25%,12.4380
50%,13.7805
75%,15.3262
max,27.7906


outlier_data shape: (500, 12)
total missing cells: 0


### Cell 24 讨论：outlier_data 和前两个文件有什么不同？

**Q1：为什么这里不做 dirty cleaning？**  
A：guide 说 outlier_data 除了 outlier 外没有其他 data anomalies，所以不要改字段值。

**Q2：为什么只看 delivery_fee？**  
A：guide 明确说 outlier rows are with respect to the delivery_fee attribute only。

**Q3：outlier detection 的基本思路是什么？**  
A：先建立合理的 delivery_fee model，预测 expected fee，再用 residual 判断哪些 rows 的 fee 明显偏离模型。

**Q4：下一步做什么？**  
A：和 missing fee 类似，先处理 loyalty discount，然后按 branch 建 delivery_fee regression model。

**下一步：**  
训练 outlier delivery_fee model，并查看 residual 分布。

## Cell 25：训练 outlier delivery_fee 模型并计算 residual

Outlier 检测使用和 missing delivery_fee 类似的模型：按 branch 分别建 linear regression，features 是 weekend/time/distance，并处理 loyalty discount。

区别是：这里不是填补值，而是用 residual 判断哪些 rows 的 `delivery_fee` 明显异常。

In [51]:
outlier_model_data = add_delivery_fee_features(outlier_working.copy())
outlier_model_data['adjusted_delivery_fee'] = np.where(
    outlier_model_data['customerHasloyalty?'] == 1,
    outlier_model_data['delivery_fee'] * 2,
    outlier_model_data['delivery_fee'],
)

initial_outlier_models = {}
initial_outlier_reports = []
initial_residual_frames = []

for branch_code, branch_rows in outlier_model_data.groupby('branch_code'):
    model = LinearRegression()
    X = branch_rows[fee_feature_columns]
    y = branch_rows['adjusted_delivery_fee']
    model.fit(X, y)
    predictions = model.predict(X)
    residuals = y - predictions
    initial_outlier_models[branch_code] = model
    initial_outlier_reports.append({
        'branch_code': branch_code,
        'rows': len(branch_rows),
        'r2_score_all_rows': r2_score(y, predictions),
        'mean_abs_residual': np.mean(np.abs(residuals)),
        'residual_std': np.std(residuals),
    })

    branch_residuals = branch_rows[['order_id', 'branch_code', 'delivery_fee', 'customerHasloyalty?'] + fee_feature_columns].copy()
    branch_residuals['adjusted_delivery_fee'] = y.to_numpy()
    branch_residuals['predicted_adjusted_fee'] = predictions
    branch_residuals['residual'] = residuals
    branch_residuals['abs_residual'] = np.abs(residuals)
    initial_residual_frames.append(branch_residuals)

initial_outlier_report_df = pd.DataFrame(initial_outlier_reports)
outlier_residuals = pd.concat(initial_residual_frames, ignore_index=True)

display(initial_outlier_report_df)

display(
    outlier_residuals
    .groupby('branch_code')['abs_residual']
    .describe()
    .reset_index()
)

display(
    outlier_residuals
    .sort_values('abs_residual', ascending=False)
    .head(30)
)

print('initial outlier model rows:', len(outlier_residuals))

,branch_code,rows,r2_score_all_rows,mean_abs_residual,residual_std
0,BK,184,0.7457,0.4924,1.3345
1,NS,158,0.4137,1.0713,2.2606
2,TP,158,0.2347,1.1313,2.3534


,branch_code,count,mean,std,min,25%,50%,75%,max
0,BK,184.0000,0.4924,1.2437,0.0045,0.0952,0.2230,0.4126,9.1012
1,NS,158.0000,1.0713,1.9969,0.0016,0.1639,0.3775,0.9099,8.2472
2,TP,158.0000,1.1313,2.0702,0.0028,0.1821,0.3345,0.6801,7.7668


,order_id,branch_code,delivery_fee,customerHasloyalty?,is_weekend,time_code,distance_to_customer_KM,adjusted_delivery_fee,predicted_adjusted_fee,residual,abs_residual
70,ORDX02124,BK,12.9103,1,1,0,9.1830,25.8205,16.7193,9.1012,9.1012
307,ORDZ10737,NS,27.7906,0,1,2,10.6220,27.7906,19.5435,8.2472,8.2472
187,ORDZ01820,NS,27.0965,0,1,2,10.0060,27.0965,18.8817,8.2148,8.2148
205,ORDZ01558,NS,12.9249,1,1,0,9.6660,25.8498,17.7603,8.0894,8.0894
247,ORDC03093,NS,23.1221,0,0,1,9.6330,23.1221,15.2272,7.8949,7.8949
426,ORDJ06587,TP,8.0474,0,1,2,11.2800,8.0474,15.8142,-7.7668,7.7668
273,ORDI10228,NS,7.1761,0,0,2,8.9870,7.1761,14.9112,-7.7352,7.7352
432,ORDY04248,TP,21.6741,0,0,1,11.3670,21.6741,13.9475,7.7265,7.7265
437,ORDY00062,TP,21.4494,0,1,1,9.5150,21.4494,13.8174,7.6320,7.6320
469,ORDB10739,TP,7.8482,0,0,2,12.3030,7.8482,15.3487,-7.5005,7.5005


initial outlier model rows: 500


### Cell 25 讨论：为什么初始 R2 可能不高？

**Q1：outlier_data 里有 outliers，为什么还先用全部数据拟合？**  
A：这是初步诊断。包含 outliers 的模型可能被异常值拉偏，所以初始 R2 不一定高。

**Q2：residual 的意义是什么？**  
A：residual 是 actual adjusted delivery_fee 和 model predicted adjusted fee 的差。delivery_fee outlier 应该表现为 residual 明显大。

**Q3：下一步怎么找 outliers？**  
A：按 branch 分别看 residual 分布，用 IQR rule 或 robust threshold 标记 extreme residual rows。

**下一步：**  
用 branch-wise residual IQR rule 标记 delivery_fee outliers。

## Cell 25.1：用 branch-wise IQR rule 标记 delivery_fee outliers

现在用 residual 做 outlier detection。

因为 delivery_fee 的计算方法按 branch 不同，所以 residual threshold 也按 branch 分别计算。这里使用 abs residual 的 IQR rule：

```text
upper_bound = Q3 + 1.5 * IQR
```

超过 upper_bound 的 rows 标记为 delivery_fee outliers。

In [52]:
outlier_threshold_records = []
outlier_flag_frames = []

for branch_code, branch_residuals in outlier_residuals.groupby('branch_code'):
    q1 = branch_residuals['abs_residual'].quantile(0.25)
    q3 = branch_residuals['abs_residual'].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    branch_flagged = branch_residuals.copy()
    branch_flagged['outlier_threshold'] = upper_bound
    branch_flagged['is_delivery_fee_outlier'] = branch_flagged['abs_residual'] > upper_bound
    outlier_flag_frames.append(branch_flagged)

    outlier_threshold_records.append({
        'branch_code': branch_code,
        'q1_abs_residual': q1,
        'q3_abs_residual': q3,
        'iqr_abs_residual': iqr,
        'upper_bound': upper_bound,
        'outlier_count': int(branch_flagged['is_delivery_fee_outlier'].sum()),
    })

outlier_thresholds_df = pd.DataFrame(outlier_threshold_records)
outlier_flagged_residuals = pd.concat(outlier_flag_frames, ignore_index=True)

delivery_fee_outlier_rows = outlier_flagged_residuals.loc[
    outlier_flagged_residuals['is_delivery_fee_outlier']
].copy()

display(outlier_thresholds_df)

display(
    delivery_fee_outlier_rows
    .sort_values(['branch_code', 'abs_residual'], ascending=[True, False])
    [[
        'order_id',
        'branch_code',
        'delivery_fee',
        'customerHasloyalty?',
        'is_weekend',
        'time_code',
        'distance_to_customer_KM',
        'adjusted_delivery_fee',
        'predicted_adjusted_fee',
        'residual',
        'abs_residual',
        'outlier_threshold',
    ]]
)

print('delivery_fee outlier rows detected:', len(delivery_fee_outlier_rows))

,branch_code,q1_abs_residual,q3_abs_residual,iqr_abs_residual,upper_bound,outlier_count
0,BK,0.0952,0.4126,0.3174,0.8887,9
1,NS,0.1639,0.9099,0.7459,2.0287,14
2,TP,0.1821,0.6801,0.4980,1.4271,21


,order_id,branch_code,delivery_fee,customerHasloyalty?,is_weekend,time_code,distance_to_customer_KM,adjusted_delivery_fee,predicted_adjusted_fee,residual,abs_residual,outlier_threshold
70,ORDX02124,BK,12.9103,1,1,0,9.1830,25.8205,16.7193,9.1012,9.1012,0.8887
109,ORDA03445,BK,7.5113,0,0,1,8.8610,7.5113,14.7720,-7.2607,7.2607,0.8887
160,ORDK00636,BK,22.3731,0,0,2,8.3560,22.3731,15.3469,7.0262,7.0262,0.8887
29,ORDK09590,BK,6.6534,0,0,0,8.3580,6.6534,13.1290,-6.4756,6.4756,0.8887
30,ORDK00865,BK,19.8712,0,0,1,7.7410,19.8712,13.5852,6.2860,6.2860,0.8887
178,ORDK02256,BK,5.2169,0,0,0,5.6040,5.2169,10.2108,-4.9939,4.9939,0.8887
141,ORDX02830,BK,14.3142,0,0,2,3.4540,14.3142,10.1526,4.1616,4.1616,0.8887
177,ORDX08202,BK,19.4176,0,1,2,10.4940,19.4176,20.3285,-0.9109,0.9109,0.8887
182,ORDA04575,BK,6.8752,1,0,0,8.0940,13.7503,12.8492,0.9011,0.9011,0.8887
307,ORDZ10737,NS,27.7906,0,1,2,10.6220,27.7906,19.5435,8.2472,8.2472,2.0287


delivery_fee outlier rows detected: 44


### Cell 25.1 讨论：为什么按 branch 分别设阈值？

**Q1：为什么不用全局 residual threshold？**  
A：guide 说不同 branch 的 delivery fee 计算方法不同。每个 branch 的 residual 分布也可能不同，所以 threshold 应该按 branch 计算。

**Q2：为什么用 abs residual？**  
A：delivery_fee outlier 可能异常偏高，也可能异常偏低。abs residual 可以同时捕捉两种情况。

**Q3：IQR rule 的优点是什么？**  
A：它比均值/标准差更 robust，不容易被极端 outlier 拉偏。

**Q4：下一步做什么？**  
A：删除这些 outlier rows，然后用保留 rows 重新训练 delivery fee model，检查 R2 是否提升到接近 1。

**下一步：**  
删除 outlier rows，并验证 retained-row model。

## Cell 25.2：删除 IQR outliers 后重训 retained-row model

现在先删除 Cell 25.1 标记出的 delivery_fee outliers，然后用保留 rows 重新训练 branch-specific models。

如果保留 rows 的 R2 仍不够接近 1，说明第一轮 IQR 没有删干净，需要进一步迭代检测。

In [53]:
iqr_outlier_order_ids = set(delivery_fee_outlier_rows['order_id'])

outlier_retained_after_iqr = outlier_working.loc[
    ~outlier_working['order_id'].isin(iqr_outlier_order_ids)
].copy()

retained_iqr_model_data = add_delivery_fee_features(outlier_retained_after_iqr.copy())
retained_iqr_model_data['adjusted_delivery_fee'] = np.where(
    retained_iqr_model_data['customerHasloyalty?'] == 1,
    retained_iqr_model_data['delivery_fee'] * 2,
    retained_iqr_model_data['delivery_fee'],
)

retained_iqr_reports = []
retained_iqr_residual_frames = []

for branch_code, branch_rows in retained_iqr_model_data.groupby('branch_code'):
    model = LinearRegression()
    X = branch_rows[fee_feature_columns]
    y = branch_rows['adjusted_delivery_fee']
    model.fit(X, y)
    predictions = model.predict(X)
    residuals = y - predictions
    retained_iqr_reports.append({
        'branch_code': branch_code,
        'retained_rows': len(branch_rows),
        'r2_score': r2_score(y, predictions),
        'mean_abs_residual': np.mean(np.abs(residuals)),
        'residual_std': np.std(residuals),
    })

    branch_residuals = branch_rows[['order_id', 'branch_code', 'delivery_fee', 'customerHasloyalty?'] + fee_feature_columns].copy()
    branch_residuals['adjusted_delivery_fee'] = y.to_numpy()
    branch_residuals['predicted_adjusted_fee'] = predictions
    branch_residuals['residual'] = residuals
    branch_residuals['abs_residual'] = np.abs(residuals)
    retained_iqr_residual_frames.append(branch_residuals)

retained_iqr_report_df = pd.DataFrame(retained_iqr_reports)
retained_iqr_residuals = pd.concat(retained_iqr_residual_frames, ignore_index=True)

display(pd.DataFrame([
    {
        'metric': 'original rows',
        'value': len(outlier_working),
    },
    {
        'metric': 'removed rows after first IQR',
        'value': len(iqr_outlier_order_ids),
    },
    {
        'metric': 'retained rows after first IQR',
        'value': len(outlier_retained_after_iqr),
    },
]))

display(retained_iqr_report_df)

display(
    retained_iqr_residuals
    .groupby('branch_code')['abs_residual']
    .describe()
    .reset_index()
)

print('minimum retained R2 after first IQR:', retained_iqr_report_df['r2_score'].min())

,metric,value
0,original rows,500
1,removed rows after first IQR,44
2,retained rows after first IQR,456


,branch_code,retained_rows,r2_score,mean_abs_residual,residual_std
0,BK,175,0.9825,0.2271,0.2902
1,NS,144,0.9601,0.2514,0.3273
2,TP,137,0.9620,0.2228,0.2774


,branch_code,count,mean,std,min,25%,50%,75%,max
0,BK,175.0000,0.2271,0.1813,0.0063,0.0775,0.1929,0.3318,0.7452
1,NS,144.0000,0.2514,0.2102,0.0053,0.0973,0.2222,0.3412,1.4008
2,TP,137.0000,0.2228,0.1658,0.0003,0.0837,0.2048,0.3171,0.7650


minimum retained R2 after first IQR: 0.9601038739855013


### Cell 25.2 讨论：第一轮 IQR 是否足够？

**Q1：为什么要重训 retained-row model？**  
A：outlier detection 的目标不是只找到大 residual，而是删除 outliers 后让正常数据重新符合 delivery_fee linear model。

**Q2：怎么看是否足够？**  
A：看 retained rows 的 branch-specific R2 是否接近 1。如果最低 R2 仍明显低于 0.95，说明还有残留 outliers。

**Q3：如果还不够怎么办？**  
A：可以在 retained rows 上再次计算 residual 和 IQR，迭代删除剩余 outliers，直到 retained model R2 达标。

**下一步：**  
根据 retained R2 决定是否需要第二轮 residual outlier detection。

## Cell 25.3：第二轮 residual IQR check

第一轮删除后 retained-row model 已经达到 R2 > 0.95。

为了确认是否还有边界 outliers，我们在 retained rows 上重新计算 residual，并再次应用 branch-wise IQR rule。

In [54]:
second_threshold_records = []
second_flag_frames = []

for branch_code, branch_residuals in retained_iqr_residuals.groupby('branch_code'):
    q1 = branch_residuals['abs_residual'].quantile(0.25)
    q3 = branch_residuals['abs_residual'].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    branch_flagged = branch_residuals.copy()
    branch_flagged['second_round_threshold'] = upper_bound
    branch_flagged['is_second_round_outlier'] = branch_flagged['abs_residual'] > upper_bound
    second_flag_frames.append(branch_flagged)

    second_threshold_records.append({
        'branch_code': branch_code,
        'q1_abs_residual': q1,
        'q3_abs_residual': q3,
        'iqr_abs_residual': iqr,
        'upper_bound': upper_bound,
        'second_round_outlier_count': int(branch_flagged['is_second_round_outlier'].sum()),
    })

second_thresholds_df = pd.DataFrame(second_threshold_records)
second_flagged_residuals = pd.concat(second_flag_frames, ignore_index=True)

second_round_outlier_rows = second_flagged_residuals.loc[
    second_flagged_residuals['is_second_round_outlier']
].copy()

display(second_thresholds_df)

display(
    second_round_outlier_rows
    .sort_values(['branch_code', 'abs_residual'], ascending=[True, False])
    [[
        'order_id',
        'branch_code',
        'delivery_fee',
        'customerHasloyalty?',
        'is_weekend',
        'time_code',
        'distance_to_customer_KM',
        'adjusted_delivery_fee',
        'predicted_adjusted_fee',
        'residual',
        'abs_residual',
        'second_round_threshold',
    ]]
)

all_iterative_outlier_order_ids = iqr_outlier_order_ids.union(set(second_round_outlier_rows['order_id']))
outlier_retained_after_second = outlier_working.loc[
    ~outlier_working['order_id'].isin(all_iterative_outlier_order_ids)
].copy()

second_retained_model_data = add_delivery_fee_features(outlier_retained_after_second.copy())
second_retained_model_data['adjusted_delivery_fee'] = np.where(
    second_retained_model_data['customerHasloyalty?'] == 1,
    second_retained_model_data['delivery_fee'] * 2,
    second_retained_model_data['delivery_fee'],
)

second_retained_reports = []
for branch_code, branch_rows in second_retained_model_data.groupby('branch_code'):
    model = LinearRegression()
    X = branch_rows[fee_feature_columns]
    y = branch_rows['adjusted_delivery_fee']
    model.fit(X, y)
    predictions = model.predict(X)
    residuals = y - predictions
    second_retained_reports.append({
        'branch_code': branch_code,
        'retained_rows': len(branch_rows),
        'r2_score': r2_score(y, predictions),
        'mean_abs_residual': np.mean(np.abs(residuals)),
        'residual_std': np.std(residuals),
    })

second_retained_report_df = pd.DataFrame(second_retained_reports)

display(pd.DataFrame([
    {
        'metric': 'first_round_outliers',
        'value': len(iqr_outlier_order_ids),
    },
    {
        'metric': 'second_round_new_outliers',
        'value': len(second_round_outlier_rows),
    },
    {
        'metric': 'total_iterative_outliers',
        'value': len(all_iterative_outlier_order_ids),
    },
    {
        'metric': 'retained_rows_after_second_round',
        'value': len(outlier_retained_after_second),
    },
]))

display(second_retained_report_df)

print('minimum retained R2 after second round:', second_retained_report_df['r2_score'].min())

,branch_code,q1_abs_residual,q3_abs_residual,iqr_abs_residual,upper_bound,second_round_outlier_count
0,BK,0.0775,0.3318,0.2543,0.7133,3
1,NS,0.0973,0.3412,0.2439,0.7071,4
2,TP,0.0837,0.3171,0.2335,0.6674,3


,order_id,branch_code,delivery_fee,customerHasloyalty?,is_weekend,time_code,distance_to_customer_KM,adjusted_delivery_fee,predicted_adjusted_fee,residual,abs_residual,second_round_threshold
122,ORDX02979,BK,15.3241,0,1,0,8.6090,15.3241,16.0693,-0.7452,0.7452,0.7133
136,ORDA08368,BK,12.3425,0,0,2,4.9260,12.3425,11.5973,0.7452,0.7452,0.7133
72,ORDA06513,BK,16.3341,0,1,0,8.1850,16.3341,15.6202,0.7139,0.7139,0.7133
177,ORDZ10422,NS,5.7770,1,0,0,7.9710,11.5541,12.9549,-1.4008,1.4008,0.7071
182,ORDC08719,NS,11.5696,0,0,1,7.0420,11.5696,12.5488,-0.9792,0.9792,0.7071
313,ORDI07669,NS,15.4761,0,0,1,9.0850,15.4761,14.5820,0.8941,0.8941,0.7071
187,ORDI08642,NS,18.1406,0,0,2,11.2940,18.1406,17.2988,0.8418,0.8418,0.7071
455,ORDB06432,TP,5.6144,1,0,0,9.3230,11.2288,11.9937,-0.7650,0.7650,0.6674
405,ORDJ03801,TP,10.9393,0,0,0,8.9260,10.9393,11.6570,-0.7177,0.7177,0.6674
374,ORDJ04818,TP,12.7551,0,0,1,8.5810,12.7551,12.0597,0.6955,0.6955,0.6674


,metric,value
0,first_round_outliers,44
1,second_round_new_outliers,10
2,total_iterative_outliers,54
3,retained_rows_after_second_round,446


,branch_code,retained_rows,r2_score,mean_abs_residual,residual_std
0,BK,172,0.9843,0.2182,0.2760
1,NS,140,0.9694,0.2266,0.2779
2,TP,134,0.9667,0.2101,0.2582


minimum retained R2 after second round: 0.9667076312538005


### Cell 25.3 讨论：是否采用第二轮 outlier 结果？

**Q1：为什么第一轮达标后还做第二轮？**  
A：为了检查是否还有边界 residual outliers。迭代后模型如果更稳定，说明第二轮有价值。

**Q2：什么时候停止迭代？**  
A：当 retained-row model R2 达标且第二轮新增 outliers 很少，或继续迭代会开始删除正常边界点时，就可以停止。

**Q3：下一步做什么？**  
A：根据第二轮结果决定最终删除的 outlier rows，并导出 outlier solution。

## Cell 26：导出教学版 outlier solution

教学版采用第一轮 branch-wise IQR residual outlier detection。

理由：

- 方法清晰、可解释；
- 按 branch 分开，符合 guide 中不同 branch 使用不同 fee 方法的设定；
- 使用 adjusted delivery_fee 处理 loyalty discount；
- 删除后 retained-row model 最低 R2 已超过 0.95。

进一步迭代或调参可以作为学生探索更优解的方向。

In [55]:
outlier_solution_output = outlier_retained_after_iqr[data['outlier'].columns].copy()
outlier_solution_path = ROOT / 'Group024_outlier_data_solution.csv'
outlier_solution_output.to_csv(outlier_solution_path, index=False)

readback_outlier = pd.read_csv(outlier_solution_path)

outlier_final_validation = pd.DataFrame([
    {
        'check': 'original rows',
        'value': len(data['outlier']),
    },
    {
        'check': 'outlier rows removed',
        'value': len(iqr_outlier_order_ids),
    },
    {
        'check': 'retained rows',
        'value': len(readback_outlier),
    },
    {
        'check': 'column order preserved',
        'value': list(readback_outlier.columns) == list(data['outlier'].columns),
    },
    {
        'check': 'missing cells in output',
        'value': int(readback_outlier.isna().sum().sum()),
    },
    {
        'check': 'minimum retained branch R2 after removal',
        'value': float(retained_iqr_report_df['r2_score'].min()),
    },
])

display(outlier_final_validation)

display(retained_iqr_report_df)

display(outlier_thresholds_df)

print('outlier solution path:', outlier_solution_path)

,check,value
0,original rows,500
1,outlier rows removed,44
2,retained rows,456
3,column order preserved,True
4,missing cells in output,0
5,minimum retained branch R2 after removal,0.9601


,branch_code,retained_rows,r2_score,mean_abs_residual,residual_std
0,BK,175,0.9825,0.2271,0.2902
1,NS,144,0.9601,0.2514,0.3273
2,TP,137,0.9620,0.2228,0.2774


,branch_code,q1_abs_residual,q3_abs_residual,iqr_abs_residual,upper_bound,outlier_count
0,BK,0.0952,0.4126,0.3174,0.8887,9
1,NS,0.1639,0.9099,0.7459,2.0287,14
2,TP,0.1821,0.6801,0.4980,1.4271,21


outlier solution path: /Users/songhaifan/Documents/GitHub/teaching-materials/courses/lincoin/materials/fit5196/ass2_2026/Group024_outlier_data_solution.csv


### Cell 26 讨论：为什么教学版停在第一轮 IQR？

**Q1：第一轮 IQR 是否满足 guide？**  
A：满足。删除后 retained model 的最低 branch R2 超过 0.95，符合 guide 对 delivery_fee model quality 的要求。

**Q2：为什么不继续第二轮？**  
A：第二轮会继续删除边界 residual rows，R2 略有提升，但可能开始过度删除正常边界点。教学版强调清晰、可解释、达标。

**Q3：学生可以探索什么？**  
A：可以尝试 iterative IQR、3-sigma residual、robust regression、cross-validation 或不同 threshold，并比较保留模型 R2 和删除数量。

**Q4：Task 1 三个 CSV 是否完成？**  
A：dirty、missing、outlier 三个 solution CSV 都已经导出。最后需要做一次总 read-back checklist。

## Cell 27：Task 1 total read-back checklist

最后一次性读回三个 Task 1 solution CSV，检查是否满足提交层面的基本要求：

- 文件可读；
- columns 和 input 完全一致；
- dirty/missing row count 保持 500；
- outlier row count 小于 500；
- missing solution 没有缺失；
- 三个输出都没有额外 index column。

In [56]:
task1_outputs = {
    'dirty': {
        'input': data['dirty'],
        'path': ROOT / 'Group024_dirty_data_solution.csv',
        'expected_rows': len(data['dirty']),
    },
    'missing': {
        'input': data['missing'],
        'path': ROOT / 'Group024_missing_data_solution.csv',
        'expected_rows': len(data['missing']),
    },
    'outlier': {
        'input': data['outlier'],
        'path': ROOT / 'Group024_outlier_data_solution.csv',
        'expected_rows': None,
    },
}

readback_rows = []
for name, config in task1_outputs.items():
    output = pd.read_csv(config['path'])
    input_df = config['input']
    readback_rows.append({
        'dataset': name,
        'file': config['path'].name,
        'readable': True,
        'rows': len(output),
        'expected_rows': config['expected_rows'] if config['expected_rows'] is not None else '< 500',
        'row_count_ok': len(output) == config['expected_rows'] if config['expected_rows'] is not None else len(output) < len(input_df),
        'columns_match': list(output.columns) == list(input_df.columns),
        'missing_cells': int(output.isna().sum().sum()),
        'extra_index_column': any(str(column).startswith('Unnamed') for column in output.columns),
    })

final_task1_checklist = pd.DataFrame(readback_rows)

display(final_task1_checklist)

all_task1_outputs_ok = (
    final_task1_checklist['readable'].all() and
    final_task1_checklist['row_count_ok'].all() and
    final_task1_checklist['columns_match'].all() and
    (final_task1_checklist['extra_index_column'] == False).all() and
    (final_task1_checklist.loc[final_task1_checklist['dataset'] == 'missing', 'missing_cells'].iloc[0] == 0)
)

print('all Task 1 output CSV checks passed:', bool(all_task1_outputs_ok))
print('outlier rows removed:', len(data['outlier']) - final_task1_checklist.loc[final_task1_checklist['dataset'] == 'outlier', 'rows'].iloc[0])

,dataset,file,readable,rows,expected_rows,row_count_ok,columns_match,missing_cells,extra_index_column
0,dirty,Group024_dirty_data_solution.csv,True,500,500,True,True,0,False
1,missing,Group024_missing_data_solution.csv,True,500,500,True,True,0,False
2,outlier,Group024_outlier_data_solution.csv,True,456,< 500,True,True,0,False


all Task 1 output CSV checks passed: True
outlier rows removed: 44


### Cell 27 讨论：Task 1 是否完成？

**Q1：这个 checklist 检查的是最终答案正确性吗？**  
A：它主要检查提交格式和基本完整性。前面的 validation cells 才检查每类业务规则和模型逻辑。

**Q2：如果这里全部通过，说明什么？**  
A：说明三个 Task 1 CSV 都能被 pandas 读回，schema 正确，没有额外 index column，行数符合任务要求。

**Q3：Task 1 下一步是什么？**  
A：CSV 层面完成后，可以整理 notebook 讲解结构，或者进入 Task 2 data reshaping。